## 0 · Rilevamento hardware e configurazione adattiva



In [ ]:
import os, sys, time, math, platform
import psutil
import torch

# --- CPU ---
CPU_PHYSICAL = psutil.cpu_count(logical=False) or 1
CPU_LOGICAL  = psutil.cpu_count(logical=True)  or 1

# --- RAM ---
vm = psutil.virtual_memory()
RAM_TOTAL_GB    = vm.total    / 1e9
RAM_AVAIL_GB    = vm.available / 1e9

# --- GPU / VRAM ---
if torch.cuda.is_available():
    _props         = torch.cuda.get_device_properties(0)
    GPU_NAME       = _props.name
    GPU_VRAM_GB    = _props.total_memory / 1e9
    GPU_COMPUTE    = (_props.major, _props.minor)
    # FP16 tensor core efficiente solo su Volta+ (compute >= 7.0)
    GPU_FP16_FAST  = _props.major >= 7
    DEVICE         = 'cuda'
elif torch.backends.mps.is_available():
    GPU_NAME       = 'Apple MPS'
    GPU_VRAM_GB    = RAM_TOTAL_GB * 0.5   # memoria condivisa: stima conservativa
    GPU_COMPUTE    = (0, 0)
    GPU_FP16_FAST  = False
    DEVICE         = 'mps'
else:
    GPU_NAME       = 'CPU (no GPU)'
    GPU_VRAM_GB    = 0.0
    GPU_COMPUTE    = (0, 0)
    GPU_FP16_FAST  = False
    DEVICE         = 'cpu'

# --- Storage: rileva se il disco che contiene 'data/' è SSD o HDD ---
def _detect_storage_type() -> str:
    """Restituisce 'SSD' o 'HDD' per il disco che contiene la CWD."""
    try:
        import subprocess
        result = subprocess.run(
            ['powershell', '-Command',
             'Get-PhysicalDisk | Select-Object FriendlyName,MediaType | ConvertTo-Json'],
            capture_output=True, text=True, timeout=10
        )
        data = result.stdout
        # Se c'è un SSD nel sistema e il path di lavoro è su C:\, usiamo SSD
        if 'SSD' in data.upper():
            return 'SSD'
        return 'HDD'
    except Exception:
        return 'UNKNOWN'

STORAGE_TYPE = _detect_storage_type()

# --- Parametri derivati ---
# TOKENS: quanti token di sequenza tenere in GPU in un batch ESM/T5
# Formula: VRAM_GB * 700 = ~4900 per 7 GB liberi; clamp [512, 8192]
if DEVICE == 'cpu':
    TOKENS = 512
else:
    _vram_free = GPU_VRAM_GB - 2.0   # 2 GB riservati al modello
    TOKENS = int(max(512, min(8192, _vram_free * 700)))

# Batch size rete: VRAM libera dopo embedding (~4–5 GB) / 0.005 GB per sample
BATCH = max(32, int(min(256, (GPU_VRAM_GB - 2.0) * 60))) if DEVICE != 'cpu' else 32

# DataLoader workers: min(4, fisici - 1) se RAM abbondante, altrimenti 0
NUM_WORKERS = 0 if RAM_AVAIL_GB < 2.0 else min(4, max(0, CPU_PHYSICAL - 1))

# PIN_MEMORY: utile solo con CUDA e RAM sufficiente
PIN_MEMORY = (DEVICE == 'cuda') and (RAM_AVAIL_GB >= 4.0)

# DTYPE: fp16 solo se tensor core efficienti (compute >= 7)
DTYPE = torch.float16 if GPU_FP16_FAST else torch.float32

# Thread PyTorch / OpenMP: usiamo tutti i core fisici
torch.set_num_threads(CPU_PHYSICAL)
os.environ['OMP_NUM_THREADS']          = str(CPU_PHYSICAL)
os.environ['MKL_NUM_THREADS']          = str(CPU_PHYSICAL)
os.environ['TOKENIZERS_PARALLELISM']   = 'false'  # evita warning HuggingFace

# Seed globale
SEED = 42
torch.manual_seed(SEED)

# --- Riepilogo ---
print('=' * 60)
print('HARDWARE RILEVATO E CONFIGURAZIONE ADATTIVA')
print('=' * 60)
print(f'  CPU            : {platform.processor()}')
print(f'  Core fisici    : {CPU_PHYSICAL}  |  Logici: {CPU_LOGICAL}')
print(f'  RAM totale     : {RAM_TOTAL_GB:.1f} GB  |  Disponibile: {RAM_AVAIL_GB:.1f} GB')
print(f'  GPU            : {GPU_NAME}')
print(f'  VRAM           : {GPU_VRAM_GB:.2f} GB')
print(f'  Compute cap.   : {GPU_COMPUTE[0]}.{GPU_COMPUTE[1]}')
print(f'  FP16 veloci    : {GPU_FP16_FAST}  (tensor core Volta+)')
print(f'  Storage CWD    : {STORAGE_TYPE}')
print('-' * 60)
print(f'  DEVICE         : {DEVICE}')
print(f'  DTYPE          : {DTYPE}')
print(f'  TOKENS/batch   : {TOKENS}')
print(f'  BATCH (rete)   : {BATCH}')
print(f'  NUM_WORKERS    : {NUM_WORKERS}')
print(f'  PIN_MEMORY     : {PIN_MEMORY}')
print(f'  OMP_THREADS    : {CPU_PHYSICAL}')
print('=' * 60)


# 02 · NeuroLoc-Δ: proteine umane del cervello — modello di sola sequenza, rete a token, ROC, varianti

- **Dati.** Circa 15.000 proteine umane del cervello
- **Modello principale: sola sequenza.** ESM-2 650M e ProtT5-XL, descrittori e motivi di sorting, struttura AlphaFold. 

## Domande da esperto e risposte che guidano questo notebook
1. *Come si dimostra che la rete serve?* Tre modelli con **gli stessi input, le stesse partizioni di omologia e lo
   stesso test**: una baseline stupida (prevalenza di classe, AUC 0.5 per costruzione), una regressione logistica
   con soglie per classe, la rete neurale. Si confrontano curve ROC, AUPRC e metriche a soglia, con un **bootstrap
   appaiato** che dice se la differenza tra rete e logistica è reale o rumore.
2. *Dove sta il margine più grande?* Nell'encoder. Si usano **ESM-2 650M e ProtT5-XL insieme** (fusione di due
   language model addestrati su dati e obiettivi diversi). Entrambi entrano in una GTX 1080 da 8 GB se usati uno alla volta.
3. *Media su tutta la proteina o pooling appreso sui residui?* Il pooling appreso (Light Attention) aiuta, ma
   richiederebbe circa 40 GB di embedding per-residuo. Qui si tengono 5 riassunti per encoder: media dell'ultimo
   layer, media di un layer intermedio, max, primi 40 e ultimi 40 residui. I due terminali sono quelli che
   contengono i segnali di sorting.
4. *Con ~22.000 proteine, quanta rete?* Poca profondità, molta regolarizzazione:
   - dropout sugli input e **dropout di blocco** (spegne un intero riassunto o encoder, così la rete non dipende da uno solo);
   - LayerNorm;
   - **media mobile esponenziale dei pesi (EMA)**;
   - ensemble di 5 fold × più semi.
5. *Quale loss per un multi-etichetta sbilanciato (Peroxisome ~1%)?* La ricerca prova tre opzioni:
   - BCE con pesi per i positivi;
   - focal loss;
   - **asymmetric loss** (ASL): attenua molto i negativi facili e poco i positivi rari.
6. *Grid search su tutto?* No: costa troppo e sovra-adatta la scelta. **Ricerca a due stadi**: molte configurazioni
   casuali su un solo fold, poi le migliori 5 in CV completa a 5 fold; la vincente si ripete con altri semi.
7. *Come scegliere le soglie senza barare?* Sempre sulle predizioni out-of-fold del training. Le soglie
   **Jaccard** riportano il numero di etichette vicino a quello reale. La calibrazione conformale si fa solo sulle
   **proteine umane** del training, più simili al test HPA.
8. *La ROC basta?* Con classi rare la ROC è ottimista: accanto si riportano AUPRC e curve precision-recall.

9. *Il language model vede già tutti i segnali di sorting?* Non del tutto. Nel pooling i segnali corti si diluiscono,
   quindi si aggiungono **24 descrittori di sorting**:
   - **momento idrofobico N-terminale** (l'elica anfipatica che apre la via mitocondriale) e residui acidi/arginine N-terminali;
   - **lunghezza dei segmenti TM**: le eliche della membrana plasmatica sono più lunghe di quelle di ER e Golgi, ed è
     un determinante noto di dove si ferma una proteina di membrana;
   - **ancora C-terminale** (proteine tail-anchored → mitocondrio esterno, perossisoma, ER), signal anchor, regola
     positive-inside;
   - motivi nelle code: YxxΦ e dileucine (endosomi/lisosomi, e nei neuroni il targeting somatodendritico), diacidico e FF
     (uscita dal RE), PTS1 esteso e PTS2 (perossisoma), NES (export nucleare);
   - **motivi neuronali**: ligandi PDZ C-terminali (ancoraggio alla densità postsinaptica, es. PSD-95), cisteine
     N-terminali palmitoilabili (PSD-95, GAP-43), domini RG/RGG (FUS e granuli di RNA), legame ad ankirina-G
     (segmento iniziale dell'assone);
   - frazione di residui da regioni disordinate.

10. *Le classi sono indipendenti?* No: il sorting è gerarchico. Prima si decide "via secretoria sì/no" (peptide
    segnale o TM), poi la tappa. Si aggiungono 3 **superclassi** come target ausiliari: via secretoria, nucleo-citoplasma,
    import in organelli. Esito: piccolo guadagno (macro-AUPRC +0.004, AUC test +0.004) e soprattutto il **Golgi**
    passa da MCC 0.10 a 0.17. Sulle varianti, la variazione di "via secretoria" è un meccanismo leggibile.
11. *Il microscopio vede le proteine secrete?* No come "extracellulare": le vede lungo la via secretoria. Sul test HPA
    una proteina predetta solo "Extracellular" resta senza etichetta valutabile (5% dei casi). Si riporta anche una
    **decodifica "microscopia"**, che sceglie la classe più probabile tra quelle osservabili: Jaccard +0.010.
    È una riga separata, perché cambia il protocollo rispetto a DeepLoc.


12. *Una probabilità di presenza è una percentuale?* No. "Presente nel citoplasma: 0.95; nel nucleo: 0.60" non vuol
    dire 61% / 39%: le probabilità rispondono a domande diverse e non sommano a 1. Una percentuale è una
    **ripartizione delle copie** della proteina, e va imparata da misure di ripartizione. L'unica disponibile su scala
    proteomica in cellule umane è il frazionamento nucleo/citosol/organelli (Hein 2024 in HEK293T, Itzhak 2016 in
    HeLa). Il pool nucleare contiene anche i mitocondri (sedimentano insieme), quindi si usa solo la **frazione
    citosolica**, molto riproducibile tra i due esperimenti (r ≈ 0.90). Test su dati reali (1.580 proteine DeepLoc,
    out-of-fold): quota normalizzata Spearman 0.39 → testa dedicata 0.46. La rete ha quindi una **testa della
    frazione citosolica**; il resto (1 − ĉ) si ripartisce tra gli altri organelli secondo la presenza calibrata.
13. *Le probabilità sono tarate?* Le loss usate (canale di rumore, focal, ASL, pesi dei positivi) spostano le uscite.
    Si applica la scala di Platt per classe in cross-fitting sulle proteine umane, e si riporta l'ECE prima e dopo.
14. *"Tutto umano": basta il test?* Il training SwissProt è per circa 3/4 non umano. Le proteine non umane restano
    (insegnano i segnali di sorting conservati) ma con un peso in griglia ({0.3, 1}), e la selezione dei modelli si
    fa sulla macro-AUPRC delle sole proteine umane.
15. *Cosa vuol dire "senza sangue" nel cervello?* Proteine plasmatiche fatte dal fegato, proteine di globuli rossi e
    leucociti ed emoglobine compaiono nei proteomi cerebrali (perfusione, contaminazione) ma non sono localizzate
    dalle cellule nervose. Sono marcate dall'atlante HPA e tolte dalla tabella finale; le microglia restano.
16. *Il tessuto nervoso ha compartimenti in più?* Oltre ad assone, dendrite e sinapsi c'è la **mielina**, che però è
    degli oligodendrociti: si addestra solo sulle proteine espresse in quelle cellule.

17. *La frazione citosolica è scritta nella sequenza?* Solo in parte. Una proteina dentro un grande complesso sedimenta
    con esso: la frazione media delle altre sottounità CORUM ha da sola Spearman 0,61 con quella della proteina. Il
    contesto molecolare (complessi, interattori, abbondanza) entra in un secondo stadio: +0,05–0,08 di Spearman.
19. *Il nucleo si può fare come il citosol?* Sì, se si corregge l'artefatto (i mitocondri sedimentano con i nuclei):
    quota normalizzata 0,23 → testa dedicata 0,66 di Spearman sulle proteine non mitocondriali.
20. *Il "citoplasma" di DeepLoc è il citosol solubile?* No: comprende citoscheletro e grandi complessi, che nel
    frazionamento finiscono nel pool sedimentabile. Il citoplasma finale = citosol solubile + la sua parte del pool
    sedimentabile. Per le ~8.000 proteine misurate le quote di citosol e nucleo sono le misure, non predizioni.

21. *Dove trovo etichette umane misurate senza un laboratorio?* Nella proteomica spaziale pubblica: tre mappe LOPIT
    (U2OS 2018 ×2, 2024) e la mappa BioID di Go 2021. Il notebook 01 le trasforma in osservazioni (classificatore sui
    marcatori curati); concordanza con DeepLoc 0,76–0,80 sulle proteine non marcatrici. Entrano nel canale di rumore,
    che ne stima sensibilità e specificità.
22. *Come si disegna la ROC dello stato dell'arte se DeepLoc 2.0 pubblica solo metriche a soglia?* Riproducendone la
    ricetta sugli stessi dati e partizioni (stadio `stile_deeploc2`) e, se disponibili, con le sue predizioni
    ufficiali scaricate dal server (sezione 9). Così la differenza misura il metodo, non i dati.
23. *Come si sceglie la metodologia senza barare?* Sulla CV (macro-AUPRC out-of-fold); il test con bootstrap appaiato
    conferma la scelta e dice se la distanza dallo stato dell'arte è più grande del rumore. Le ablazioni dicono
    quanto vale ogni ingrediente.

24. *Perché il lisosoma è la classe peggiore?* Solo l'8% delle lisosomiali è messo al primo posto; il 38% finisce in
    membrana plasmatica e il 20% tra le extracellulari. Le idrolasi lisosomiali usano lo stesso peptide segnale delle
    proteine secrete: il loro segnale vero è il mannosio-6-fosfato, aggiunto su una "toppa" di lisine della proteina
    **ripiegata** (conformazionale, non lineare). → Aiutano la famiglia (embedding + k-NN), le misure (LOPIT, BioID) e,
    in prospettiva, la struttura 3D (AlphaFold).
25. *E il Golgi?* 23% → citoplasma, 23% → RE. Molte proteine del Golgi sono periferiche (golgine, GTPasi Rab/Arf): ci
    arrivano legandosi a un partner, senza segnale proprio. Quelle di membrana si distinguono da RE e membrana per la
    lunghezza del TM e per motivi brevi in coda. → punteggi di motivo terminali e partner molecolari (sezione 8c).
26. *Il perossisoma non ha un segnale chiaro (PTS1)?* Solo il 39% delle perossisomiali ha un PTS1 canonico (le altre:
    PTS2, import "a cavalcioni" di un partner, proteine di membrana via PEX19); il 33% finisce nel citoplasma. Il
    contesto a monte del tripeptide conta, come sanno i predittori PTS1 dedicati. → matrici posizione-specifiche
    sugli ultimi 12 residui.
27. *E il RE?* KDEL c'è solo nel 4% delle proteine del RE: quelle di membrana sono trattenute da TM corti, code KKXX o
    partner; il 23% finisce in membrana plasmatica.
28. *Cambiare la loss o l'ottimizzatore aiuta?* No. ASL, logit adjustment (dal riconoscimento a coda lunga nelle
    immagini) e SAM (ottimizzazione "piatta", dalla visione) restano entro ±0,003, sotto il rumore tra semi (±0,004).
    Il limite è l'informazione, non l'addestramento.
29. *Cosa funziona davvero?* (macro-AUPRC sul test)
    - più capacità (rete larga e profonda): 0,525 → 0,545 → nella griglia anche 2048 neuroni;
    - **punteggi di motivo terminali** (log-odds posizione-specifici, stimati fuori fold): 0,545 → 0,557, Golgi
      0,15 → 0,20 → blocco `motivi`, sempre accanto ai descrittori;
    - **seconda famiglia di modelli** (gradient boosting) 0,552 e **ensemble** scelto in CV 0,563 → sezione 8b;
    - **partner molecolari**: 0,53 → 0,61 sulle proteine con partner → sezione 8c (riga separata: non è solo sequenza);
    - **fusione con le misure** (LOPIT, BioID): 0,51 → 0,78 sulle proteine misurate → tabella del cervello (12c-bis).
    Scartati: one-hot posizionale grezzo dei terminali (sovra-adatta), k-NN su 3-meri (senza embedding l'omologia
    residua è troppo bassa).
30. *Un modello sui singoli residui (convoluzioni + attention pooling, "Light Attention") vede meglio i segnali corti?*
    Addestrato da zero su one-hot, no: 0,43 contro 0,52 della rete sui descrittori (un fold, stesse condizioni), perché
    ~8.000 sequenze non bastano per imparare la grammatica da zero. In letteratura l'attention pooling funziona sugli
    embedding per-residuo **pre-addestrati** (+2–4 punti su DeepLoc), che però richiedono ~40 GB: resta un'opzione
    futura, non entra nei notebook.

31. *Cosa vuol dire "solo cervello" se le localizzazioni sono misurate in cellule di laboratorio?* Il perimetro è
    fatto dalle **proteine**: umane, espresse nel cervello (tessuti, neuroni o glia), non del sangue. Le etichette
    vengono da esperimenti umani anche su linee cellulari, perché i segnali degli organelli sono universali e misure
    su larga scala nel tessuto cerebrale non esistono. I compartimenti del neurone e la mielina vengono da fonti
    neuronali (SynGO, UniProt) e sono addestrati solo sulle proteine dei neuroni o degli oligodendrociti.
32. *Quanto costa togliere le altre specie?* Test reale (DeepLoc 1, test umano): solo 2.200 proteine umane danno 0,46
    di macro-AUPRC, tutte le specie con l'uomo pesato di più 0,51. Qui il costo è minore: le proteine umane con
    etichette sono molte di più (DeepLoc umano + HPA, UniProt, OpenCell, LOPIT, BioID).
35. *Quale modello per il testo?* Un modello di frasi biomediche (PubMedBERT addestrato su coppie di frasi di PubMed,
    768 numeri): i modelli generici capiscono peggio "acid hydrolase" o "beta-oxidation". Le schede lunghe si leggono
    a blocchi da 512 token.
36. *Perché i pezzi iniziali, centrali e finali?* La media su tutta la proteina diluisce segnali di 3–30 residui. Le
    finestre seguono la biologia: peptide segnale (15–30 aa) e presequenza mitocondriale (20–60) all'inizio; PTS1,
    KDEL, KKXX, ligandi PDZ negli ultimi 3–15 residui; code citoplasmatiche con YxxΦ e dileucine negli ultimi 50;
    domini ed eliche TM nel terzo centrale.
37. *Come si uniscono sequenza e testo?* Concatenando tutto, i 20.000 numeri degli embedding di sequenza schiacciano
    i 768 della scheda. Qui ogni blocco diventa un token della stessa dimensione (proiezione + LayerNorm), e con
    `fusione = 2` un piccolo Transformer li fa interagire, come nei modelli multimodali. Il dropout di blocco
    ("modality dropout", ModDrop) insegna alla rete a funzionare anche con schede povere o assenti. La fusione
    tardiva (media con gli alberi, sezione 8b) resta come alternativa scelta in CV.

In [ ]:
import copy
import warnings
import re
from itertools import combinations, product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import rankdata, spearmanr
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, f1_score, jaccard_score, matthews_corrcoef,
                             precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
# tqdm compatibile notebook/terminale
from tqdm.auto import tqdm

DATA = Path('data')
ESM  = 'facebook/esm2_t33_650M_UR50D'
T5   = 'Rostlab/prot_t5_xl_half_uniref50-enc'
USE_T5 = True

# DEVICE / DTYPE / TOKENS / BATCH: derivati dalla cella HW (cella 0).
# Se si salta la cella 0, i valori predefiniti sono conservativi (CPU).
if 'DEVICE' not in dir():
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if 'DTYPE' not in dir():
    DTYPE  = torch.float32
if 'TOKENS' not in dir():
    TOKENS = [2000, 6000][torch.cuda.is_available()]
if 'BATCH' not in dir():
    BATCH  = 128
if 'SEED' not in dir():
    SEED   = 42
    torch.manual_seed(SEED)

CLASSES = ['Cytoplasm', 'Nucleus', 'Extracellular', 'Cell membrane', 'Mitochondrion',
           'Endoplasmic reticulum', 'Lysosome/Vacuole', 'Golgi apparatus', 'Peroxisome']
NEURO = ['Assone', 'Dendrite', 'Presinapsi', 'Postsinapsi', 'Mielina']
ESCLUDI_SANGUE = True
FRAC_W = 0.5
ALL = CLASSES + NEURO
HPA_EVAL = ['Cytoplasm', 'Nucleus', 'Cell membrane', 'Mitochondrion', 'Endoplasmic reticulum', 'Golgi apparatus']
MAX_RES, WIN = 1022, 15
SUMMARIES = ['mean', 'mid', 'max', 'n20', 'n60', 'centro', 'c15', 'c50']
NSUM = len(SUMMARIES)
MAX_VARIANTS = 4000
EMA_DECAY = 0.995
PATIENCE   = 6
warnings.filterwarnings('ignore', message='An input array is constant')
rng = np.random.default_rng(SEED)

proteins = pd.read_parquet(DATA / 'proteine.parquet')
obs      = pd.read_parquet(DATA / 'osservazioni.parquet')
cv       = pd.read_parquet(DATA / 'clinvar_missense.parquet')
lab_var  = pd.read_parquet(DATA / 'varianti_sperimentali.parquet')
AUX = list(proteins.columns[proteins.columns.str.startswith('aux_')])
CTX = list(proteins.columns[proteins.columns.str.startswith('ctx_')]) + ['brain_ntpm', 'neuron_ncpm', 'glia_ncpm']
print(proteins['source'].value_counts().to_string(), '\nTarget ausiliari:', AUX)


## 1 · Sequenze da codificare: proteine + mutanti
per le varianti:oltre 1022 aa si tengono i due terminali

In [ ]:
seq_of = proteins.set_index('pid')['sequence']
L_wt   = cv['pid'].map(seq_of.str.len())
cv = cv[(L_wt <= MAX_RES) | (cv['pos'] <= MAX_RES // 2) | (cv['pos'] > L_wt - MAX_RES // 2)].copy()
cv['priority'] = cv['clin'].map({'P/LP': 0, 'B/LB': 0, 'VUS': 1}) + rng.random(len(cv))
cv = cv.sort_values('priority').head(MAX_VARIANTS).reset_index(drop=True)
L_wt    = cv['pid'].map(seq_of.str.len())
cv['pos_t'] = np.where((L_wt <= MAX_RES) | (cv['pos'] <= MAX_RES // 2),
                        cv['pos'], cv['pos'] - L_wt + MAX_RES)
wt           = cv['pid'].map(seq_of)
cv['mut_sequence'] = [s[:p - 1] + m + s[p:] for s, p, m in zip(wt, cv['pos'], cv['mt'])]

seqs   = pd.concat([proteins['sequence'], cv['mut_sequence']], ignore_index=True)
N_PROT = len(proteins)
cv['row'] = np.arange(N_PROT, len(seqs))
center    = np.full(len(seqs), -10 ** 6)
center[cv['row']] = cv['pos_t']
need_res  = np.zeros(len(seqs), bool)
need_res[cv['pid'].unique()] = True
# Troncamento biologicamente motivato: conserva N-term (segnali di sorting) e C-term (PTS1, KDEL, PDZ)
trunc   = seqs.where(seqs.str.len() <= MAX_RES,
                     seqs.str[:MAX_RES // 2] + seqs.str[-(MAX_RES // 2):])
lengths = trunc.str.len()
print(f'Sequenze: {len(seqs):,} ({N_PROT:,} proteine + {len(cv):,} mutanti)')


In [ ]:
import hashlib
import shutil

impronta = hashlib.sha256('\n'.join(list(seqs) + list(proteins['uniprot_id'].fillna('')) + list(proteins['card_protein'].fillna(''))).encode()).hexdigest()
f_impronta = DATA / 'impronta_embedding.txt'
salvata = ''.join(p.read_text().strip() for p in DATA.glob('impronta_embedding.txt'))
vecchi = (sorted(DATA.glob('emb_*')) + sorted(DATA.glob('res_*'))) * int(salvata != impronta)
dest = DATA / 'cache_vecchia' / pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
for p in vecchi:
    dest.mkdir(parents=True, exist_ok=True)
    shutil.move(str(p), str(dest / p.name))
f_impronta.write_text(impronta)
print(f"Impronta delle sequenze: {impronta[:16]}… | cache degli embedding: "
      f"{['valida, si riusa', f'non corrisponde, {len(vecchi)} file o cartelle spostati in {dest}: si ricalcola'][int(salvata != impronta)]}")


## 2a · Embedding ESM-2 (ripartibile)
Per ogni sequenza, 8 riassunti (`E`), cioè la proteina intera e i suoi **pezzi**:
- intera: media dell'ultimo layer, media del layer a 2/3, massimo;
- **inizio**: primi 20 residui (peptide segnale, presequenza mitocondriale, miristoilazione) e primi 60;
- **centro**: il terzo centrale (domini, NLS interni, eliche TM);
- **fine**: ultimi 15 residui (PTS1, KDEL, KKXX, ligandi PDZ, CAAX) e ultimi 50 (code citoplasmatiche, YxxΦ).
Le finestre seguono le lunghezze tipiche dei segnali di smistamento. Più la media locale ±15 residui attorno alla
variante (`LOC`, solo mutanti).
Per le proteine WT con varianti si salvano anche l'embedding per-residuo e i log-probabilità (vista locale e LLR).
Se cambia l'elenco delle sequenze la cache si rinnova da sola (sezione 1b). I commenti dentro la cella che parlano di
GTX 1080 e di risultati identici "bit per bit" valgono per la versione in fp32: qui il calcolo è in fp16 su RTX A5000.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoTokenizer, EsmModel, EsmForMaskedLM

# --- Tokenizzatore e modello encoder puro (no LM head) ---
tok = AutoTokenizer.from_pretrained(ESM)
esm_enc = EsmModel.from_pretrained(ESM, torch_dtype=DTYPE).to(DEVICE).eval()
D   = esm_enc.config.hidden_size                              # 1280 per ESM-2 650M
MID = round(esm_enc.config.num_hidden_layers * 2 / 3)        # layer ~22/33
tag = ESM.split('/')[-1]
AA     = 'ACDEFGHIKLMNPQRSTVWY'
AA_IDS = tok.convert_tokens_to_ids(list(AA))
RES    = DATA / f'res_{tag}'
RES.mkdir(exist_ok=True)

# --- Memmap ripartibile ---
paths = {k: DATA / f'emb_{tag}_{k}8.npy' for k in ['E', 'LOC', 'DONE']}
E    = np.lib.format.open_memmap(paths['E'],    mode=['w+', 'r+'][paths['E'].exists()],
                                  dtype=np.float16, shape=(len(seqs), NSUM * D))
LOC  = np.lib.format.open_memmap(paths['LOC'],  mode=['w+', 'r+'][paths['LOC'].exists()],
                                  dtype=np.float16, shape=(len(seqs), D))
DONE = np.lib.format.open_memmap(paths['DONE'], mode=['w+', 'r+'][paths['DONE'].exists()],
                                  dtype=bool, shape=(len(seqs),))

todo = lengths[~DONE[:]].sort_values()
print(f'ESM-2 da calcolare: {len(todo):,}')
if not len(todo):
    print('Tutti gli embedding ESM-2 già presenti.')
else:
    # --- OPT 1: hook sul layer MID ---
    # Invece di output_hidden_states=True (33 tensori in VRAM) usiamo un forward hook
    # che cattura solo il layer MID senza materializzare gli altri 32.
    _mid_buf: dict = {}
    def _mid_hook(module, inp, out_t):
        _mid_buf['h'] = out_t[0]   # out_t = (hidden_state, ...) per TransformerLayer ESM
    _hh = esm_enc.encoder.layer[MID].register_forward_hook(_mid_hook)

    # --- Funzione di tokenizzazione eseguita in thread (OPT 6: prefetch) ---
    def _tok_batch(seq_list):
        return tok(seq_list, return_tensors='pt', padding=True)

    # --- Parametri ottimizzati ---
    BAND_W      = 32          # OPT 3: bande più strette → meno padding medio
    FLUSH_EVERY = 20          # OPT 5: flush ogni 20 bande
    _fc = 0
    _t0 = time.perf_counter()

    bands = list(todo.groupby(todo // BAND_W))

    with tqdm(total=len(todo), desc='ESM-2 encoding', unit='seq',
              ncols=100, dynamic_ncols=True) as pbar, \
         ThreadPoolExecutor(max_workers=1) as _pool:

        # Pre-avvia tokenizzazione del primo batch
        _nxt_enc = None
        if bands:
            _bs0 = max(1, TOKENS // int(bands[0][1].max() + 2))
            _nxt_enc = _pool.submit(_tok_batch, trunc[bands[0][1].index[:_bs0]].tolist())

        for bi, (_, band) in enumerate(bands):
            bs = max(1, TOKENS // int(band.max() + 2))
            for s0 in range(0, len(band), bs):
                idx = band.index[s0:s0 + bs].to_numpy()

                # Recupera batch pre-tokenizzato
                enc_cpu = _nxt_enc.result() if _nxt_enc is not None else _tok_batch(trunc[idx].tolist())

                # Pre-avvia tokenizzazione del batch successivo (OPT 6)
                ns = s0 + bs
                if ns < len(band):
                    _nxt_enc = _pool.submit(_tok_batch, trunc[band.index[ns:ns + bs]].tolist())
                elif bi + 1 < len(bands):
                    nb2 = bands[bi + 1][1]
                    bs2 = max(1, TOKENS // int(nb2.max() + 2))
                    _nxt_enc = _pool.submit(_tok_batch, trunc[nb2.index[:bs2]].tolist())
                else:
                    _nxt_enc = None

                # OPT 7: trasferimento su GPU non bloccante
                enc = {k: v.to(DEVICE, non_blocking=True) for k, v in enc_cpu.items()}
                torch.cuda.synchronize()   # assicura il trasferimento prima del forward

                # Forward senza output_hidden_states (hook cattura MID)
                with torch.no_grad():
                    out = esm_enc(**enc, output_hidden_states=False)

                last  = out.last_hidden_state.float()
                mid_h = _mid_buf['h'].float()

                # Maschere residui reali (BOS=pos0, EOS=posL+1 → escludiamo entrambi)
                S     = last.shape[1]
                L_tok = (enc['attention_mask'].sum(1, keepdim=True) - 2).long()
                pos_t = torch.arange(S, device=DEVICE)[None]           # [1, S]
                res   = ((pos_t >= 1) & (pos_t <= L_tok)).unsqueeze(-1) # [B, S, 1]
                r0    = (pos_t - 1).float()

                # 5 pezzi terminali (segnali di sorting biologici, invariati)
                Lt_f = L_tok.float()
                pieces = [
                    res & (r0 < 20).unsqueeze(-1),
                    res & (r0 < 60).unsqueeze(-1),
                    res & ((r0 >= Lt_f / 3) & (r0 < 2 * Lt_f / 3)).unsqueeze(-1),
                    res & (r0 >= Lt_f - 15).unsqueeze(-1),
                    res & (r0 >= Lt_f - 50).unsqueeze(-1),
                ]
                local = res & (
                    (pos_t - torch.tensor(center[idx], device=DEVICE)[:, None]).abs() <= WIN
                ).unsqueeze(-1)

                # OPT 4: conversione fp16 sulla GPU → elimina .astype() su CPU
                E[idx] = torch.cat(
                    [(last  * res).sum(1) / res.float().sum(1),
                     (mid_h * res).sum(1) / res.float().sum(1),
                     last.masked_fill(~res, -1e4).max(1).values] +
                    [(last * m).sum(1) / m.float().sum(1).clamp(min=1) for m in pieces],
                    dim=1
                ).to(torch.float16).cpu().numpy()

                LOC[idx] = (
                    (last * local).sum(1) / local.float().sum(1).clamp(min=1)
                ).to(torch.float16).cpu().numpy()

                DONE[idx] = True
                pbar.update(len(idx))

                # OPT 8: libera VRAM subito
                del last, mid_h, out, enc, pieces, local, res
                _mid_buf.clear()

            # OPT 5: flush ridotto (ogni 20 bande invece di ogni banda)
            _fc += 1
            if _fc % FLUSH_EVERY == 0:
                for mm in (E, LOC, DONE): mm.flush()

        for mm in (E, LOC, DONE): mm.flush()   # flush finale garantito

    _hh.remove()   # rimuovi hook per non interferire con usi successivi

    # -----------------------------------------------------------------------
    # OPT 2b — Mini-pass LLR: solo per le sequenze WT con varianti (need_res)
    # EsmForMaskedLM viene caricato DOPO aver scaricato esm_enc dalla VRAM:
    # sui ~7 GB liberi ESM-2 650M FP32 occupa ~2.5 GB → i due modelli insieme
    # supererebbero la VRAM disponibile. Il mini-pass riguarda ~3000 sequenze
    # (≈15% del totale) → tempo aggiuntivo trascurabile.
    # -----------------------------------------------------------------------
    llr_idx = np.flatnonzero(need_res)
    if len(llr_idx) > 0:
        print(f'Mini-pass LLR per {len(llr_idx):,} sequenze WT con varianti...')
        del esm_enc
        torch.cuda.empty_cache()

        esm_lm = EsmForMaskedLM.from_pretrained(ESM, torch_dtype=DTYPE).to(DEVICE).eval()
        llr_lens = lengths.iloc[llr_idx].sort_values()

        with tqdm(total=len(llr_idx), desc='LLR varianti', unit='seq',
                  ncols=100, dynamic_ncols=True) as pbar_llr:
            for _, llr_band in llr_lens.groupby(llr_lens // BAND_W):
                bs_l = max(1, TOKENS // int(llr_band.max() + 2))
                for s0 in range(0, len(llr_band), bs_l):
                    idx_l = llr_band.index[s0:s0 + bs_l].to_numpy()
                    enc_l = tok(trunc[idx_l].tolist(),
                                return_tensors='pt', padding=True).to(DEVICE)
                    with torch.no_grad():
                        out_l = esm_lm(**enc_l, output_hidden_states=True)
                    L_l = (enc_l['attention_mask'].sum(1) - 2).long()
                    last_l = out_l.hidden_states[-1].float()
                    logp   = out_l.logits.float().log_softmax(-1)[..., AA_IDS]
                    for jl in np.flatnonzero(need_res[idx_l]):
                        n = int(L_l[jl])
                        np.savez(RES / f'{idx_l[jl]}.npz',
                                 h=last_l[jl, 1:n+1].to(torch.float16).cpu().numpy(),
                                 lp=logp[jl, 1:n+1].cpu().numpy())
                    pbar_llr.update(len(idx_l))
                    del out_l, enc_l, last_l, logp
        del esm_lm
        torch.cuda.empty_cache()
        print('Mini-pass LLR completato.')
    else:
        del esm_enc
        torch.cuda.empty_cache()

    _elapsed = time.perf_counter() - _t0
    print(f'ESM-2 completo: {DONE[:].all()} — tempo totale: {_elapsed/60:.1f} min')


## 2b · Embedding ProtT5-XL (ripartibile)
ProtT5 è un encoder-decoder T5 addestrato su BFD e UniRef50: impara cose diverse da ESM-2, e in DeepLoc 2.0 è il migliore.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, Future

# INVARIATO: struttura memmap, dtype, shape
D5 = 1024
paths5  = {k: DATA / f'emb_prot_t5_{k}8.npy' for k in ['E', 'DONE']}
E5      = np.lib.format.open_memmap(paths5['E'],    mode=['w+', 'r+'][paths5['E'].exists()],
                                     dtype=np.float16, shape=(len(seqs), NSUM * D5))
DONE5   = np.lib.format.open_memmap(paths5['DONE'], mode=['w+', 'r+'][paths5['DONE'].exists()],
                                     dtype=bool, shape=(len(seqs),))

for _ in range(int(USE_T5)):
    from transformers import T5EncoderModel, T5Tokenizer  # richiede: pip install sentencepiece

    # INVARIATO: caricamento modello e tokenizzatore
    tok5 = T5Tokenizer.from_pretrained(T5, do_lower_case=False)
    t5   = T5EncoderModel.from_pretrained(T5, torch_dtype=DTYPE).to(DEVICE).eval()
    MID5 = round(t5.config.num_layers * 2 / 3)

    # INVARIATO: preprocessing ProtT5 — residui separati da spazio, U/Z/O/B → X
    # (ProtT5 non ha token per questi amminoacidi rari/ambigui)
    spaced = trunc.str.replace(r'[UZOB]', 'X', regex=True).map(' '.join)

    todo5 = lengths[~DONE5[:]].sort_values()
    print(f'ProtT5 da calcolare: {len(todo5):,}')
    if not len(todo5):
        print('Tutti gli embedding ProtT5 già presenti.')
        del t5
        break

    # INVARIATO: hook sul layer MID5 (T5Block restituisce una tupla;
    # il primo elemento è l'hidden state del layer)
    _mid5_buf: dict = {}
    def _mid5_hook(module, inp, out_t):
        _mid5_buf['h'] = out_t[0]
    _hh5 = t5.encoder.block[MID5].register_forward_hook(_mid5_hook)

    # OPT 4: funzione di tokenizzazione eseguita in thread separato
    # (sovrappone tokenizzazione CPU con trasferimento e forward GPU)
    def _tok5_batch(seq_list):
        return tok5(seq_list, return_tensors='pt', padding=True)

    # OPT 2: helper per ottenere i parametri del batch di una banda+offset
    # Pre-calcola bs5 in modo deterministico e senza overhead nel loop
    def _band5_slice(bi, s0_override=0):
        """Restituisce (idx_array, seq_list) per la banda bi a partire da s0_override."""
        bd  = bands5[bi][1]
        bs  = max(1, TOKENS // int(bd.max() + 1))
        idx = bd.index[s0_override:s0_override + bs].to_numpy()
        return idx, spaced[idx].tolist()

    BAND_W5 = 32          # INVARIATO: bande strette → padding medio ridotto

    # OPT 7: flush adattivo basato sul tempo trascorso
    # Su SSD i flush frequenti non costano molto, su HDD ogni flush
    # sincronizza la testina: meglio farne uno ogni ~60 s di elaborazione.
    FLUSH_INTERVAL_S = 60.0   # secondi di wall-time tra flush successivi
    _t0           = time.perf_counter()
    _t_last_flush = _t0

    bands5 = list(todo5.groupby(todo5 // BAND_W5))

    with tqdm(total=len(todo5), desc='ProtT5 encoding', unit='seq',
              ncols=100, dynamic_ncols=True) as pbar5, \
         ThreadPoolExecutor(max_workers=1) as _pool5:

        # OPT 4+5: triplo buffer
        # _nxt5   → Future del batch corrente (già in tokenizzazione)
        # _nxt5_2 → Future del batch successivo+1 (avviato il prima possibile)
        _nxt5:   Future | None = None
        _nxt5_2: Future | None = None

        # Pre-avvia tokenizzazione del primo batch
        if bands5:
            _idx0, _seqs0 = _band5_slice(0)
            _nxt5 = _pool5.submit(_tok5_batch, _seqs0)
            # Pre-avvia subito anche il secondo batch se esiste
            if len(bands5[0][1]) > len(_idx0):
                # secondo slice nella stessa banda 0
                _bs5_0 = len(_idx0)
                _, _seqs0_2 = _band5_slice(0, s0_override=_bs5_0)
                if _seqs0_2:
                    _nxt5_2 = _pool5.submit(_tok5_batch, _seqs0_2)
            elif len(bands5) > 1:
                _, _seqs1 = _band5_slice(1)
                _nxt5_2 = _pool5.submit(_tok5_batch, _seqs1)

        for bi5, (_, band5) in enumerate(bands5):
            # OPT 2: bs5 calcolato una volta per banda, non ricalcolato ad ogni step
            bs5 = max(1, TOKENS // int(band5.max() + 1))

            for s0 in range(0, len(band5), bs5):
                idx5 = band5.index[s0:s0 + bs5].to_numpy()

                # OPT 4: recupera il batch pre-tokenizzato dal Future
                enc5_cpu = _nxt5.result() if _nxt5 is not None else _tok5_batch(spaced[idx5].tolist())

                # Ruota il buffer: _nxt5_2 diventa il prossimo _nxt5
                _nxt5   = _nxt5_2
                _nxt5_2 = None

                # OPT 4: avvia immediatamente la tokenizzazione del batch successivo+1
                # (batch+2 dal punto di vista del loop corrente), così mentre
                # trasferimento e forward occupano GPU, la CPU tokenizza già due
                # batch in anticipo.
                ns5 = s0 + bs5
                if ns5 < len(band5):
                    ns5_2 = ns5 + bs5
                    if ns5_2 < len(band5):
                        # terzo slice nella stessa banda
                        _nxt5_2 = _pool5.submit(
                            _tok5_batch, spaced[band5.index[ns5_2:ns5_2 + bs5]].tolist())
                    elif bi5 + 1 < len(bands5):
                        # primo slice della banda successiva
                        nb5_2 = bands5[bi5 + 1][1]
                        bs5_2 = max(1, TOKENS // int(nb5_2.max() + 1))
                        _nxt5_2 = _pool5.submit(
                            _tok5_batch, spaced[nb5_2.index[:bs5_2]].tolist())
                elif bi5 + 1 < len(bands5):
                    nb5_2 = bands5[bi5 + 1][1]
                    bs5_2 = max(1, TOKENS // int(nb5_2.max() + 1))
                    ns5_2 = bs5_2   # secondo slice della banda successiva
                    if ns5_2 < len(nb5_2):
                        _nxt5_2 = _pool5.submit(
                            _tok5_batch, spaced[nb5_2.index[ns5_2:ns5_2 + bs5_2]].tolist())
                    elif bi5 + 2 < len(bands5):
                        nb5_3 = bands5[bi5 + 2][1]
                        bs5_3 = max(1, TOKENS // int(nb5_3.max() + 1))
                        _nxt5_2 = _pool5.submit(
                            _tok5_batch, spaced[nb5_3.index[:bs5_3]].tolist())

                # OPT 1 (posizionamento ottimale di synchronize):
                # Il trasferimento GPU viene avviato ORA, DOPO aver già
                # lanciato il prefetch del prossimo batch sul thread CPU.
                # synchronize() blocca solo fino al completamento del trasferimento,
                # ma nel frattempo il thread CPU tokenizza già il batch+2.
                enc5 = {k: v.to(DEVICE, non_blocking=True) for k, v in enc5_cpu.items()}
                del enc5_cpu   # libera RAM CPU immediatamente
                torch.cuda.synchronize()   # garantisce trasferimento completo prima del forward

                # OPT 8: gestione eccezioni con ripristino stato — se un batch
                # fallisce (OOM, NaN, corruzione) puliamo lo stato e saltiamo
                # senza interrompere l'intera elaborazione
                try:
                    # INVARIATO: forward pass con output_hidden_states=False
                    # (hook cattura MID5 senza materializzare tutti gli strati)
                    with torch.no_grad():
                        out5 = t5(**enc5, output_hidden_states=False)

                    # INVARIATO: logica di mascheratura ProtT5
                    # ProtT5 non ha BOS: il tokenizzatore aggiunge solo </s> alla fine
                    # → L = somma mask - 1 (esclude il solo token speciale finale)
                    S5   = out5.last_hidden_state.shape[1]
                    L5   = (enc5['attention_mask'].sum(1, keepdim=True) - 1).long()
                    pos5 = torch.arange(S5, device=DEVICE)[None]   # [1, S]
                    res5 = (pos5 < L5).unsqueeze(-1)                # [B, S, 1]
                    L5_f = L5.float()

                    # INVARIATO: conversione float32 per stabilità numerica del pooling
                    last5  = out5.last_hidden_state.float()
                    mid5_h = _mid5_buf['h'].float()

                    # OPT 6: res5_f calcolato UNA SOLA VOLTA e riusato in tutto il blocco
                    # (nell'originale .float() su res5 veniva chiamato 3 volte separatamente)
                    res5_f = res5.float()   # [B, S, 1] float32

                    # INVARIATO: maschere per i 5 sotto-segmenti terminali
                    # (segnali biologici di sorting: N-term, centro, C-term)
                    pieces5 = [
                        res5 & (pos5 < 20).unsqueeze(-1),
                        res5 & (pos5 < 60).unsqueeze(-1),
                        res5 & ((pos5 >= L5_f / 3) & (pos5 < 2 * L5_f / 3)).unsqueeze(-1),
                        res5 & (pos5 >= L5_f - 15).unsqueeze(-1),
                        res5 & (pos5 >= L5_f - 50).unsqueeze(-1),
                    ]

                    # INVARIATO: concatenazione degli 8 summary (mean last, mean mid,
                    # max last, + 5 segmenti terminali) → shape [B, 8*D5]
                    # OPT 6: usa res5_f pre-calcolato invece di res5.float() ripetuto
                    E5[idx5] = torch.cat(
                        [(last5  * res5).sum(1) / res5_f.sum(1),
                         (mid5_h * res5).sum(1) / res5_f.sum(1),
                         last5.masked_fill(~res5, -1e4).max(1).values] +
                        [(last5 * m).sum(1) / m.float().sum(1).clamp(min=1) for m in pieces5],
                        dim=1
                    ).to(torch.float16).cpu().numpy()  # INVARIATO: fp16 su GPU → RAM

                    DONE5[idx5] = True
                    pbar5.update(len(idx5))

                    # OPT 2: del aggressivo — tutte le variabili temporanee VRAM
                    # rilasciate immediatamente dopo l'uso
                    del last5, mid5_h, out5, enc5, pieces5, res5, res5_f, L5, pos5, L5_f

                except Exception as _e5:
                    # OPT 8: ripristino stato in caso di errore su un batch
                    # (es. OOM su sequenze molto lunghe)
                    # Non interrompiamo l'intera elaborazione: il batch viene
                    # saltato e DONE5[idx5] rimane False → sarà ricalcolato
                    # alla prossima esecuzione della cella.
                    print(f'  [WARN] Batch idx5[0:{len(idx5)}] fallito: {_e5!r}'
                          f' — saltato, verrà ricalcolato al prossimo run.')
                    try:
                        del out5
                    except NameError:
                        pass
                    try:
                        del enc5
                    except NameError:
                        pass
                finally:
                    # INVARIATO: pulizia buffer hook — sempre, anche in caso di errore
                    _mid5_buf.clear()
                    if DEVICE == 'cuda':
                        torch.cuda.empty_cache()  # OPT 2: libera VRAM dopo ogni batch

            # OPT 7: flush adattivo — flush solo se sono trascorsi
            # FLUSH_INTERVAL_S secondi dall'ultimo flush (invece di ogni N bande)
            _t_now = time.perf_counter()
            if _t_now - _t_last_flush >= FLUSH_INTERVAL_S:
                E5.flush(); DONE5.flush()
                _t_last_flush = _t_now

        # Flush finale garantito
        E5.flush(); DONE5.flush()

    # INVARIATO: rimozione hook per non interferire con usi successivi del modello
    _hh5.remove()
    _elapsed = time.perf_counter() - _t0
    print(f'ProtT5 completo: {DONE5[:].all()} — tempo: {_elapsed/60:.1f} min')
    del t5
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()  # OPT 2: pulizia finale VRAM


## 2c · Embedding delle card_protein (PubMedBERT)
La scheda testuale di ogni proteina viene letta da un modello di linguaggio biomedico addestrato su PubMed, che la riassume in 768 numeri.
Le schede lunghe si spezzano in blocchi da 512 token che si sovrappongono di 64; l'embedding finale è la media dei blocchi. 

Domande da esperto:
- *Il testo aiuta davvero?* In letteratura sì: SherLoc (testi PubMed + sequenza) e ProtST (sequenza + descrizioni
  UniProt, +2,5 punti di accuratezza su DeepLoc con le sole sonde lineari). Test reale su DeepLoc 1 umano con i soli
  nomi delle proteine: 0,51 → 0,55 di macro-AUPRC con i nomi ripuliti.
- *E il rischio di barare?* Con i nomi **non** ripuliti il guadagno raddoppia (0,51 → 0,59): metà era la risposta
  scritta nel nome ("Mitochondrial…"). ProtST metteva la localizzazione nel testo di pre-addestramento: i suoi numeri
  possono esserne gonfiati. Qui la scheda è ripulita nel notebook 01 e il confronto con DeepLoc si fa anche **senza
  scheda** (ablazione `abl_senza_card`).
- *E le proteine poco studiate, con schede povere?* Sono quelle per cui servono predizioni nuove. Il dropout di blocco
  spegne a caso la scheda durante l'addestramento: la rete impara a cavarsela con la sola sequenza.

In [ ]:
from transformers import AutoTokenizer, AutoModel

TEXT_MODEL = 'NeuML/pubmedbert-base-embeddings'
path_card  = DATA / 'emb_card.npy'
CARD_E     = np.zeros((N_PROT, 768), np.float32)

for _ in range(int(not path_card.exists())):
    tok_t  = AutoTokenizer.from_pretrained(TEXT_MODEL)
    bert   = AutoModel.from_pretrained(TEXT_MODEL).to(DEVICE).eval()
    cards  = proteins['card_protein'].fillna('').replace('', 'Unknown protein.').tolist()

    # BUG FIX A5: averaging dei chunk riscritta in modo robusto.
    # Il vecchio codice usava np.add.at + divisione per bincount locale al batch,
    # il che funzionava solo perché ogni batch di 32 proteine è indipendente.
    # Qui usiamo contatori espliciti per essere immuni a cambiamenti di batch_size.
    chunk_counts = np.zeros(N_PROT, np.int32)   # quanti chunk ha contribuito ogni proteina

    BERT_BATCH = 32
    with tqdm(range(0, N_PROT, BERT_BATCH), desc='PubMedBERT card',
              unit='batch', ncols=100, dynamic_ncols=True) as pbar:
        for start in pbar:
            enc = tok_t(
                cards[start:start + BERT_BATCH],
                truncation=True, max_length=512, stride=64,
                return_overflowing_tokens=True, padding=True,
                return_tensors='pt'
            )
            # owner: indice locale [0..BERT_BATCH-1] del campione di origine di ogni chunk
            owner_local = enc.pop('overflow_to_sample_mapping').numpy()
            owner_global = start + owner_local          # indice globale nella lista `cards`
            with torch.no_grad():
                h = bert(
                    input_ids=enc['input_ids'].to(DEVICE),
                    attention_mask=enc['attention_mask'].to(DEVICE)
                ).last_hidden_state
            m_att = enc['attention_mask'].to(DEVICE).unsqueeze(-1).float()
            # Media dei token (weighted mean pooling con attention mask)
            chunk_emb = ((h * m_att).sum(1) / m_att.sum(1)).cpu().numpy()  # [n_chunks, 768]
            # Accumula: ogni proteina riceve la somma dei chunk
            np.add.at(CARD_E, owner_global, chunk_emb)
            np.add.at(chunk_counts, owner_global, 1)

    # Normalizza: dividi per il numero di chunk per ogni proteina
    safe_counts = np.maximum(chunk_counts, 1)[:, None]   # evita divisione per 0
    CARD_E /= safe_counts

    np.save(path_card, CARD_E)
    del bert
    torch.cuda.empty_cache()

CARD_E = np.load(path_card)
print('card_protein lette:', CARD_E.shape)


## 2d · Struttura 3D da AlphaFold (descrittori geometrici + confidenza)


In [ ]:
import urllib.request
import urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.spatial.distance import cdist as _cdist

STRUCT = ['plddt_mean', 'plddt_std', 'plddt_frac_basso', 'plddt_frac_alto', 'plddt_nterm', 'plddt_cterm',
          'rg_norm', 'anisotropia', 'asfericita_canonica', 'densita_contatti', 'ordine_contatto',
          'idr_frazione', 'idr_nterm', 'idr_cterm', 'struttura_presente']
AFDIR      = DATA / 'alphafold'
AFDIR.mkdir(exist_ok=True)
path_struct = DATA / 'emb_struct.npy'
accs        = proteins['uniprot_id'].fillna('').to_numpy()
STRUCT_E    = np.full((N_PROT, len(STRUCT)), np.nan, np.float32)


def _parse_cif_atoms(cif_path: Path):
    """Estrae coordinate Cα e pLDDT da un file mmCIF AlphaFold.

    BUG FIX A1: legge l'header del loop_ per determinare gli indici delle colonne
    invece di usare indici hardcoded, rendendo il parsing robusto a future versioni.

    Returns
    -------
    xyz : list[list[float]] — coordinate [x, y, z] dei Cα
    plddt : list[float]     — pLDDT (B_iso) dei Cα
    """
    xyz, plddt = [], []
    if not (cif_path.exists() and cif_path.stat().st_size > 0):
        return xyz, plddt

    # Legge l'header del loop_ atom_site per mappare nomi → indici
    col_map: dict[str, int] = {}
    in_atom_loop = False
    col_idx = 0
    lines = cif_path.read_text(errors='ignore').splitlines()

    for line in lines:
        stripped = line.strip()
        if stripped == 'loop_':
            in_atom_loop = False
            col_map = {}
            col_idx = 0
        elif in_atom_loop and stripped.startswith('_atom_site.'):
            col_map[stripped] = col_idx
            col_idx += 1
        elif stripped.startswith('_atom_site.'):
            in_atom_loop = True
            col_map[stripped] = col_idx
            col_idx += 1
        elif in_atom_loop and stripped and not stripped.startswith('_') and not stripped.startswith('#'):
            # Siamo nella sezione dati del loop_
            break

    # Fallback sugli indici storici AlphaFold v4 se l'header non è stato trovato
    ix = col_map.get('_atom_site.Cartn_x', col_map.get('_atom_site.Cartn_x', 10))
    iy = col_map.get('_atom_site.Cartn_y', 11)
    iz = col_map.get('_atom_site.Cartn_z', 12)
    ib = col_map.get('_atom_site.B_iso_or_equiv', 14)
    i_grp    = col_map.get('_atom_site.group_PDB', 0)
    i_atom   = col_map.get('_atom_site.label_atom_id', 3)

    for line in lines:
        p = line.split()
        if len(p) > max(ix, iy, iz, ib) and p[i_grp] == 'ATOM' and p[i_atom] == 'CA':
            try:
                xyz.append([float(p[ix]), float(p[iy]), float(p[iz])])
                plddt.append(float(p[ib]))
            except (ValueError, IndexError):
                continue   # riga malformata: salta
    return xyz, plddt


def _compute_struct_features(xyz, plddt, n_atoms):
    """Calcola i descrittori geometrici da coordinate Cα e pLDDT.

    BUG FIX A2: usa scipy.spatial.distance.cdist invece del broadcasting O(n²) puro.
    BUG FIX C2: asfericitità calcolata con la formula canonica di Theodorou-Suter (1985).
    """
    r = np.array(xyz, np.float32)
    b = np.array(plddt, np.float32)
    n = len(r)
    c = r - r.mean(0)

    # Tensore di girazione
    gyr = (c.T @ c) / n
    ev  = np.sort(np.linalg.eigvalsh(gyr))   # λ1 ≤ λ2 ≤ λ3
    rg  = np.sqrt(ev.sum())

    # Anisotropia (kappa): 0 = sfera perfetta, 1 = asta
    kappa = 1.5 * (ev ** 2).sum() / (ev.sum() ** 2) - 0.5

    # Asfericitità canonica (Theodorou & Suter 1985):
    # delta = 1 - 3*(λ1λ2 + λ1λ3 + λ2λ3) / (λ1+λ2+λ3)²
    # BUG FIX C2: sostituisce la formula non standard ev[2] - (ev[0]+ev[1])/2
    lam_sum = ev.sum()
    lam_pairs_sum = ev[0]*ev[1] + ev[0]*ev[2] + ev[1]*ev[2]
    asfericita = 1.0 - 3.0 * lam_pairs_sum / (lam_sum ** 2 + 1e-12)

    # BUG FIX A2: cdist è 3-5x più veloce del broadcasting puro per n > 100
    d   = _cdist(c, c, 'euclidean')   # shape [n, n]
    sep = np.abs(np.arange(n)[:, None] - np.arange(n)[None, :])
    contatti = (d < 8.0) & (sep >= 3)
    n_cont   = contatti.sum() / 2
    idx_u    = np.triu_indices(n, k=1)
    lungo    = sep[idx_u][contatti[idx_u]]

    basso = b < 50
    # Corsa disordinata più lunga (loop O(n) in Python, ma n ≤ 1022)
    run = best = 0
    for v in basso:
        run  = run + 1 if v else 0
        best = max(best, run)

    return [
        b.mean(), b.std(), basso.mean(), (b >= 90).mean(),
        b[:30].mean(), b[-30:].mean(),
        rg / n ** 0.4, kappa, float(asfericita),
        n_cont / n,
        lungo.mean() / n if len(lungo) else 0.0,
        best / n, basso[:30].mean(), basso[-30:].mean(), 1.0
    ]


# Il download e il calcolo si fanno nella cella 3a. La vecchia fase di download di questa cella chiedeva i file
# `model_v4`, che AlphaFold DB non serve più: scriveva un file vuoto per proteina e nessuna struttura. Qui restano le
# funzioni di lettura e di calcolo dei descrittori, che la 3a usa
STRUCT_E = np.load(path_struct) if path_struct.exists() else STRUCT_E
STRUCT_E[:, -1] = np.nan_to_num(STRUCT_E[:, -1])
print(f'Strutture AlphaFold usate: {int(STRUCT_E[:, -1].sum()):,} / {N_PROT:,} proteine')


## 3 · Feature biologiche, struttura 3D recuperata, scheda ripulita e blocchi di input

41. *Perché la logistica si è fermata?* Tre proteine DeepLoc su 6.275 non hanno etichette: dopo la mappatura per
    sequenza due voci DeepLoc puntano allo stesso accession e le osservazioni vanno solo alla prima. Tre NaN bastano:
    la baseline diventa NaN, le matrici dei motivi (somme sulle etichette) diventano NaN per tutte le proteine e la
    logistica rifiuta l'input. Ora l'"oro" sono solo le proteine con l'etichetta presente.
42. *Una fonte senza positivi è informativa?* No, è dannosa. Nell'esecuzione reale MitoCarta mancava (il Broad blocca
    i download automatici), ma la regola delle assenze l'aveva trasformata in 14.624 "assente dal mitocondrio". Ora il
    notebook 01 legge MitoCarta e SynGO scaricati a mano (e GOA in automatico) e, se una fonte resta comunque senza
    positivi, la sezione 4 la toglie e lo dice. SynGO è l'aggiunta con il guadagno atteso più alto: senza, presinapsi e
    postsinapsi si imparano solo da UniProt.
44. *Le proteine aggiunte possono copiare il test?* Sì. Il 01 toglie solo le sequenze identiche al test, ma un paralogo
    al 90% con la sua annotazione HPA è quasi la risposta: il test HPA è annotato con lo stesso microscopio. Il coseno
    tra embedding medi non misura l'omologia (mediana test→training 0,956 anche tra proteine non imparentate). Qui si
    usava la quota di 5-meri condivisi, ma con identità p un 5-mero sopravvive con probabilità p⁵ (0,002 a p = 0,3):
    gli omologhi lontani sfuggivano. Ora i 5-meri e il coseno scelgono solo i candidati (le 3 e le 5 proteine più vicine)
    e un allineamento locale (Biopython, BLOSUM62) misura l'identità vera: omologhe = identità ≥ 30% su almeno metà della
    proteina più corta, la soglia delle partizioni di DeepLoc 2.0. Le proteine aggiunte omologhe a una del test restano
    predette ma non entrano nel training. Le aggiunte omologhe tra loro formano una famiglia e la famiglia va tutta nel
    fold in cui sta la maggior parte delle sue omologhe DeepLoc (il coseno resta solo per le famiglie che non ne hanno);
    un'aggiunta che resta con un'omologa DeepLoc in un altro fold esce dal training e dalle valutazioni fuori fold.
    Limite: è una ricerca tra candidati, non un confronto di tutti contro tutti come MMseqs2 o GraphPart, quindi qualche
    omologo lontano può ancora sfuggire; le partizioni interne di DeepLoc restano le sue.
45. *I punteggi di motivo erano davvero fuori fold?* Al primo ordine sì, al secondo no: le righe di training del fold k
    avevano punteggi stimati anche con le etichette del fold k (target encoding non annidato). Ora ogni fold esterno k
    ha le sue matrici: una riga della partizione j riceve punteggi stimati fuori da j e da k.

In [ ]:
# 3a · recupero delle strutture AlphaFold (DB v6) e ricalcolo dei descrittori con le funzioni della cella 2d
import functools
import gzip
import json
import os
from concurrent.futures import ThreadPoolExecutor
from itertools import compress

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

AF_THREADS = 4                                 # con 16 thread il server rifiutava quasi tutte le richieste (429)
http = requests.Session()
http.headers['User-Agent'] = 'neuroloc-delta/4.0 (academic research; UNISA)'
http.mount('https://', HTTPAdapter(pool_connections=AF_THREADS, pool_maxsize=AF_THREADS,
                                   max_retries=Retry(total=6, connect=1, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504],
                                                     respect_retry_after_header=True)))
AF_VER = 'v6'
try:
    AF_VER = f"v{http.get('https://alphafold.ebi.ac.uk/api/prediction/P04637', timeout=30).json()[0]['latestVersion']}"
except Exception as err:                       # rete assente o risposta inattesa: si prova con la v6
    print(f'API di AlphaFold DB non raggiungibile ({type(err).__name__}): uso {AF_VER}')
# se hai scaricato il proteoma intero (ftp.ebi.ac.uk/pub/databases/alphafold/latest/UP000005640_9606_HUMAN_v*.tar) ed
# estratto il tar in data/alphafold/, i file AF-<accession>-F1-model_v*.cif.gz si decomprimono con il nome atteso
af_gz = pd.Series(sorted(AFDIR.glob('AF-*-F1-model_v*.cif.gz')), dtype=object)
af_acc = af_gz.map(str).str.extract(r'AF-([^-]+)-F1-model', expand=False)
for f_gz, a_ in zip(af_gz[af_acc.isin(set(accs))], af_acc[af_acc.isin(set(accs))]):
    for dest in compress([AFDIR / f'{a_}.cif'], [not ((AFDIR / f'{a_}.cif').exists() and (AFDIR / f'{a_}.cif').stat().st_size > 0)]):
        dest.write_bytes(gzip.decompress(f_gz.read_bytes()))
tentati = AFDIR / f'assenti_{AF_VER}.txt'      # solo le accession con risposta 404: le altre (rifiutate, errori) si ritentano
tentati.touch()
pieno = np.array([p.exists() and p.stat().st_size > 0 for p in (AFDIR / f'{a}.cif' for a in accs)])
todo_af = np.flatnonzero(~pieno & (accs != '') & ~np.isin(accs, tentati.read_text().split()))
print(f'AlphaFold DB {AF_VER}: file già validi {pieno.sum():,}, da scaricare {len(todo_af):,}')
get = functools.partial(http.get, timeout=60)
esiti_af = pd.Series(0, index=[200, 404, 429, 500, 502, 503, 504])          # quante risposte per codice HTTP
try:
    with ThreadPoolExecutor(AF_THREADS) as pool:
        for ch in tqdm(np.array_split(todo_af, max(1, len(todo_af) // 256)), desc=f'AlphaFold {AF_VER}', unit='blocco'):
            resp = list(pool.map(get, [f'https://alphafold.ebi.ac.uk/files/AF-{accs[i]}-F1-model_{AF_VER}.cif' for i in ch]))
            codici = np.array([r.status_code for r in resp])
            for i, r in zip(compress(ch, codici == 200), compress(resp, codici == 200)):
                (AFDIR / f'{accs[i]}.cif').write_bytes(r.content)
            with open(tentati, 'a') as fh:
                fh.write(' '.join(accs[ch][codici == 404]) + ' ')
            esiti_af = esiti_af + pd.Series(codici).value_counts().reindex(esiti_af.index, fill_value=0)
except requests.RequestException as err:
    print(f'Download interrotto ({type(err).__name__}): riparte da qui al prossimo avvio')

# Cα e pLDDT con un'espressione regolare (10× più veloce del parser riga per riga); se il formato cambia e la regex non
# trova atomi, si ripiega sul parser della cella 2d, che legge l'intestazione delle colonne
CA_RE = re.compile(r'^ATOM\s+\d+\s+C\s+CA\s+\S+\s+\S+\s+\S+\s+\S+\s+(\d+)\s+\S+\s+(\S+)\s+(\S+)\s+(\S+)\s+\S+\s+(\S+)', re.M)
manca_s = np.flatnonzero(np.nan_to_num(STRUCT_E[:, -1]) == 0)
files = [AFDIR / f'{accs[i]}.cif' for i in manca_s]
leggibili = [p.exists() and p.stat().st_size > 0 for p in files]
sel = np.array(list(compress(manca_s, leggibili)), int)
ca = [np.array(CA_RE.findall(p.read_text(errors='ignore')), float).reshape(-1, 5) for p in compress(files, leggibili)]
for j in np.flatnonzero([len(a) < 10 for a in ca]):
    xyz_, b_ = _parse_cif_atoms(AFDIR / f'{accs[sel[j]]}.cif')
    ca[j] = np.c_[np.arange(1, len(b_) + 1), np.array(xyz_, float).reshape(-1, 3), np.array(b_, float)]
usa = np.flatnonzero([len(a) >= 10 for a in ca])
# i descrittori usano numpy/scipy (cdist, autovalori), che rilasciano il GIL: i thread lavorano in parallelo
with ThreadPoolExecutor(os.cpu_count()) as pool:
    feats = list(tqdm(pool.map(_compute_struct_features, [ca[j][:, 1:4] for j in usa], [ca[j][:, 4] for j in usa],
                               [len(ca[j]) for j in usa]), total=len(usa), desc='Descrittori 3D', unit='proteina'))
STRUCT_E[sel[usa]] = np.array(feats, np.float32).reshape(-1, len(STRUCT))
STRUCT_E[:, -1] = np.nan_to_num(STRUCT_E[:, -1])
np.save(path_struct, STRUCT_E)
print(f'Strutture AlphaFold usate: {int(STRUCT_E[:, -1].sum()):,} / {N_PROT:,} proteine (nuove in questa esecuzione: {len(usa):,})')
print('Risposte del server per codice HTTP:', esiti_af[esiti_af > 0].to_dict())
# se restano molte proteine senza struttura conviene il proteoma intero dal FTP (un solo file, nessun limite di richieste):
# ftp.ebi.ac.uk/pub/databases/alphafold/latest/UP000005640_9606_HUMAN_v6.tar → estrarlo in data/alphafold/ e rilanciare la cella
print(['', '>>> Meno del 50% delle proteine ha la struttura: scarica il tar del proteoma umano (vedi commento) e rilancia.'][
      int(STRUCT_E[:, -1].mean() < 0.5)])

Descrittori classici dei segnali di sorting (valgono anche per i mutanti). Le regex e le scale sono quelle della
letteratura citata nel secondo giro.

In [ ]:
# 3b · descrittori di sorting dalla sequenza
# la cella di ProtT5 salta un batch che fallisce e lascia le sue righe a zero: qui ci si ferma se un embedding manca
assert bool(np.all(DONE[:])), 'embedding ESM-2 incompleti: rilancia la cella 2a'
assert bool(np.all(DONE5[:])), 'embedding ProtT5 incompleti: rilancia la cella 2b'
KD = dict(zip(AA, [1.8, 2.5, -3.5, -3.5, 2.8, -0.4, -3.2, 4.5, -3.9, 3.8, 1.9, -3.5, -1.6, -3.5, -4.5, -0.8, -0.7, 4.2, -0.9, -1.3]))
n_len, nt, ct = seqs.str.len(), seqs.str[:30], seqs.str[-30:]
hand = pd.DataFrame({f'aa_{a}': seqs.str.count(a) / n_len for a in AA})
hand['log_len'] = np.log(n_len)
hand['nt_kd'] = sum(nt.str.count(a) * KD[a] for a in AA) / nt.str.len()
hand['nt_charge'] = (nt.str.count('[KR]') - nt.str.count('[DE]')) / 30      # presequenza mitocondriale: positiva
hand['nt_hydro_core'] = nt.str.contains(r'[AILMFVWC]{7,}').astype(int)     # nucleo idrofobico del peptide segnale
hand['tm_like'] = seqs.str.count(r'[^DEKRHNQPS]{17,}')
hand['nls'] = seqs.str.count(r'K[KR].[KR]') + seqs.str.count(r'[KR]{2}.{10,12}[KR]{3}')
hand['kdel'] = seqs.str.contains(r'[KRHQSA][DENQ]EL$').astype(int)
hand['kkxx'] = seqs.str.contains(r'K.K..$|KK..$').astype(int)
hand['pts1'] = seqs.str.contains(r'[SAC][KRH][LM]$').astype(int)
hand['caax'] = seqs.str.contains(r'C[AILMV][AILMV].$').astype(int)
hand['myr'] = seqs.str.contains(r'^MG[^EDRKHPFYW]..[STAGCN][^P]').astype(int)
hand['ct_kd'] = sum(ct.str.count(a) * KD[a] for a in AA) / ct.str.len()

# descrittori di secondo livello (domanda 9): un ciclo sulle sequenze, ~10 s
EIS = dict(zip(AA, [0.62, 0.29, -0.90, -0.74, 1.19, 0.48, -0.40, 1.38, -1.50, 1.06, 0.64, -0.78, 0.12, -0.85, -2.53, -0.18, -0.05, 1.08, 0.81, 0.26]))
helix = np.exp(1j * np.deg2rad(100) * np.arange(18))                  # 100° per residuo = elica α
sorting = []
for s in seqs:
    n = len(s)
    kd_s = np.array([KD.get(a, 0.0) for a in s])
    win = np.convolve(kd_s, np.ones(19) / 19, 'valid')                   # finestre di 19 residui (un'elica TM)
    covered = np.zeros(n, bool)
    for i0 in np.flatnonzero(win >= 1.6):
        covered[i0:i0 + 19] = True
    seg = list(re.finditer('1+', ''.join(np.where(covered, '1', '0'))))
    runs, starts = [m.end() - m.start() for m in seg], [m.start() for m in seg]
    s0, r0 = (starts + [n])[0], (runs + [0])[0]                          # primo segmento TM (n e 0 se non c'è)
    eis = np.array([EIS.get(a, 0.0) for a in s[:40]])
    sorting.append({'nt_muH': max(abs((eis[i0:i0 + 18] * helix[:len(eis[i0:i0 + 18])]).sum()) / 18 for i0 in range(max(1, len(eis) - 17))),
                    'nt_acid20': sum(a in 'DE' for a in s[:20]), 'nt_R20': s[:20].count('R'),
                    'tm_n': len(runs), 'tm_maxlen': max(runs + [0]), 'tm_meanlen': sum(runs) / max(len(runs), 1),
                    'tm_first_start': s0 / n, 'tail_anchor': int(len(runs) == 1 and n - (s0 + r0) < 30),
                    'signal_anchor': int(len(runs) >= 1 and s0 < 50),
                    'pos_inside': sum(a in 'KR' for a in s[max(0, s0 - 15):s0]) - sum(a in 'KR' for a in s[s0 + r0:s0 + r0 + 15])})
sorting = pd.DataFrame(sorting)
ct40, nt40 = seqs.str[-40:], seqs.str[:40]
sorting['yxxphi_ct'] = ct40.str.count(r'Y..[LIMFV]')                    # endosoma/lisosoma; somatodendritico nei neuroni
sorting['dileu_ct'] = ct40.str.count(r'[DE]...L[LI]')                    # dileucina acida
sorting['diacid_ct'] = ct40.str.count(r'D.E')                            # uscita dal RE
sorting['ff_ct'] = ct40.str.count('FF')
sorting['pts2'] = nt40.str.contains(r'[RK][LVIQ]..[LVIHQ][LSGAK].[HQ][LAF]').astype(int)
sorting['pts1_ext'] = seqs.str.contains(r'[SACKNP][KRH][LMI]$').astype(int)
sorting['nes'] = seqs.str.count(r'[LIVFM].{2,3}[LIVFM].{2,3}[LIVFM].[LIVFM]') / n_len * 100
sorting['pdz_I'] = seqs.str.contains(r'[ST].[VIL]$').astype(int)        # ligando PDZ classe I → densità postsinaptica
sorting['pdz_II'] = seqs.str.contains(r'[VLIFY].[VLIFY]$').astype(int)
sorting['nt_cys_palm'] = nt.str[:20].str.count('C')                     # palmitoilazione N-terminale (PSD-95, GAP-43)
sorting['rg_rich'] = seqs.str.count('RG|GR') / n_len * 100               # domini RGG (FUS, granuli di RNA)
sorting['glyc_sequon'] = seqs.str.count(r'N[^P][ST]') / n_len * 100
sorting['idr_like'] = seqs.str.count('[PESQK]') / n_len
sorting['ankG_bind'] = seqs.str.contains(r'V[PA][IL]A..E[SD]D').astype(int)  # ankirina-G, segmento iniziale dell'assone
hand = pd.concat([hand, sorting.set_index(hand.index)], axis=1).astype(np.float32)

In [ ]:
# 3c · seconda pulizia della card_protein e nuovo embedding (PubMedBERT, stesso metodo della cella 2c)
SCRUB2 = (r'(?i)synaptopathy gene \(synaptopathydb\):\s*(?:yes|no)\.?|\b(?:cell projections?|neuron projections?|junction\w*|'
          r'desmosom\w*|hemidesmosom\w*|focal adhesions?|adherens|coated pits?|centromer\w*|telomer\w*|sarcomer\w*|z lines?|'
          r'microsom\w*|phagosom\w*|melanosom\w*|autophagosom\w*|perikary\w*|nerve terminals?|signal-anchor|'
          r'signal(?! transduc| recogn))\b')
cards = proteins['card_protein'].fillna('').replace('', 'Unknown protein.')
cards2 = (cards.str.replace(SCRUB2, '', regex=True).str.replace(r'\s+([.,;:])', r'\1', regex=True)
          .str.replace(r'[;,]\s*\.', '.', regex=True).str.replace(r'\s+', ' ', regex=True))
# questo controllo dice solo che i termini della lista sono stati tolti; quanto la scheda dica ancora sulla localizzazione
# per via indiretta (funzione, complessi, processi) lo misura la sezione 4b
print(f"Schede con termini della lista di pulizia: {cards.str.contains(SCRUB2).mean():.1%} prima, "
      f"{cards2.str.contains(SCRUB2).mean():.1%} dopo la seconda pulizia")
path_card2 = DATA / 'emb_card_v2.npy'
for _ in range(int(not path_card2.exists())):
    tok_t = AutoTokenizer.from_pretrained(TEXT_MODEL)
    bert = AutoModel.from_pretrained(TEXT_MODEL).to(DEVICE).eval()
    CARD2 = np.zeros((N_PROT, 768), np.float32)
    n_chunk = np.zeros(N_PROT, np.float32)
    lista = cards2.tolist()
    for start in tqdm(range(0, N_PROT, 64), desc='PubMedBERT card v2', unit='batch'):
        enc = tok_t(lista[start:start + 64], truncation=True, max_length=512, stride=64, return_overflowing_tokens=True,
                    padding=True, return_tensors='pt')
        owner = start + enc.pop('overflow_to_sample_mapping').numpy()
        with torch.no_grad():
            m_att = enc['attention_mask'].to(DEVICE).unsqueeze(-1).float()
            h = bert(input_ids=enc['input_ids'].to(DEVICE), attention_mask=enc['attention_mask'].to(DEVICE)).last_hidden_state.float()
            np.add.at(CARD2, owner, ((h * m_att).sum(1) / m_att.sum(1)).cpu().numpy())
        np.add.at(n_chunk, owner, 1)
    np.save(path_card2, CARD2 / np.maximum(n_chunk, 1)[:, None])
    del bert
    torch.cuda.empty_cache()
CARD2 = np.load(path_card2)

# blocchi di input: ogni riassunto di ogni encoder è un blocco; i mutanti prendono scheda e struttura della WT
BLOCKS = {f'esm_{s}': E[:, i * D:(i + 1) * D] for i, s in enumerate(SUMMARIES)}
BLOCKS.update({f't5_{s}': E5[:, i * D5:(i + 1) * D5] for i, s in enumerate(SUMMARIES[:NSUM * USE_T5])})
BLOCKS['hand'] = hand.to_numpy(np.float16)
BLOCKS['card'] = np.vstack([CARD2, CARD2[cv['pid'].to_numpy()]]).astype(np.float16)
BLOCKS['struct'] = np.nan_to_num(np.vstack([STRUCT_E, STRUCT_E[cv['pid'].to_numpy()]])).astype(np.float16)
print({b: v.shape[1] for b, v in BLOCKS.items()})

## 4 · Target, omologia e partizioni

In [ ]:
from scipy import sparse
from sklearn.feature_extraction.text import HashingVectorizer

# osservazioni: solo classi note, solo fonti che riportano almeno un positivo (domanda 42)
noti = obs['classe'].isin(ALL) & obs['pid'].between(0, N_PROT - 1)
pos_fonte = obs[noti].groupby('fonte')['obs'].sum()
VUOTE = list(pos_fonte.index[pos_fonte == 0])
print(f'Osservazioni con classe sconosciuta: {(~noti).sum():,} | fonti senza positivi, tolte: {VUOTE}')
obs = obs[noti & ~obs['fonte'].isin(VUOTE)]
SOURCES = ['deeploc'] + sorted(set(obs['fonte']) - {'deeploc'})
S, C, nC = len(SOURCES), len(ALL), len(CLASSES)
OBS = np.full((N_PROT, S, C), np.nan, np.float32)
OBS[obs['pid'].to_numpy(), obs['fonte'].map({s: i for i, s in enumerate(SOURCES)}).to_numpy(),
    obs['classe'].map({c: i for i, c in enumerate(ALL)}).to_numpy()] = obs['obs'].to_numpy()
# una proteina con un'annotazione neuronale positiva resta "neuronale" anche se nel single-cell sta sotto la soglia
# (l'RNA dei neuroni è misurato con molti zeri): prima le sue annotazioni venivano cancellate
neuro_pos = np.nansum(OBS[:, :, nC:ALL.index('Mielina')], axis=(1, 2)) > 0
print(f"Proteine con un'annotazione neuronale ma sotto la soglia di espressione nei neuroni, tenute: "
      f"{int((neuro_pos & (proteins['is_neuronal'].to_numpy() == 0)).sum()):,}")
proteins['is_neuronal'] = (proteins['is_neuronal'].to_numpy().astype(bool) | neuro_pos).astype(int)
OBS[proteins['is_neuronal'].to_numpy() == 0, :, nC:ALL.index('Mielina')] = np.nan
mielina_pos = np.nansum(OBS[:, :, ALL.index('Mielina')], axis=1) > 0
proteins['is_oligo'] = (proteins['is_oligo'].to_numpy().astype(bool) | mielina_pos).astype(int)
OBS[proteins['is_oligo'].to_numpy() == 0, :, ALL.index('Mielina')] = np.nan

# superclassi della via di sorting (dalle etichette DeepLoc; NaN fuori da DeepLoc → fuori dalla loss)
SUPER = {'aux_via_secretoria': ['Extracellular', 'Cell membrane', 'Endoplasmic reticulum', 'Lysosome/Vacuole', 'Golgi apparatus'],
         'aux_nucleo_citoplasma': ['Cytoplasm', 'Nucleus'], 'aux_import_organelli': ['Mitochondrion', 'Peroxisome']}
for sup, members in SUPER.items():
    proteins[sup] = np.where(np.isnan(OBS[:, 0, 0]), np.nan, np.nan_to_num(OBS[:, 0, [ALL.index(m) for m in members]]).max(1))
AUX = list(proteins.columns[proteins.columns.str.startswith('aux_')])
A = len(AUX)
AUXY = proteins[AUX].to_numpy(np.float32)

src = proteins['source'].to_numpy()
part = proteins['partition'].to_numpy().astype(int).copy()
is_brain, is_neur = proteins['is_brain'].to_numpy().astype(bool), proteins['is_neuronal'].to_numpy().astype(bool)
is_oligo, is_blood = proteins['is_oligo'].to_numpy().astype(bool), proteins['is_blood'].to_numpy().astype(bool)
is_human = proteins['is_human'].to_numpy().astype(bool) | proteins['gene'].notna().to_numpy()
# frazioni misurate: il pool nucleare contiene anche i mitocondri, quindi per le proteine solo mitocondriali la quota
# nucleare vera è 0, per quelle anche nucleari è ignota
strong = np.isin(SOURCES, ['deeploc', 'uniprot_forte', 'hpa_alta', 'mitocarta'])
is_mito = np.nan_to_num(OBS[:, strong, CLASSES.index('Mitochondrion')]).max(1) == 1
is_nucl = np.nan_to_num(OBS[:, strong, CLASSES.index('Nucleus')]).max(1) == 1
FRAC = proteins[['frac_cyto', 'frac_nuc']].to_numpy(np.float32).copy()
FRAC[is_mito, 1] = np.where(is_nucl[is_mito], np.nan, 0.0)

GOLD = OBS[:, 0, :nC]
GOLD_M = ~np.isnan(GOLD)
gold_rows = np.flatnonzero((src == 'deeploc') & GOLD_M.all(1))           # domanda 41: solo proteine con etichetta
human_gold = gold_rows[is_human[gold_rows]]
extra_rows = np.flatnonzero(src == 'human_extra')
train_seq = set(proteins.loc[np.r_[np.flatnonzero(src == 'deeploc'), extra_rows], 'sequence'])
test_rows = np.flatnonzero((src == 'hpa_test') & ~proteins['sequence'].isin(train_seq).to_numpy())
Y_test = proteins.loc[test_rows].reindex(columns=CLASSES).fillna(0).to_numpy(np.float32)
print(f"Oro DeepLoc: {len(gold_rows):,} (senza etichetta, tolte: {(src == 'deeploc').sum() - len(gold_rows)}) | "
      f"test HPA: {len(test_rows):,} (identiche al training, tolte: {(src == 'hpa_test').sum() - len(test_rows)})")

# similarità per k-NN e ripiego delle partizioni: embedding medi dei due encoder, normalizzati e concatenati
unit = np.hstack([BLOCKS[b][:N_PROT].astype(np.float32) for b in ['esm_mean'] + ['t5_mean'] * USE_T5])
unit = np.hstack([u / (np.linalg.norm(u, axis=1, keepdims=True) + 1e-9) for u in np.split(unit, 1 + USE_T5, axis=1)])
unit /= np.linalg.norm(unit, axis=1, keepdims=True) + 1e-9

# omologia (domanda 44, rivista dopo la revisione). I candidati vengono da due filtri veloci: i 5-meri condivisi (omologhi
# vicini) e il coseno degli embedding medi (omologhi lontani, che al 30% di identità non condividono quasi nessun 5-mero).
# Poi un allineamento locale (BLOSUM62) misura l'identità vera. Omologhe = identità ≥ 30% su almeno metà della proteina
# più corta, la soglia con cui DeepLoc 2.0 costruisce le partizioni. Si allinea la sequenza che la rete vede davvero
# (oltre 1.022 residui: le due metà terminali)
from Bio import Align
from Bio.Align import substitution_matrices
ALN = Align.PairwiseAligner(mode='local', substitution_matrix=substitution_matrices.load('BLOSUM62'), open_gap_score=-11,
                            extend_gap_score=-1)
ID_MIN, COV_MIN, MIN_SHARED = 0.30, 0.50, 3
# candidati per ogni domanda da tre filtri: 5-meri condivisi (omologhi vicini), coseno degli embedding medi (omologhi
# lontani) e coseno dei profili di 3-meri pesati TF-IDF (somiglianza locale, che resiste sotto il 40% di identità, dove i
# 5-meri condivisi spariscono: è il principio del prefiltro di MMseqs2). Per il test la ricerca è molto più profonda:
# lì un'omologa sfuggita cambia i numeri da riportare. Numero di candidati: (5-meri, embedding, 3-meri)
# (la domanda aggiunte→test non ha 3-meri: per le fughe verso il test c'è la ricerca dal test, molto più profonda)
K_DOM = {'oro': (3, 5, 4), 'aggiunte': (4, 6, 4), 'test': (3, 5, 0), 'test_oro': (10, 20, 30), 'test_agg': (10, 20, 30)}
from sklearn.feature_extraction.text import TfidfTransformer
KM3 = TfidfTransformer(sublinear_tf=True).fit_transform(
    HashingVectorizer(analyzer='char', ngram_range=(3, 3), n_features=2 ** 16, alternate_sign=False, norm=None,
                      lowercase=False, dtype=np.float32).transform(proteins['sequence'])).astype(np.float32).tocsr()
KM = HashingVectorizer(analyzer='char', ngram_range=(5, 5), n_features=2 ** 22, alternate_sign=False, norm=None, binary=True,
                       lowercase=False, dtype=np.float32).transform(proteins['sequence'])
# i 5-meri presenti in più di 50 proteine (bassa complessità, poli-Q, poli-E) uniscono proteine non imparentate
KM = (KM @ sparse.diags((np.asarray(KM.sum(0)).ravel() <= 50).astype(np.float32))).tocsr()
KM.eliminate_zeros()
# BLOSUM62 non ha selenocisteina (U) e pirrolisina (O): ogni lettera fuori dal suo alfabeto diventa X
seq_al = trunc[:N_PROT].str.upper().str.replace('[^ARNDCQEGHILKMFPSTWYVBZX]', 'X', regex=True).to_numpy()
# coppie candidate per cinque domande: aggiunte→oro (partizione), aggiunte→aggiunte (famiglie tra le proteine aggiunte),
# aggiunte→test e test→aggiunte (fughe verso il test, cercate nei due versi), test→oro (strati del test). Tra le
# aggiunte il primo candidato è la proteina stessa: se ne prende uno in più
cand_h = []
for dom, Q, T in [('oro', extra_rows, gold_rows), ('aggiunte', extra_rows, extra_rows), ('test', extra_rows, test_rows),
                  ('test_oro', test_rows, gold_rows), ('test_agg', test_rows, extra_rows)]:
    KT, KT3 = KM[T].T.tocsc(), KM3[T].T.tocsc()
    k_km, k_cos, k_3m = K_DOM[dom]
    for c in np.array_split(np.arange(len(Q)), max(1, len(Q) // 1000)):
        sh = (KM[Q[c]] @ KT).toarray()
        top_k = np.argsort(-sh, 1)[:, :k_km]
        top_c = np.argsort(-(unit[Q[c]] @ unit[T].T), 1)[:, :k_cos]
        top_3 = np.argsort(-(KM3[Q[c]] @ KT3).toarray(), 1)[:, :k_3m]
        ok_k = (np.take_along_axis(sh, top_k, 1) >= MIN_SHARED).ravel()
        cand_h.append(pd.DataFrame({'domanda': dom, 'q': np.r_[np.repeat(Q[c], top_k.shape[1])[ok_k], np.repeat(Q[c], top_c.shape[1]),
                                                              np.repeat(Q[c], top_3.shape[1])],
                                    't': np.r_[T[top_k.ravel()[ok_k]], T[top_c.ravel()], T[top_3.ravel()]]}))
cand_h = pd.concat(cand_h, ignore_index=True)
cand_h = cand_h[cand_h['q'] != cand_h['t']]
# tra le aggiunte la coppia (a, b) e la coppia (b, a) sono la stessa: si allinea una volta sola
tra_agg = (cand_h['domanda'] == 'aggiunte').to_numpy()
q_h, t_h = cand_h['q'].to_numpy(), cand_h['t'].to_numpy()
cand_h = cand_h.assign(q=np.where(tra_agg, np.minimum(q_h, t_h), q_h), t=np.where(tra_agg, np.maximum(q_h, t_h), t_h)).drop_duplicates(ignore_index=True)
al_id, al_cov = np.zeros(len(cand_h)), np.zeros(len(cand_h))
for n_, (a_, b_) in enumerate(tqdm(cand_h[['q', 't']].to_numpy(), desc='allineamenti', unit='coppia')):
    ct_ = ALN.align(seq_al[a_], seq_al[b_])[0].counts()
    al_id[n_] = ct_.identities / max(ct_.identities + ct_.mismatches + ct_.gaps, 1)
    al_cov[n_] = (ct_.identities + ct_.mismatches) / min(len(seq_al[a_]), len(seq_al[b_]))
cand_h['identita'], cand_h['copertura'] = al_id, al_cov
cand_h['omologa'] = (cand_h['identita'] >= ID_MIN) & (cand_h['copertura'] >= COV_MIN)
cand_h.to_parquet(DATA / 'omologia_allineamenti.parquet', index=False)

# partizione delle aggiunte, per famiglie. Le aggiunte omologhe tra loro formano una famiglia (componenti connesse del
# grafo di omologia) e la famiglia va tutta nello stesso fold: quello in cui sta la maggior parte delle sue omologhe
# curate; senza omologhe curate, quello della proteina curata più vicina per embedding al primo membro
from scipy.sparse.csgraph import connected_components
om = cand_h[cand_h['omologa']]
ee = om[om['domanda'] == 'aggiunte']
n_fam, fam = connected_components(sparse.coo_matrix((np.ones(len(ee)), (ee['q'].to_numpy(), ee['t'].to_numpy())), shape=(N_PROT, N_PROT)),
                                  directed=False)
og = om[om['domanda'] == 'oro'].sort_values('identita', ascending=False)
og = og.assign(fam=fam[og['q'].to_numpy()], p_t=part[og['t'].to_numpy()])
part_fam = (og.groupby(['fam', 'p_t']).size().rename('n').reset_index().sort_values('n', ascending=False)
            .drop_duplicates('fam').set_index('fam')['p_t'])
near_cos = np.concatenate([(unit[c] @ unit[gold_rows].T).argmax(1) for c in np.array_split(extra_rows, max(1, len(extra_rows) // 2000))])
part_cos = pd.Series(part[gold_rows[near_cos]], index=fam[extra_rows]).groupby(level=0).first()
fam_x = pd.Series(fam[extra_rows])
part[extra_rows] = fam_x.map(part_fam).fillna(fam_x.map(part_cos)).astype(int).to_numpy()
# conflitti rimasti: un'aggiunta con un'omologa curata in un altro fold (la famiglia tocca più partizioni di DeepLoc).
# Non si può sistemare spostandola, quindi esce dal training (come le omologhe del test) e dalle valutazioni fuori fold:
# il modello che la predice ha visto la sua omologa
og = og.assign(p_q=part[og['q'].to_numpy()])
conflitto = og.loc[og['p_q'] != og['p_t'], 'q'].unique()
# omologhe di una proteina del test: si predicono ma non entrano mai nel training
leak_ids = np.union1d(om.loc[om['domanda'] == 'test', 'q'].unique(), om.loc[om['domanda'] == 'test_agg', 't'].unique()).astype(int)
train_ok = np.ones(N_PROT, bool)
train_ok[leak_ids] = False
train_ok[conflitto] = False
valutabile = np.ones(N_PROT, bool)                 # lo 03 valuta fuori fold solo queste
valutabile[conflitto] = False
# strati del test: con o senza un'omologa (≥ 30%) tra le proteine curate del training
test_hom = np.isin(test_rows, om.loc[om['domanda'] == 'test_oro', 'q'].to_numpy())
dim_fam = pd.Series(fam[extra_rows]).value_counts()
print(f'Allineamenti: {len(cand_h):,} | aggiunte con un\'omologa curata: {og["q"].nunique() / len(extra_rows):.1%} | '
      f'famiglie di aggiunte con più di un membro: {(dim_fam > 1).sum():,} (la più grande: {int(dim_fam.max()):,} proteine)')
print(f'Aggiunte fuori dal training: omologhe del test {len(leak_ids):,}, con un\'omologa curata in un altro fold {len(conflitto):,} '
      f'| coppie omologhe aggiunta–aggiunta in fold diversi: {int((part[ee["q"].to_numpy()] != part[ee["t"].to_numpy()]).sum())} '
      f'| test con un\'omologa curata (≥ {ID_MIN:.0%}): {test_hom.mean():.1%}')
print('Proteine di training per fold (curate + aggiunte):', pd.Series(part[np.r_[gold_rows, extra_rows][train_ok[np.r_[gold_rows, extra_rows]]]]).value_counts().sort_index().to_dict())
del KM, KT, KM3, KT3, sh

# il controllo obbligatorio è quello qui sopra, dentro il notebook. In più, facoltativa, la conferma tutti contro tutti con
# MMseqs2: si scrivono i due FASTA; se in data/esterni/ c'è il risultato, le omologhe che trova si APPLICANO (le aggiunte
# omologhe del test escono dal training, le proteine del test con un'omologa curata passano nello strato "con omologa").
# Comando:
#   mmseqs easy-search data/omologia_test.fasta data/omologia_training.fasta data/esterni/mmseqs_test_vs_training.m8 tmp \
#          --min-seq-id 0.3 -c 0.5 --cov-mode 2 -s 7.5
usate = np.flatnonzero(train_ok & np.isin(src, ['deeploc', 'human_extra']))
(DATA / 'omologia_test.fasta').write_text(''.join(f'>{i}\n{q}\n' for i, q in zip(test_rows, seq_al[test_rows])))
(DATA / 'omologia_training.fasta').write_text(''.join(f'>{i}\n{q}\n' for i, q in zip(usate, seq_al[usate])))
m8 = pd.concat([pd.read_csv(f, sep='\t', header=None).iloc[:, :3] for f in sorted((DATA / 'esterni').glob('mmseqs_test_vs_training*.m8'))] +
               [pd.DataFrame(columns=[0, 1, 2])], ignore_index=True).astype({0: int, 1: int, 2: float})
m8 = m8[m8[2] / [1, 100][int(m8[2].max() > 1.5)] >= ID_MIN]
m8_agg = np.intersect1d(m8[1].astype(int).to_numpy(), extra_rows)
nuove_fughe = np.setdiff1d(m8_agg, leak_ids)
leak_ids = np.union1d(leak_ids, m8_agg).astype(int)
train_ok[m8_agg] = False
test_hom = test_hom | np.isin(test_rows, m8.loc[np.isin(m8[1], gold_rows), 0].astype(int).to_numpy())
print(['MMseqs2: nessun file data/esterni/mmseqs_test_vs_training.m8 (conferma facoltativa: il controllo obbligatorio è quello sopra).',
       f'MMseqs2: aggiunte omologhe del test sfuggite al controllo interno e tolte ora dal training: {len(nuove_fughe):,} | '
       f'test con un\'omologa curata dopo MMseqs2: {test_hom.mean():.1%}'][
      int(len(sorted((DATA / 'esterni').glob('mmseqs_test_vs_training*.m8'))) > 0)])

# punteggi di motivo terminale annidati (domanda 45): per il fold esterno k, una riga della partizione j riceve
# log-odds posizione-specifici stimati sulle proteine oro fuori da j e da k (j = k e j = −1, il test, usano "fuori da k")
part_all = np.r_[part, part[cv['pid']]]
TERM_OH = [np.stack([(s_.str[j].to_numpy()[:, None] == np.array(list(AA))[None]) for j in range(n_)], 1).astype(np.float32)
           for n_, s_ in [(12, seqs.str[-12:].str.rjust(12, 'X')), (30, seqs.str[:30].str.ljust(30, 'X'))]]
PWM = {}                                        # matrici log-odds per fold: il notebook 03 le usa per le proteine nuove
for k in range(5):
    MOT = np.zeros((len(seqs), 2 * nC), np.float32)
    for j in [0, 1, 2, 3, 4, -1]:
        fit, app = gold_rows[~np.isin(part[gold_rows], [j, k])], np.flatnonzero(part_all == j)
        G = GOLD[fit]
        for t, oh in enumerate(TERM_OH):
            pos = (np.einsum('nc,njl->cjl', G, oh[fit]) + 0.5) / (G.sum(0)[:, None, None] + 10)
            neg = (np.einsum('nc,njl->cjl', 1 - G, oh[fit]) + 0.5) / ((1 - G).sum(0)[:, None, None] + 10)
            PWM[f'k{k}_j{j}_t{t}'] = np.log(pos / neg)
            MOT[app, t * nC:(t + 1) * nC] = np.einsum('njl,cjl->nc', oh[app], PWM[f'k{k}_j{j}_t{t}'])
    BLOCKS[f'motivi@{k}'] = MOT.astype(np.float16)
BLOCKS['motivi'] = BLOCKS['motivi@0']           # solo per larghezza ed elenco: ogni fold usa le sue matrici
del TERM_OH, MOT

sim_test = np.concatenate([(unit[c] @ unit[gold_rows].T).max(1) for c in np.array_split(test_rows, max(1, len(test_rows) // 1000))])
print(f'Coseno max test→training: mediana {np.median(sim_test):.3f} (non è identità di sequenza: vedi domanda 44)')
print(f'Proteine con frazione citosolica / nucleare misurata: {np.isfinite(FRAC).sum(0)}')
print(pd.DataFrame({'positivi': np.nansum(OBS, axis=(0, 2)), 'osservati': (~np.isnan(OBS)).sum((0, 2))}, index=SOURCES).astype(int))
print('Positivi neuronali per fonte:')
print(pd.DataFrame(np.nansum(OBS[:, :, nC:], 0).astype(int), index=SOURCES, columns=NEURO).to_string())

## 4b · Quanto dice la scheda testuale sulla localizzazione


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import chi2
from sklearn.model_selection import PredefinedSplit

vec_a = CountVectorizer(binary=True, min_df=20, stop_words='english', token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z-]{2,}\b')
X_a = vec_a.fit_transform(cards2.to_numpy()[human_gold])
voc_a = np.asarray(vec_a.get_feature_names_out())
parole = {}
for c in range(nC):
    y_a = GOLD[human_gold, c].astype(int)
    chi_a = np.nan_to_num(chi2(X_a, y_a)[0])
    piu_a = np.asarray(X_a[y_a == 1].mean(0)).ravel() > np.asarray(X_a[y_a == 0].mean(0)).ravel()   # solo parole "a favore"
    parole[CLASSES[c]] = ', '.join(voc_a[np.argsort(-np.where(piu_a, chi_a, -1))[:12]])
print('Parole della scheda più associate a ogni classe (da controllare a mano):')
print('\n'.join(f'{k:<22} {v}' for k, v in parole.items()))

ps_a = PredefinedSplit(part[gold_rows])
lr_a = OneVsRestClassifier(make_pipeline(StandardScaler(), LogisticRegression(C=0.001, max_iter=2000)))
sola = {}
for nome_x, X_x in [('solo scheda testuale', CARD2[gold_rows]),
                    ('soli embedding medi di sequenza', np.hstack([BLOCKS[b][gold_rows] for b in ['esm_mean'] + ['t5_mean'] * USE_T5]))]:
    P_x = cross_val_predict(lr_a, X_x.astype(np.float32), GOLD[gold_rows].astype(int), cv=ps_a, method='predict_proba')
    sola[nome_x] = np.mean([average_precision_score(GOLD[gold_rows, c], P_x[:, c]) for c in range(nC)])
print('\nRegressione logistica, AUPRC macro fuori fold:', {k: round(float(v), 3) for k, v in sola.items()})


## 5 · Modello 1: baseline "stupida"
Ogni proteina riceve la frequenza della classe nel training. Non guarda la sequenza: AUC = 0,5 per costruzione.

In [ ]:
P_oof_dummy = np.full((N_PROT, nC), np.nan, np.float32)
P_oof_dummy[gold_rows] = GOLD[gold_rows].mean(0)
P_hpa_dummy = np.tile(GOLD[gold_rows].mean(0), (len(test_rows), 1))
print(pd.Series(GOLD[gold_rows].mean(0), index=CLASSES).round(3).to_string())

## 6 · Modello 2: regressione logistica + soglie per classe
Una logistica L2 per compartimento su **tutti** i blocchi standardizzati (gli stessi input della rete, motivi annidati
compresi): il confronto misura solo il modello. Pesi delle classi bilanciati, C scelto sul fold 0, poi CV a 5 fold sulle
partizioni di omologia; il test riceve la media dei 5 modelli. Si costruisce la matrice solo per oro e test (prima
comprendeva anche i 4.000 mutanti) e le 9 classi si stimano in parallelo.

In [ ]:
BASE_B = [f'esm_{s}' for s in SUMMARIES] + [f't5_{s}' for s in SUMMARIES[:NSUM * USE_T5]] + ['hand', 'struct']     # sola sequenza: la scheda non entra (revisione, G4)
# righe: curate (training e valutazione), test HPA, e le altre proteine con una partizione (le aggiunte), che si predicono
# fuori fold come le curate
altre = np.setdiff1d(np.flatnonzero(part >= 0), gold_rows)
rows_lr = np.r_[gold_rows, test_rows, altre]
gi, ti = np.arange(len(gold_rows)), len(gold_rows) + np.arange(len(test_rows))
ai = len(gold_rows) + len(test_rows) + np.arange(len(altre))
d_base = sum(BLOCKS[b].shape[1] for b in BASE_B)
X_lr = np.empty((len(rows_lr), d_base + 2 * nC), np.float32)
X_lr[:, :d_base] = np.hstack([np.asarray(BLOCKS[b][rows_lr], np.float32) for b in BASE_B])
p_gold = part[gold_rows]
C_GRID = [1e-3, 3e-3, 1e-2]
X_lr[:, d_base:] = BLOCKS['motivi@0'][rows_lr]
c_score = []
for c_reg in C_GRID:
    lr = make_pipeline(StandardScaler(), OneVsRestClassifier(LogisticRegression(C=c_reg, max_iter=1000, class_weight='balanced'), n_jobs=-1))
    p0 = lr.fit(X_lr[gi[p_gold != 0]], GOLD[gold_rows[p_gold != 0]]).predict_proba(X_lr[gi[p_gold == 0]])
    c_score.append(np.mean([average_precision_score(GOLD[gold_rows[p_gold == 0], c], p0[:, c]) for c in range(nC)]))
    print(f'C={c_reg:g}: macro-AUPRC fold 0 = {c_score[-1]:.4f}')
C_BEST = C_GRID[int(np.argmax(c_score))]
P_oof_lr = np.full((N_PROT, nC), np.nan, np.float32)
hpa_lr = []
for k in range(5):
    X_lr[:, d_base:] = BLOCKS[f'motivi@{k}'][rows_lr]
    lr = make_pipeline(StandardScaler(), OneVsRestClassifier(LogisticRegression(C=C_BEST, max_iter=1000, class_weight='balanced'), n_jobs=-1))
    lr.fit(X_lr[gi[p_gold != k]], GOLD[gold_rows[p_gold != k]])
    pk = np.r_[gi[p_gold == k], ai[part[altre] == k]]
    P_oof_lr[rows_lr[pk]] = lr.predict_proba(X_lr[pk])
    hpa_lr.append(lr.predict_proba(X_lr[ti]))
P_hpa_lr = np.mean(hpa_lr, 0)
print(f'Logistica, C={C_BEST:g}: macro-AUPRC CV = '
      f'{np.mean([average_precision_score(GOLD[gold_rows, c], P_oof_lr[gold_rows, c]) for c in range(nC)]):.4f}')
del X_lr

## 7 · Modello 3: rete neurale a token, ricerca a stadi
**Architettura.** Ogni blocco (un pezzo di ESM-2, un pezzo di ProtT5, i descrittori, i motivi, la struttura; la scheda
testuale solo nella variante `con_scheda`) passa per una sua proiezione lineare + LayerNorm e diventa un **token** di `tok` numeri. Con `fusione = 0` i
token si concatenano e vanno al classificatore; con `fusione = 2` due strati di attenzione (Transformer, 4 teste) li
fanno interagire prima del classificatore. Il dropout di blocco è il "modality dropout": la rete impara a lavorare anche
senza ProtT5 o senza struttura.

**output.** 9 organelli + 5 compartimenti neuronali, i target ausiliari (meccanismi e superclassi) e due teste di
frazione (citosolica, nucleare).


In [ ]:
PEZZI = ['n20', 'n60', 'centro', 'c15', 'c50']
ESM_B = ['esm_mean', 'esm_mid'] + [f'esm_{p}' for p in PEZZI]
T5_B = (['t5_mean', 't5_mid'] + [f't5_{p}' for p in PEZZI]) * USE_T5
HM = ['hand', 'motivi', 'struct']
MEDIE = ['esm_mean'] + ['t5_mean'] * USE_T5
# la ricerca è di sola sequenza: la scheda testuale porta conoscenza dei curatori (sezione 4b) e non entra nella griglia.
# Entrano invece anche le configurazioni semplici (solo le medie): nell'esecuzione precedente l'"ablazione" con le sole
# medie batteva la configurazione scelta, e una regola di scelta è coerente solo se il candidato migliore può vincere
GRID = dict(blocks=[tuple(MEDIE), tuple(MEDIE + HM), tuple(ESM_B + HM), tuple(ESM_B + T5_B + HM),
                    tuple(ESM_B + T5_B + ['esm_max'] + ['t5_max'] * USE_T5 + HM)],
            fusione=[0, 2], tok=[128, 256], hidden=[512, 1024, 2048], depth=[0, 1], dropout=[0.3, 0.5], in_drop=[0.1, 0.25],
            block_drop=[0.0, 0.15], loss=['bce', 'focal', 'asl'], lr=[3e-4, 1e-3], wd=[1e-3, 1e-2], aux_weight=[0.0, 0.3],
            pool=['oro', 'oro+fonti', 'esteso'], peso_non_umani=[0.3])
LOSS = {'bce': (0.0, 0.0, 0.0, 1.0), 'focal': (2.0, 2.0, 0.0, 1.0), 'asl': (0.0, 4.0, 0.05, 0.0)}   # γ+, γ−, m, pesi positivi
N_SCREEN, NET_BATCH, PATIENCE, EMA_DECAY, CHUNK = 48, 128, 6, 0.995, 8192
FOLDS = [0, 1, 2, 3, 4]
# ogni variante, ogni prova sui dati e i riferimenti girano con 3 semi, come la rete vincente: con un seme solo le
# differenze stavano dentro il rumore, e il vincitore (scelto come il massimo di cinque) partiva avvantaggiato. Così
# tutti i modelli a confronto sono la media di 15 reti (3 semi × 5 fold)
SEMI3 = [0, 1, 2]
STAGES = pd.DataFrame([('screening', 'griglia', N_SCREEN, [0], 25, [0], {}),
                       ('cv', 'screening', 5, FOLDS, 40, [0], {}),
                       ('semi', 'cv', 1, FOLDS, 40, [1, 2], {}),
                       # varianti della rete vincente: 'togli' e 'aggiungi' cambiano i suoi blocchi, qualunque essi siano
                       ('con_scheda', 'cv', 1, FOLDS, 40, SEMI3, {'aggiungi': ('card',)}),
                       ('abl_solo_medie', 'cv', 1, FOLDS, 40, SEMI3, {'blocks': tuple(MEDIE + HM)}),
                       ('abl_senza_struct', 'cv', 1, FOLDS, 40, SEMI3, {'togli': ('struct',)}),
                       ('abl_solo_esm2', 'cv', 1, FOLDS, 40, SEMI3, {'togli': tuple(f't5_{s_}' for s_ in SUMMARIES)}),
                       ('abl_senza_sorting', 'cv', 1, FOLDS, 40, SEMI3, {'togli': tuple(HM)}),
                       ('abl_con_attenzione', 'cv', 1, FOLDS, 40, SEMI3, {'fusione': 2}),
                       ('abl_solo_deeploc', 'cv', 1, FOLDS, 40, SEMI3, {'pool': 'oro'}),
                       ('abl_deeploc_fonti', 'cv', 1, FOLDS, 40, SEMI3, {'pool': 'oro+fonti'}),
                       # riferimento semplice: ProtT5 medio + percettrone sulle etichette DeepLoc umane. NON è DeepLoc 2.0
                       # (che usa l'attenzione sui residui e ~28.000 proteine eucariote): è il modello più semplice
                       # ragionevole. Ha la sua piccola ricerca di iperparametri (prima li ereditava dalla rete vincente)
                       ('rif_screening', 'griglia_rif', 8, [0], 25, [0], {}),
                       ('stile_deeploc2', 'rif_screening', 1, FOLDS, 40, SEMI3, {}),
                       # lo stesso riferimento sugli stessi dati della rete (tutte le fonti, pool esteso): separa il guadagno
                       # dei dati da quello del modello, la domanda che un revisore fa per prima
                       ('stile_deeploc2_stessi_dati', 'rif_screening', 1, FOLDS, 40, SEMI3, {'pool': 'esteso'}),
                       # le etichette del test vengono da HPA: se il guadagno dei dati venisse dalle fonti HPA del training
                       # sarebbe allineamento alla fonte del test, non conoscenza. Qui le due fonti HPA sono tolte
                       ('abl_senza_hpa', 'cv', 1, FOLDS, 40, SEMI3, {'escludi': 'hpa_alta,hpa_approved'}),
                       ('cervello', 'cv', 1, FOLDS, 40, [0], {'pool': 'esteso'})],
                      columns=['stage', 'da', 'top', 'folds', 'epochs', 'seeds', 'cambia'])

# bf16 sulle GPU che lo supportano (Ampere e successive); altrimenti fp32. TF32 per i prodotti fp32 rimasti
AMP = (DEVICE == 'cuda') and torch.cuda.is_bf16_supported()
DEV_TYPE = DEVICE.split(':')[0]
torch.set_float32_matmul_precision('high')
NAMES = list(BLOCKS)
start_of = dict(zip(NAMES, np.cumsum([0] + [BLOCKS[b].shape[1] for b in NAMES[:-1]])))
XG = torch.tensor(np.hstack([np.asarray(BLOCKS[b], np.float16) for b in NAMES]), device=DEVICE)    # tutti i blocchi, ~1 GB
var_rows = np.r_[cv['pid'].unique(), cv['row']]
OBS_t = torch.tensor(np.nan_to_num(OBS), device=DEVICE)
# il peso dei positivi bilancia le classi rare solo sulle etichette curate (fonte 0, canale fissato). Applicato anche alle
# altre fonti gonfiava le sensibilità stimate dal canale di rumore: la loss pesata spinge q verso l'alto e il modello lo
# assorbe alzando la sensibilità della fonte
SOLO_ORO = torch.zeros((S, 1), device=DEVICE)
SOLO_ORO[0] = 1
# i compartimenti neuronali non hanno etichette curate: senza un'ancora sensibilità e specificità delle fonti non sono
# identificabili. Ancora: le annotazioni positive basate su esperimenti (SynGO, UniProt con evidenza sperimentale, GO
# sperimentale) si considerano vere (specificità fissata a 1) per tutte le classi neuronali. È l'ipotesi dei problemi
# "positivi e non etichettati": resta ignota la scala (quante proteine vere mancano), quindi le probabilità neuronali
# vanno lette come ordinamento più che come frequenze
ANCORA_SPEC = torch.ones((S, C), device=DEVICE)
ANCORA_SPEC[torch.tensor(np.isin(SOURCES, ['syngo', 'uniprot_forte', 'goa_exp']), device=DEVICE), nC:] = 0
OBSM_t = torch.tensor(~np.isnan(OBS), device=DEVICE).float()
AUX_t = torch.tensor(np.nan_to_num(AUXY), device=DEVICE)
AUXM_t = torch.tensor(~np.isnan(AUXY), device=DEVICE).float()
FRAC_t = torch.tensor(np.nan_to_num(FRAC), device=DEVICE)
FRACM_t = torch.tensor(np.isfinite(FRAC), device=DEVICE).float()
HUMAN_t = torch.tensor(is_human, device=DEVICE).float()
GOLD_t = torch.tensor(np.nan_to_num(GOLD), device=DEVICE)
GOLDM_t = torch.tensor(GOLD_M, device=DEVICE).float()
HG_MASK = np.isin(np.arange(N_PROT), human_gold)
print(f'Rete: bf16 = {AMP} | tensore degli input {tuple(XG.shape)} su {DEVICE}')

RANK = {'griglia': pd.DataFrame(list(product(*GRID.values())), columns=list(GRID)).sample(N_SCREEN, random_state=SEED).assign(score=0.0)}
# la griglia del riferimento semplice: un solo blocco (la media di ProtT5, o di ESM-2 senza ProtT5), 8 combinazioni
GRID_RIF = dict(GRID, blocks=[[('esm_mean',), ('t5_mean',)][int(USE_T5)]], fusione=[0], tok=[256], hidden=[512, 2048], depth=[0],
                dropout=[0.3, 0.5], in_drop=[0.1], block_drop=[0.0], loss=['bce'], lr=[3e-4, 1e-3], wd=[1e-3], aux_weight=[0.0],
                pool=['oro'], peso_non_umani=[1.0])
RANK['griglia_rif'] = pd.DataFrame(list(product(*GRID_RIF.values())), columns=list(GRID_RIF)).assign(score=0.0)
RUNS = []
for st in STAGES.itertuples(index=False):
    cand = RANK[st.da].sort_values('score', ascending=False).head(st.top)
    cand = cand.drop(columns=['score', 'seed', 'stage', 'run'], errors='ignore').merge(pd.DataFrame({'seed': st.seeds}), how='cross')
    for key, val in st.cambia.items():
        cand[key] = [val] * len(cand)
    # 'togli' e 'aggiungi' agiscono sui blocchi della configurazione di partenza
    cand['blocks'] = [tuple(compress(bl_, [b_ not in st.cambia.get('togli', ()) for b_ in bl_])) + tuple(st.cambia.get('aggiungi', ()))
                      for bl_ in cand['blocks']]
    cand['stage'] = st.stage
    for i, run in cand.iterrows():
        widths = [BLOCKS[b].shape[1] for b in run['blocks']]
        offs = np.cumsum([0] + widths)
        spans = list(zip(offs[:-1], offs[1:]))                                   # dove sta ogni blocco tra le colonne scelte
        bsize = torch.tensor(widths, device=DEVICE)
        nb, T = len(widths), int(run['tok'])
        rows_pool = (src == 'deeploc') | ((src == 'human_extra') & (run['pool'] == 'esteso'))
        src_on = torch.tensor([1.0] + [float(run['pool'] != 'oro')] * (S - 1), device=DEVICE)          # fonti usate dal pool
        src_on = src_on * torch.tensor(~np.isin(SOURCES, str(run.get('escludi', '')).split(',')), dtype=torch.float32, device=DEVICE)
        MASK = OBSM_t * src_on[None, :, None] * (HUMAN_t + (1 - HUMAN_t) * run['peso_non_umani'])[:, None, None]
        gp, gn, marg, use_pw = LOSS[run['loss']]
        oof = np.full((N_PROT, C + A + 2), np.nan, np.float32)
        hpa_pred, var_pred, chan = [], [], []
        for k in st.folds:
            # ogni fold usa i suoi punteggi di motivo (domanda 45)
            cols = torch.tensor(np.concatenate([np.arange(start_of[b], start_of[b] + BLOCKS[b].shape[1])
                                                for b in [{'motivi': f'motivi@{k}'}.get(b_, b_) for b_ in run['blocks']]]), device=DEVICE)
            tr = np.flatnonzero(rows_pool & train_ok & (part != k) & (part != (k + 1) % 5))
            va = np.flatnonzero(rows_pool & train_ok & (part == (k + 1) % 5))
            # si predicono tutte le proteine della partizione k, anche quelle fuori dal pool di training (le aggiunte per il
            # pool "oro"): nessuna è nel training di questo fold, e così ogni metodo ha una predizione fuori fold per ogni
            # proteina (serve al confronto appaiato sul test HOU dello 03)
            te = np.flatnonzero(part == k)
            tr_t, va_t = torch.tensor(tr, device=DEVICE), torch.tensor(va, device=DEVICE)
            # media e deviazione sul training, accumulate a blocchi in float64; poi un unico tensore standardizzato fp16
            s1 = sum(XG[c_[:, None], cols].double().sum(0) for c_ in tr_t.split(CHUNK))
            s2 = sum(XG[c_[:, None], cols].double().pow(2).sum(0) for c_ in tr_t.split(CHUNK))
            mu = (s1 / len(tr)).float()
            sd = ((s2 / len(tr) - (s1 / len(tr)) ** 2).clamp(min=0).sqrt() + 1e-3).float()
            XF = torch.cat([((XG[c_[:, None], cols].float() - mu) / sd).clamp(-20, 20).half() for c_ in torch.cat([tr_t, va_t]).split(CHUNK)])
            va_l = torch.arange(len(tr), len(tr) + len(va), device=DEVICE)
            pos_n = (OBS_t[tr_t] * MASK[tr_t]).sum((0, 1))
            pos_weight = ((MASK[tr_t].sum((0, 1)) - pos_n) / pos_n.clamp(min=1)).sqrt().clamp(1, 10) ** use_pw
            torch.manual_seed(1000 * run['seed'] + k)
            proj = nn.ModuleList([nn.Sequential(nn.Dropout(run['in_drop']), nn.Linear(int(b1 - b0), T), nn.LayerNorm(T)) for b0, b1 in spans])
            attn = [nn.TransformerEncoder(nn.TransformerEncoderLayer(T, 4, 2 * T, run['dropout'], activation='gelu', batch_first=True,
                                                                     norm_first=True), max(1, int(run['fusione'])), enable_nested_tensor=False)] * int(run['fusione'] > 0)
            hidden = [nn.GELU(), *attn, nn.Flatten(1), nn.Dropout(run['dropout']), nn.Linear(nb * T, run['hidden']),
                      nn.LayerNorm(run['hidden']), nn.GELU(), nn.Dropout(run['dropout'])]
            hidden += sum([[nn.Linear(run['hidden'], run['hidden']), nn.LayerNorm(run['hidden']), nn.GELU(),
                            nn.Dropout(run['dropout'])] for _ in range(run['depth'])], [])
            net = nn.ModuleDict({'proj': proj, 'body': nn.Sequential(*hidden, nn.Linear(run['hidden'], C + A + 2))}).to(DEVICE)
            ema = copy.deepcopy(net).eval()
            # canale per fonte inizializzato dai dati (sovrapposizione con DeepLoc + prior Beta); DeepLoc resta perfetta
            g, gm = np.nan_to_num(GOLD[tr])[:, None, :], GOLD_M[tr][:, None, :]
            o_tr, om = np.nan_to_num(OBS[tr, :, :nC]), ~np.isnan(OBS[tr, :, :nC]) & gm
            sens0, spec0 = np.full((S, C), 0.85), np.full((S, C), 0.97)
            sens0[:, :nC] = ((o_tr * g * om).sum(0) + 8.5) / ((g * om).sum(0) + 10)
            spec0[:, :nC] = (((1 - o_tr) * (1 - g) * om).sum(0) + 9.7) / (((1 - g) * om).sum(0) + 10)
            sens_raw = nn.Parameter(torch.tensor(np.log(sens0 / (1 - sens0)), dtype=torch.float32, device=DEVICE))
            spec_raw = nn.Parameter(torch.tensor(np.log(spec0 / (1 - spec0)), dtype=torch.float32, device=DEVICE))
            free = torch.full((S, 1), float(run['pool'] != 'oro'), device=DEVICE)
            free[0] = 0
            free_spec = free * ANCORA_SPEC
            n_batch = max(1, len(tr) // NET_BATCH)
            opt = torch.optim.AdamW([{'params': net.parameters(), 'lr': run['lr'], 'weight_decay': run['wd']},
                                     {'params': [sens_raw, spec_raw], 'lr': 10 * run['lr'], 'weight_decay': 0.0}])
            sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[run['lr'], 10 * run['lr']], total_steps=st.epochs * n_batch, pct_start=0.1)
            best_val, bad, ep, best = np.inf, 0, 0, copy.deepcopy(ema.state_dict())
            while ep < st.epochs and bad < PATIENCE:
                net.train()
                for bl in torch.randperm(len(tr), device=DEVICE)[:n_batch * min(NET_BATCH, len(tr))].view(n_batch, -1):
                    gb = tr_t[bl]
                    keep_b = (torch.rand(len(bl), nb, device=DEVICE) >= run['block_drop']).float() / (1 - run['block_drop'])
                    xb = XF[bl].float() * torch.repeat_interleave(keep_b, bsize, dim=1)      # dropout di blocco
                    with torch.autocast(DEV_TYPE, dtype=torch.bfloat16, enabled=AMP):
                        out = net['body'](torch.stack([pj(xb[:, b0:b1]) for pj, (b0, b1) in zip(net['proj'], spans)], 1))
                    out = out.float()
                    p = torch.sigmoid(out[:, :C])[:, None, :].clamp(1e-5, 1 - 1e-5)
                    sens = 1 - free * (1 - torch.sigmoid(sens_raw))
                    spec = 1 - free_spec * (1 - torch.sigmoid(spec_raw))
                    q = (sens * p + (1 - spec) * (1 - p)).clamp(1e-5, 1 - 1e-5)               # P(la fonte dice 1)
                    qm = (q - marg).clamp(min=1e-5)
                    o, mk = OBS_t[gb], MASK[gb]
                    loss_loc = -(mk * ((1 + (pos_weight - 1) * SOLO_ORO) * o * (1 - q) ** gp * q.log() + (1 - o) * qm ** gn * (1 - qm).log())).sum() / mk.sum().clamp(min=1)
                    loss_aux = (AUXM_t[gb] * F.binary_cross_entropy_with_logits(out[:, C:C + A], AUX_t[gb], reduction='none')).sum() / AUXM_t[gb].sum().clamp(min=1)
                    loss_frac = (FRACM_t[gb] * F.binary_cross_entropy_with_logits(out[:, -2:], FRAC_t[gb], reduction='none')).sum() / FRACM_t[gb].sum().clamp(min=1)
                    loss = loss_loc + run['aux_weight'] * loss_aux + FRAC_W * loss_frac
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    opt.step()
                    sched.step()
                    with torch.no_grad():
                        for pe, pn in zip(ema.parameters(), net.parameters()):
                            pe.lerp_(pn, 1 - EMA_DECAY)
                # validazione con i pesi EMA: solo la partizione di validazione, il resto alla fine
                with torch.no_grad(), torch.autocast(DEV_TYPE, dtype=torch.bfloat16, enabled=AMP):
                    zv = torch.cat([ema['body'](torch.stack([pj(xc[:, b0:b1]) for pj, (b0, b1) in zip(ema['proj'], spans)], 1)).float()
                                    for c_ in va_l.split(CHUNK) for xc in [XF[c_].float()]])[:, :nC]
                vloss = float((GOLDM_t[va_t] * (F.softplus(zv) - GOLD_t[va_t] * zv)).sum() / GOLDM_t[va_t].sum().clamp(min=1))
                best = [best, {n_: v_.detach().clone() for n_, v_ in ema.state_dict().items()}][int(vloss < best_val)]
                bad = [bad + 1, 0][int(vloss < best_val)]
                best_val = min(best_val, vloss)
                ep += 1
            # predizioni con i pesi EMA dell'epoca migliore: fold di test, test HPA e varianti, una volta sola
            ema.load_state_dict(best)
            # i 5 modelli dello stadio "cervello" si salvano: il notebook 03 li usa per le proteine nuove
            for _ in range(int(st.stage == 'cervello')):
                (DATA / 'modello_cervello').mkdir(exist_ok=True)
                torch.save({'state': {n_: v_.cpu() for n_, v_ in best.items()}, 'mu': mu.cpu(), 'sd': sd.cpu(), 'run': run.to_dict(),
                            'spans': [(int(b0), int(b1)) for b0, b1 in spans], 'C': C, 'A': A}, DATA / 'modello_cervello' / f'fold{k}.pt')
            with torch.no_grad(), torch.autocast(DEV_TYPE, dtype=torch.bfloat16, enabled=AMP):
                PRED = {n_: torch.cat([ema['body'](torch.stack([pj(xc[:, b0:b1]) for pj, (b0, b1) in zip(ema['proj'], spans)], 1)).float()
                                       for c_ in torch.tensor(rr, device=DEVICE).split(CHUNK)
                                       for xc in [((XG[c_[:, None], cols].float() - mu) / sd).clamp(-20, 20)]]).cpu().numpy()
                        for n_, rr in [('te', te), ('hpa', test_rows), ('var', var_rows)]}
            oof[te] = PRED['te']
            hpa_pred.append(PRED['hpa'])
            var_pred.append(PRED['var'])
            # sensibilità e specificità effettive (con le parti fissate: 1 per le etichette curate e per la specificità di SynGO)
            chan.append(np.stack([(1 - free * (1 - torch.sigmoid(sens_raw))).detach().cpu().numpy(),
                                  (1 - free_spec * (1 - torch.sigmoid(spec_raw))).detach().cpu().numpy()]))
            del XF, net, ema, opt
        ok = HG_MASK & ~np.isnan(oof[:, 0])
        cand.loc[i, 'score'] = np.mean([average_precision_score(GOLD[ok, c], oof[ok, c]) for c in range(nC)])
        cand.loc[i, 'run'] = len(RUNS)
        RUNS.append({'oof': oof, 'hpa': np.mean(hpa_pred, 0), 'var': np.stack(var_pred), 'chan': np.mean(chan, 0), 'folds': list(st.folds)})
        print(f"[{st.stage} {i + 1}/{len(cand)}] {run['pool']:<9} {len(run['blocks'])} blocchi{' + card' * ('card' in run['blocks'])} "
              f"fus={run['fusione']} tok={run['tok']} h={run['hidden']} d={run['depth']} drop={run['dropout']}/{run['in_drop']}/{run['block_drop']} "
              f"{run['loss']} lr={run['lr']} wd={run['wd']} aux={run['aux_weight']} seed={run['seed']} → macro-AUPRC {cand.loc[i, 'score']:.4f}")
    RANK[st.stage] = cand.astype({'run': int})
    torch.cuda.empty_cache()

## 7b · Cosa ha imparato la ricerca
Effetto medio di ogni scelta nello screening: dice *perché* una configurazione vince e dove stringere la griglia.

In [ ]:
scr = RANK['screening'].assign(blocks=RANK['screening']['blocks'].map('+'.join))
for col in ['blocks', 'fusione', 'tok', 'hidden', 'depth', 'dropout', 'in_drop', 'block_drop', 'loss', 'lr', 'wd', 'aux_weight', 'pool']:
    print(scr.groupby(col)['score'].agg(['mean', 'max', 'count']).round(4), '\n')
print('CV completa delle 5 migliori:')
print(RANK['cv'].assign(blocks=RANK['cv']['blocks'].map(len).astype(str) + ' blocchi').drop(columns=['stage']).round(4).to_string())

## 7c · Rete avanzata: attenzione sui residui + fusione multimodale (NeuroLoc-LA)
La rete a token vede la proteina attraverso 8 riassunti fissi (media, inizio, centro, fine). È il suo limite: un
segnale di smistamento è un tratto corto in una posizione precisa (un PTS1 negli ultimi 3 residui, una presequenza
nei primi 20–60, un NLS ovunque nella sequenza) e una media lo diluisce. Lo stato dell'arte (DeepLoc 2.0, Light
Attention di Stärk et al. 2021) guarda invece **i singoli residui** e impara dove guardare. Questa rete unisce le due
cose.

**Domande da esperto e scelte di progetto**
- *Quale rappresentazione dei residui?* L'ultimo layer di ESM-2 per ogni residuo (la stessa sequenza troncata dello 02),
  ridotto da 1.280 a 256 numeri con una PCA stimata su 400 proteine: la PCA conserva la quasi totalità della varianza
  e porta lo spazio su disco da ~20 a ~4 GB (float16), abbastanza per tenerlo tutto in memoria (o sulla A5000).
- *Come sa la rete "dove" sta un residuo?* I segnali sono ancorati ai terminali, non a una posizione assoluta: ogni
  residuo riceve la posizione relativa, la distanza (log) dall'N- e dal C-terminale, l'appartenenza ai primi 30 e agli
  ultimi 15 residui e due onde sinusoidali. È la "codifica terminale" che serve ai segnali di sorting.
- *Come vede i motivi corti?* Tre convoluzioni 1D residue con dilatazione 1, 2, 4 (campo recettivo ~57 residui):
  coprono peptidi segnale (15–30 aa), presequenze, eliche TM (~20 aa) e motivi di coda.
- *Come riassume la sequenza?* Con **attenzione multi-testa sui residui** (Light Attention estesa): 12 teste, ognuna
  una distribuzione sui residui; ogni testa produce un "token di segnale". Le teste si specializzano (in letteratura
  una guarda l'N-terminale, una il C-terminale, una i tratti idrofobici): l'attenzione è leggibile come mappa dei
  segnali.
- *Come si uniscono sequenza, scheda, struttura e descrittori?* I 12 token di segnale entrano, insieme ai token dei
  blocchi globali della rete vincente (riassunti ESM-2/ProtT5, card_protein, struttura AlphaFold, descrittori, motivi),
  in un **Transformer di fusione** a 2 strati con un token di classe e un'**embedding di tipo** per ogni modalità.
- *Come si addestra senza sovra-adattare 6–15.000 proteine?* Stesse armi della rete a token più due specifiche:
  dropout dei residui (10%: la rete non può affidarsi a un solo residuo), dropout di blocco (modality dropout), EMA dei
  pesi, taglio del gradiente, OneCycle, bf16 sulla A5000. Stessa loss con canale di rumore per fonte, stessi target
  ausiliari (meccanismi, superclassi) e stesse teste di frazione: il confronto misura l'architettura, non i dati.
- *Quali iperparametri?* Quelli della configurazione vincente in CV (blocchi, token, pool, loss, dropout, lr), più le
  scelte della letteratura per la parte sui residui. Ensemble di 5 fold × 2 semi, come la rete finale.


In [ ]:
# 7c-1 · embedding per-residuo di ESM-2 ridotti con PCA (ripartibile, una volta sola: ~10–15 min su A5000)
RES_DIM = min(256, D)
lens_r = lengths.iloc[:N_PROT].to_numpy().astype(int)
offs_r = np.r_[0, np.cumsum(lens_r)]
path_rp, path_rpd, path_pca = DATA / f'res_pca_{tag}.npy', DATA / f'res_pca_{tag}_DONE.npy', DATA / f'res_pca_{tag}_base.npz'
RESE = np.lib.format.open_memmap(path_rp, mode=['w+', 'r+'][path_rp.exists()], dtype=np.float16, shape=(int(offs_r[-1]), RES_DIM))
DONE_R = np.lib.format.open_memmap(path_rpd, mode=['w+', 'r+'][path_rpd.exists()], dtype=bool, shape=(N_PROT,))
print(f'Residui da codificare: {(~DONE_R[:]).sum():,} proteine su {N_PROT:,}')
for _ in range(int((~DONE_R[:]).any())):
    esm_r = EsmModel.from_pretrained(ESM, torch_dtype=DTYPE).to(DEVICE).eval()
    # base della PCA: residui di 400 proteine a caso
    for _p in range(int(not path_pca.exists())):
        campione = []
        for i in rng.choice(np.flatnonzero(src != 'hpa_test'), 400, replace=False):      # mai proteine del test
            with torch.no_grad():
                h_ = esm_r(**tok([trunc[i]], return_tensors='pt').to(DEVICE)).last_hidden_state[0, 1:lens_r[i] + 1].float()
            campione.append(h_)
        campione = torch.cat(campione)
        mu_r = campione.mean(0)
        comp = torch.linalg.svd(campione - mu_r, full_matrices=False)[2][:RES_DIM]
        np.savez(path_pca, mu=mu_r.cpu().numpy(), comp=comp.cpu().numpy())
    base = np.load(path_pca)
    mu_r, comp = torch.tensor(base['mu'], device=DEVICE), torch.tensor(base['comp'], device=DEVICE)
    todo_r = pd.Series(lens_r[~DONE_R[:]], index=np.flatnonzero(~DONE_R[:])).sort_values()
    for _, band in tqdm(list(todo_r.groupby(todo_r // 32)), desc='Residui ESM-2 → PCA', unit='banda'):
        bs = max(1, TOKENS // int(band.max() + 2))
        for s0 in range(0, len(band), bs):
            idx = band.index[s0:s0 + bs].to_numpy()
            with torch.no_grad():
                h_ = esm_r(**tok(trunc[idx].tolist(), return_tensors='pt', padding=True).to(DEVICE)).last_hidden_state.float()
                z_ = ((h_ - mu_r) @ comp.T).half().cpu().numpy()
            for j, i in enumerate(idx):
                RESE[offs_r[i]:offs_r[i + 1]] = z_[j, 1:lens_r[i] + 1]
            DONE_R[idx] = True
        RESE.flush()
        DONE_R.flush()
    del esm_r
    torch.cuda.empty_cache()
# tutti i residui in memoria; sulla GPU se c'è spazio (≥ 16 GB), altrimenti in RAM e un batch alla volta
RES_DEV = ['cpu', DEVICE][int(DEVICE == 'cuda' and torch.cuda.get_device_properties(0).total_memory > 16e9)]
RESE_t = torch.from_numpy(np.asarray(RESE)).to(RES_DEV)
START_t, LEN_t = torch.tensor(offs_r[:-1], device=RES_DEV), torch.tensor(lens_r, device=RES_DEV)
print(f'Embedding per-residuo: {tuple(RESE_t.shape)} su {RES_DEV}')

In [ ]:
# 7c-2 · NeuroLoc-LA: attenzione multi-testa sui residui + Transformer di fusione con i blocchi globali
best_cfg = RANK['cv'].sort_values('score', ascending=False).iloc[0]
ADV = dict(d=192, H=12, epochs=40, seeds=[0, 1], lr=5e-4, wd=1e-2, res_drop=0.10, batch=32, patience=8, clip=1.0)
blocks_a, T = list(best_cfg['blocks']), int(best_cfg['tok'])
widths_a = [BLOCKS[b].shape[1] for b in blocks_a]
offs_a = np.cumsum([0] + widths_a)
spans_a = list(zip(offs_a[:-1], offs_a[1:]))
bsize_a, nb_a = torch.tensor(widths_a, device=DEVICE), len(widths_a)
gp, gn, marg, use_pw = LOSS[best_cfg['loss']]
rows_pool_a = (src == 'deeploc') | ((src == 'human_extra') & (best_cfg['pool'] == 'esteso'))
src_on_a = torch.tensor([1.0] + [float(best_cfg['pool'] != 'oro')] * (S - 1), device=DEVICE)
MASK_A = OBSM_t * src_on_a[None, :, None] * (HUMAN_t + (1 - HUMAN_t) * best_cfg['peso_non_umani'])[:, None, None]
LOG1K = float(np.log(1024.0))
oof_adv = np.zeros((N_PROT, C + A + 2), np.float32)
n_adv = np.zeros(N_PROT, np.float32)
hpa_adv = []
print(f"Configurazione di partenza (vincente in CV): {nb_a} blocchi, tok {T}, pool {best_cfg['pool']}, loss {best_cfg['loss']}")
for seed in ADV['seeds']:
    for k in FOLDS:
        cols = torch.tensor(np.concatenate([np.arange(start_of[b], start_of[b] + BLOCKS[b].shape[1])
                                            for b in [{'motivi': f'motivi@{k}'}.get(b_, b_) for b_ in blocks_a]]), device=DEVICE)
        tr = np.flatnonzero(rows_pool_a & train_ok & (part != k) & (part != (k + 1) % 5))
        va = np.flatnonzero(rows_pool_a & train_ok & (part == (k + 1) % 5))
        te = np.flatnonzero(part == k)                    # tutta la partizione k, anche le aggiunte (come la rete)
        tr_t = torch.tensor(tr, device=DEVICE)
        s1 = sum(XG[c_[:, None], cols].double().sum(0) for c_ in tr_t.split(CHUNK))
        s2 = sum(XG[c_[:, None], cols].double().pow(2).sum(0) for c_ in tr_t.split(CHUNK))
        mu = (s1 / len(tr)).float()
        sd = ((s2 / len(tr) - (s1 / len(tr)) ** 2).clamp(min=0).sqrt() + 1e-3).float()
        pos_n = (OBS_t[tr_t] * MASK_A[tr_t]).sum((0, 1))
        pos_weight = ((MASK_A[tr_t].sum((0, 1)) - pos_n) / pos_n.clamp(min=1)).sqrt().clamp(1, 10) ** use_pw
        torch.manual_seed(7000 + 100 * seed + k)
        d_, H_ = ADV['d'], ADV['H']
        adv = nn.ModuleDict({
            'res_in': nn.Sequential(nn.Linear(RES_DIM + 8, d_), nn.LayerNorm(d_)),
            'conv': nn.ModuleList([nn.Conv1d(d_, d_, 9, padding=4 * dl, dilation=dl) for dl in [1, 2, 4]]),
            'att': nn.Conv1d(d_, H_, 9, padding=4), 'val': nn.Conv1d(d_, d_, 9, padding=4),
            'sig': nn.Sequential(nn.Linear(d_, T), nn.LayerNorm(T)),
            'proj': nn.ModuleList([nn.Sequential(nn.Dropout(best_cfg['in_drop']), nn.Linear(int(b1 - b0), T), nn.LayerNorm(T)) for b0, b1 in spans_a]),
            'tipo': nn.Embedding(1 + nb_a + H_, T),
            'fuse': nn.TransformerEncoder(nn.TransformerEncoderLayer(T, 4, 2 * T, best_cfg['dropout'], activation='gelu', batch_first=True,
                                                                     norm_first=True), 2, enable_nested_tensor=False),
            'head': nn.Sequential(nn.LayerNorm(2 * T), nn.Dropout(best_cfg['dropout']), nn.Linear(2 * T, best_cfg['hidden']), nn.GELU(),
                                  nn.Dropout(best_cfg['dropout']), nn.Linear(best_cfg['hidden'], C + A + 2))}).to(DEVICE)
        ema = copy.deepcopy(adv).eval()
        g, gm = np.nan_to_num(GOLD[tr])[:, None, :], GOLD_M[tr][:, None, :]
        o_tr, om = np.nan_to_num(OBS[tr, :, :nC]), ~np.isnan(OBS[tr, :, :nC]) & gm
        sens0, spec0 = np.full((S, C), 0.85), np.full((S, C), 0.97)
        sens0[:, :nC] = ((o_tr * g * om).sum(0) + 8.5) / ((g * om).sum(0) + 10)
        spec0[:, :nC] = (((1 - o_tr) * (1 - g) * om).sum(0) + 9.7) / (((1 - g) * om).sum(0) + 10)
        sens_raw = nn.Parameter(torch.tensor(np.log(sens0 / (1 - sens0)), dtype=torch.float32, device=DEVICE))
        spec_raw = nn.Parameter(torch.tensor(np.log(spec0 / (1 - spec0)), dtype=torch.float32, device=DEVICE))
        free = torch.full((S, 1), float(best_cfg['pool'] != 'oro'), device=DEVICE)
        free[0] = 0
        free_spec = free * ANCORA_SPEC
        n_batch = max(1, len(tr) // ADV['batch'])
        opt = torch.optim.AdamW([{'params': adv.parameters(), 'lr': ADV['lr'], 'weight_decay': ADV['wd']},
                                 {'params': [sens_raw, spec_raw], 'lr': 10 * ADV['lr'], 'weight_decay': 0.0}])
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[ADV['lr'], 10 * ADV['lr']], total_steps=ADV['epochs'] * n_batch, pct_start=0.1)
        best_val, bad, ep, fase_g, best = np.inf, 0, 0, 0, copy.deepcopy(ema.state_dict())
        pred_rows = np.r_[te, test_rows]
        # un solo ciclo per addestramento, validazione e predizione: fase_g = 0 addestra e valida, 1 predice con i pesi migliori
        while fase_g < 2:
            for _ in range(fase_g):
                ema.load_state_dict(best)
            # il piano dell'epoca: (fase, proteine del batch)
            piano = [[('train', tr_t[b]) for b in torch.randperm(len(tr), device=DEVICE)[:n_batch * min(ADV['batch'], len(tr))].view(n_batch, -1)]
                     + [('val', c_) for c_ in torch.tensor(va, device=DEVICE).split(256)],
                     [('pred', c_) for c_ in torch.tensor(pred_rows, device=DEVICE).split(256)]][fase_g]
            uscite = {'train': [], 'val': [], 'pred': []}
            for fase, righe in piano:
                train = int(fase == 'train')
                mdl = [ema, adv][train]
                mdl.train(bool(train))
                with torch.set_grad_enabled(bool(train)), torch.autocast(DEV_TYPE, dtype=torch.bfloat16, enabled=AMP):
                    # residui: gather dalla matrice piatta, maschera di padding, codifica terminale
                    r_dev = righe.to(RES_DEV)
                    ln = LEN_t[r_dev]
                    ar = torch.arange(int(ln.max()), device=RES_DEV)
                    m_r = (ar[None] < ln[:, None])
                    x_r = RESE_t[(START_t[r_dev][:, None] + ar[None]).clamp(max=RESE_t.shape[0] - 1)].to(DEVICE).float() * m_r[..., None].to(DEVICE)
                    p_, L_ = ar.to(DEVICE).float()[None].expand(len(righe), -1), ln.to(DEVICE).float()[:, None]
                    dC = (L_ - 1 - p_).clamp(min=0)
                    posf = torch.stack([p_ / L_, torch.log1p(p_) / LOG1K, torch.log1p(dC) / LOG1K, (p_ < 30).float(), (dC < 15).float(),
                                        torch.sin(p_ / 16), torch.sin(dC / 16), torch.cos(p_ / 16)], -1)
                    m_d = m_r.to(DEVICE) & (torch.rand(m_r.shape, device=DEVICE) >= ADV['res_drop'] * train)       # dropout dei residui
                    h = mdl['res_in'](torch.cat([x_r, posf], -1)).transpose(1, 2) * m_d[:, None]
                    for cv_ in mdl['conv']:
                        h = (h + F.gelu(cv_(h))) * m_d[:, None]
                    att = mdl['att'](h).masked_fill(~m_d[:, None], -1e4).softmax(-1)                   # [batch, testa, residuo]
                    sig = mdl['sig'](torch.einsum('bhl,bdl->bhd', att, mdl['val'](h)))                  # 12 token di segnale
                    # blocchi globali come nella rete a token, con dropout di blocco in addestramento
                    xg = ((XG[righe[:, None], cols].float() - mu) / sd).clamp(-20, 20)
                    keep_b = (torch.rand(len(righe), nb_a, device=DEVICE) >= best_cfg['block_drop'] * train).float() / (1 - best_cfg['block_drop'] * train)
                    xg = xg * torch.repeat_interleave(keep_b, bsize_a, dim=1)
                    glob = torch.stack([pj(xg[:, b0:b1]) for pj, (b0, b1) in zip(mdl['proj'], spans_a)], 1)
                    tok_all = torch.cat([mdl['tipo'].weight[:1][None].expand(len(righe), -1, -1), glob, sig], 1) + mdl['tipo'].weight[None]
                    z_ = mdl['fuse'](tok_all)
                    out = mdl['head'](torch.cat([z_[:, 0], z_[:, 1 + nb_a:].mean(1)], -1))
                out = out.float()
                uscite[fase].append(out.detach())
                # loss con canale di rumore (identica alla rete a token), solo in addestramento
                for _ in range(train):
                    p = torch.sigmoid(out[:, :C])[:, None, :].clamp(1e-5, 1 - 1e-5)
                    sens = 1 - free * (1 - torch.sigmoid(sens_raw))
                    spec = 1 - free_spec * (1 - torch.sigmoid(spec_raw))
                    q = (sens * p + (1 - spec) * (1 - p)).clamp(1e-5, 1 - 1e-5)
                    qm = (q - marg).clamp(min=1e-5)
                    o, mk = OBS_t[righe], MASK_A[righe]
                    loss = (-(mk * ((1 + (pos_weight - 1) * SOLO_ORO) * o * (1 - q) ** gp * q.log() + (1 - o) * qm ** gn * (1 - qm).log())).sum() / mk.sum().clamp(min=1)
                            + best_cfg['aux_weight'] * (AUXM_t[righe] * F.binary_cross_entropy_with_logits(out[:, C:C + A], AUX_t[righe], reduction='none')).sum()
                            / AUXM_t[righe].sum().clamp(min=1)
                            + FRAC_W * (FRACM_t[righe] * F.binary_cross_entropy_with_logits(out[:, -2:], FRAC_t[righe], reduction='none')).sum()
                            / FRACM_t[righe].sum().clamp(min=1))
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(adv.parameters(), ADV['clip'])
                    opt.step()
                    sched.step()
                    with torch.no_grad():
                        for pe, pn in zip(ema.parameters(), adv.parameters()):
                            pe.lerp_(pn, 1 - EMA_DECAY)
            zv = torch.cat(uscite['val'] + [torch.zeros((0, C + A + 2), device=DEVICE)])[:, :nC]
            va_t = torch.tensor(va, device=DEVICE)[:len(zv)]                 # nella fase di predizione zv è vuoto
            vloss = float((GOLDM_t[va_t] * (F.softplus(zv) - GOLD_t[va_t] * zv)).sum() / GOLDM_t[va_t].sum().clamp(min=1)) + 1e9 * fase_g
            best = [best, {n_: v_.detach().clone() for n_, v_ in ema.state_dict().items()}][int(vloss < best_val)]
            bad = [bad + 1, 0][int(vloss < best_val)]
            best_val = min(best_val, vloss)
            ep += 1 - fase_g
            pred_out = torch.cat(uscite['pred'] + [torch.zeros((0, C + A + 2), device=DEVICE)]).cpu().numpy()
            fase_g = fase_g + int(fase_g == 1 or ep >= ADV['epochs'] or bad >= ADV['patience'])
        oof_adv[te] += pred_out[:len(te)]
        n_adv[te] += 1
        hpa_adv.append(pred_out[len(te):])
        print(f'[NeuroLoc-LA seme {seed} fold {k}] epoche {ep}, loss di validazione migliore {best_val:.4f}')
        del adv, ema, opt
        torch.cuda.empty_cache()
oof_adv = np.where(n_adv[:, None] > 0, oof_adv / np.maximum(n_adv, 1)[:, None], np.nan)
P_oof_adv = 1 / (1 + np.exp(-oof_adv[:, :nC]))
P_hpa_adv = (1 / (1 + np.exp(-np.stack(hpa_adv)[:, :, :nC]))).mean(0)
print(f'NeuroLoc-LA: macro-AUPRC CV (DeepLoc umane) = '
      f'{np.mean([average_precision_score(GOLD[human_gold, c], P_oof_adv[human_gold, c]) for c in range(nC)]):.4f}')

## 8 · Sistema finale della rete e k-NN
La rete finale è l'ensemble della configurazione vincente con 3 semi × 5 fold, con i pesi EMA. "Rete + k-NN": le
etichette delle proteine di training più simili vengono mescolate alle probabilità della rete, con α scelto sull'OOF.

In [ ]:
best_runs = [int(RANK['cv']['run'].iloc[RANK['cv']['score'].argmax()])] + list(RANK['semi']['run'])
P_oof_nn = (1 / (1 + np.exp(-np.stack([RUNS[r]['oof'][:, :nC] for r in best_runs])))).mean(0)
P_hpa_nn = (1 / (1 + np.exp(-np.stack([RUNS[r]['hpa'][:, :nC] for r in best_runs])))).mean(0)

KNN, ALPHAS = 10, np.linspace(0, 0.6, 13)
knn_oof = np.full((N_PROT, nC), np.nan, np.float32)
for k in range(5):
    ref, qry = gold_rows[part[gold_rows] != k], np.flatnonzero(part == k)
    for q_ in np.array_split(qry, max(1, len(qry) // 2000)):
        sim = unit[q_] @ unit[ref].T
        nb_ = np.argpartition(-sim, KNN, axis=1)[:, :KNN]
        w = np.take_along_axis(sim, nb_, 1).clip(0) ** 8                    # i vicini molto simili pesano molto di più
        knn_oof[q_] = (w[..., None] * GOLD[ref][nb_]).sum(1) / (w.sum(1, keepdims=True) + 1e-9)
sim = unit[test_rows] @ unit[gold_rows].T
nb_ = np.argpartition(-sim, KNN, axis=1)[:, :KNN]
w = np.take_along_axis(sim, nb_, 1).clip(0) ** 8
knn_hpa = (w[..., None] * GOLD[gold_rows][nb_]).sum(1) / (w.sum(1, keepdims=True) + 1e-9)
a_score = [np.mean([average_precision_score(GOLD[gold_rows, c], (1 - a) * P_oof_nn[gold_rows, c] + a * knn_oof[gold_rows, c])
                    for c in range(nC)]) for a in ALPHAS]
ALPHA = ALPHAS[int(np.argmax(a_score))]
P_oof_knn, P_hpa_knn = (1 - ALPHA) * P_oof_nn + ALPHA * knn_oof, (1 - ALPHA) * P_hpa_nn + ALPHA * knn_hpa
print(f"Run nell'ensemble: {best_runs} | α k-NN = {ALPHA:.2f}")

## 8b · Una seconda famiglia di modelli e l'ensemble scelto in CV
Modelli di famiglie diverse sbagliano in modo diverso. Il **gradient boosting** lavora bene su descrittori, motivi e
struttura, dove contano le soglie ("TM lungo ≥ 20 e coda con YxxΦ"). L'ensemble si sceglie con la macro-AUPRC out-of-fold tra le combinazioni dei membri, e la scelta è
**annidata**: per valutare il fold k la combinazione si sceglie sugli altri quattro. Scegliere e misurare sugli stessi
dati dava una stima ottimistica; il notebook stampa le due per confronto e riporta quella annidata.

In [ ]:
TAB_BASE = np.hstack([np.asarray(BLOCKS['hand'][:N_PROT], np.float32), np.asarray(BLOCKS['struct'][:N_PROT], np.float32)] +
                     # la PCA si stima senza le proteine del test e si applica a tutte
                     [PCA(64, random_state=SEED).fit(X_b[src != 'hpa_test']).transform(X_b)
                      for b in ['esm_mean', 'esm_n20', 'esm_c15'] + ['t5_mean'] * USE_T5 for X_b in [np.asarray(BLOCKS[b][:N_PROT], np.float32)]])
pg = part[gold_rows]
P_oof_hgb = np.full((N_PROT, nC), np.nan, np.float32)
hpa_hgb = []
for k in range(5):
    X_tab = np.hstack([TAB_BASE, np.asarray(BLOCKS[f'motivi@{k}'][:N_PROT], np.float32)])
    # l'arresto anticipato usa una partizione di omologia (la successiva), come la rete; prima una divisione casuale
    tr, va, te = gold_rows[~np.isin(pg, [k, (k + 1) % 5])], gold_rows[pg == (k + 1) % 5], np.flatnonzero(part == k)
    hp = np.zeros((len(test_rows), nC), np.float32)
    for c in range(nC):
        # senza class_weight: con pesi per campione scikit-learn calcola i quantili dei bin in modo pesato, decine di volte
        # più lento; AUC e AUPRC dipendono solo dall'ordinamento, che i pesi di classe non migliorano
        hgb_c = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06, max_leaf_nodes=31, l2_regularization=1.0,
                                               early_stopping=True, random_state=SEED).fit(X_tab[tr], GOLD[tr, c], X_val=X_tab[va], y_val=GOLD[va, c])
        P_oof_hgb[te, c] = hgb_c.predict_proba(X_tab[te])[:, 1]
        hp[:, c] = hgb_c.predict_proba(X_tab[test_rows])[:, 1]
    hpa_hgb.append(hp)
P_hpa_hgb = np.mean(hpa_hgb, 0)
print(f'Alberi: macro-AUPRC CV = {np.mean([average_precision_score(GOLD[gold_rows, c], P_oof_hgb[gold_rows, c]) for c in range(nC)]):.4f}')

# membri: tutti di sola sequenza. La scelta tra le combinazioni è annidata: per il fold k la combinazione si sceglie sugli
# altri quattro, così la metrica fuori fold dell'ensemble non è misurata sugli stessi dati che l'hanno scelto
MEMBERS = {'rete': (P_oof_nn, P_hpa_nn), 'rete avanzata': (P_oof_adv, P_hpa_adv), 'k-NN': (knn_oof, knn_hpa), 'alberi': (P_oof_hgb, P_hpa_hgb),
           'logistica': (P_oof_lr, P_hpa_lr)}
combos = [cmb for r_ in range(1, len(MEMBERS) + 1) for cmb in combinations(MEMBERS, r_)]
M_gold = [np.mean([MEMBERS[m][0][gold_rows] for m in cmb], 0) for cmb in combos]
ens_score = [np.mean([average_precision_score(GOLD[gold_rows, c], M_[:, c]) for c in range(nC)]) for M_ in M_gold]
ENS = combos[int(np.argmax(ens_score))]                       # la combinazione finale (per il test e per lo 03): scelta su tutti i fold
P_oof_ens, P_hpa_ens = [np.mean([MEMBERS[m][i] for m in ENS], 0) for i in [0, 1]]
scelte_k = []
for k in range(5):
    sc_k = [np.mean([average_precision_score(GOLD[gold_rows[pg != k], c], M_[pg != k, c]) for c in range(nC)]) for M_ in M_gold]
    scelte_k.append(' + '.join(combos[int(np.argmax(sc_k))]))
    P_oof_ens[part == k] = np.mean([MEMBERS[m][0][part == k] for m in combos[int(np.argmax(sc_k))]], 0)
ENS_NAME = f"Ensemble scelto in CV ({' + '.join(ENS)})"
print(pd.Series(ens_score, index=[' + '.join(cmb) for cmb in combos]).sort_values(ascending=False).head(8).round(4).to_string())
print(f"Ensemble scelto in CV: {' + '.join(ENS)} | scelte nei 5 fold annidati: {scelte_k}")
print(f"AUPRC macro fuori fold dell'ensemble: {np.mean([average_precision_score(GOLD[gold_rows, c], P_oof_ens[gold_rows, c]) for c in range(nC)]):.4f} "
      f"con la scelta annidata (quella da riportare) | {max(ens_score):.4f} scegliendo sugli stessi dati (ottimistica)")


## 8c · Contesto molecolare: i partner stanno di solito nello stesso compartimento
Molte proteine del Golgi, dei centrosomi o dei complessi nucleari non hanno un segnale proprio: ci arrivano legandosi a
un partner. Per ogni proteina si calcola la quota dei partner localizzati in ogni compartimento, con
le sole etichette di training e misure esterne; per le proteine del fold k contano solo partner fuori dal fold k. Una
logistica per classe unisce l'ensemble e il contesto. 

In [ ]:
grafo = pd.read_parquet(DATA / 'grafo.parquet')
grafo = grafo[(grafo['pid_a'] < N_PROT) & (grafo['pid_b'] < N_PROT)]
lab_src = np.isin(SOURCES, ['deeploc', 'uniprot_forte', 'hpa_alta', 'opencell', 'bioid', 'lopit_hyper', 'lopit_dc', 'lopit_2024', 'mitocarta', 'goa_exp'])
LAB = np.where(np.isnan(OBS[:, lab_src, :nC]), -1, OBS[:, lab_src, :nC]).max(1)
LAB = np.where(LAB < 0, np.nan, LAB)
NBL = np.full((N_PROT, nC), np.nan, np.float32)
for k in [0, 1, 2, 3, 4, -1]:
    lab_k = np.where((part != k)[:, None], LAB, np.nan)
    e = grafo[part[grafo['pid_a'].to_numpy()] == k]
    nb_mean = pd.DataFrame(lab_k[e['pid_b']], index=e['pid_a'].to_numpy()).groupby(level=0).mean()
    NBL[nb_mean.index] = nb_mean.to_numpy()
has_nb = ~np.isnan(NBL).all(1)
lg_ens = np.log(np.clip(P_oof_ens, 1e-4, 1 - 1e-4) / (1 - np.clip(P_oof_ens, 1e-4, 1 - 1e-4)))
lg_hpa = np.log(np.clip(P_hpa_ens, 1e-4, 1 - 1e-4) / (1 - np.clip(P_hpa_ens, 1e-4, 1 - 1e-4)))
P_oof_ctx, P_hpa_ctx = P_oof_ens.copy(), P_hpa_ens.copy()
for c in range(nC):
    Z = np.c_[lg_ens[:, c], np.nan_to_num(NBL[:, c]), has_nb]
    Z[test_rows, 0] = lg_hpa[:, c]
    for k in range(5):
        tr, te = gold_rows[part[gold_rows] != k], np.flatnonzero(part == k)
        P_oof_ctx[te, c] = LogisticRegression(max_iter=1000).fit(Z[tr], GOLD[tr, c]).predict_proba(Z[te])[:, 1]
    P_hpa_ctx[:, c] = LogisticRegression(max_iter=1000).fit(Z[gold_rows], GOLD[gold_rows, c]).predict_proba(Z[test_rows])[:, 1]
print(f'Proteine con partner etichettati: training {has_nb[gold_rows].sum():,}, test HPA {has_nb[test_rows].sum():,}')
con_part = part >= 0
for nome_c, P_c in [('rete', P_oof_nn), ('rete avanzata', P_oof_adv), ('k-NN', knn_oof), ('alberi', P_oof_hgb), ('logistica', P_oof_lr),
                    ('ensemble', P_oof_ens), ('partner', P_oof_ctx)]:
    assert np.isfinite(P_c[con_part]).all(), f'{nome_c}: manca una predizione fuori fold per qualche proteina con partizione'
print(f'Predizioni fuori fold complete per tutti i modelli: {int(con_part.sum()):,} proteine (curate e aggiunte)')

## 8d · Lo stato dell'arte a parità di condizioni
Tre metodi, un solo protocollo: quello di **DeepLoc 2.0**, preso dal suo codice. Etichette curate di DeepLoc (proteine
umane, 9 classi + membrana), training sulle 4 partizioni diverse da k, validazione sul 9% scelto a caso (lo stesso per
tutti), focal loss con i pesi dei positivi di DeepLoc, AdamW con lr 1e-3, ReduceLROnPlateau, arresto dopo 5 epoche,
massimo 14 epoche, 3 semi × 5 fold. **Nessun iperparametro è scelto guardando le nostre proteine.**

| | ingressi | aggregazione e classificatore |
|---|---|---|
| DeepLoc 2.0 | ProtT5 per residuo, 4.000 residui | attenzione con filtro gaussiano, supervisionata dai segnali di smistamento |
| Wefers ProtT5-MHA | gli stessi | auto-attenzione a 8 teste, MLP con BatchNorm (dropout, lunghezza e lotti: i valori di DeepLoc) |
| NeuroLoc-Δ, sola ProtT5 | gli stessi | 7 riassunti fissi dei residui (con le due estremità), la nostra rete a blocchi |
| NeuroLoc-Δ, rappresentazione completa | ESM-2 + ProtT5, descrittori, motivi, struttura | la nostra rete a blocchi |

La riga «sola ProtT5» ha gli stessi ingressi delle repliche: misura solo il metodo. La differenza tra le due righe di
NeuroLoc-Δ misura il valore degli ingressi in più. Limite dichiarato: con circa 6.400 proteine umane i modelli ad
attenzione, pensati per 20–28.000 proteine, possono essere svantaggiati.


In [ ]:
from transformers import T5EncoderModel, T5Tokenizer
import math
import shutil

CLIP_DL = 4000
seq4k = proteins['sequence'].where(proteins['sequence'].str.len() <= CLIP_DL,
                                   proteins['sequence'].str[:CLIP_DL // 2] + proteins['sequence'].str[-(CLIP_DL // 2):])
L4 = seq4k.str.len().to_numpy()
off4 = np.r_[0, np.cumsum(L4)]
path_r5, path_r5d = DATA / 'res_prott5_4k.npy', DATA / 'res_prott5_4k_DONE.npy'
print(f'Embedding per residuo di ProtT5: {int(off4[-1]):,} residui, {off4[-1] * 1024 * 2 / 1e9:.1f} GB su disco '
      f'(liberi ora: {shutil.disk_usage(DATA).free / 1e9:.0f} GB)')
assert path_r5.exists() or shutil.disk_usage(DATA).free > 1.05 * off4[-1] * 1024 * 2, \
    f'servono {off4[-1] * 1024 * 2 / 1e9:.0f} GB liberi in {DATA.resolve()} per gli embedding per residuo'
R5 = np.lib.format.open_memmap(path_r5, mode=['w+', 'r+'][int(path_r5.exists())], dtype=np.float16, shape=(int(off4[-1]), 1024))
DONE_R5 = np.lib.format.open_memmap(path_r5d, mode=['w+', 'r+'][int(path_r5d.exists())], dtype=bool, shape=(N_PROT,))
todo5 = np.flatnonzero(~DONE_R5[:])
todo5 = todo5[np.argsort(-L4[todo5], kind='stable')]
# lotti per numero di residui (le proteine più lunghe da sole), al massimo 32 proteine
inizi5 = [0]
while inizi5[-1] < len(todo5):
    inizi5.append(inizi5[-1] + int(min(32, max(1, TOKENS // L4[todo5[inizi5[-1]]]))))
print(f'Proteine da codificare: {len(todo5):,} in {len(inizi5) - 1:,} lotti')
for _ in range(int(len(todo5) > 0)):
    tok_r5 = T5Tokenizer.from_pretrained(T5, do_lower_case=False)
    t5_r = T5EncoderModel.from_pretrained(T5, torch_dtype=DTYPE).to(DEVICE).eval()
    for n_l, (a_, b_) in enumerate(tqdm(list(zip(inizi5[:-1], inizi5[1:])), desc='ProtT5 per residuo', unit='lotto')):
        idx_ = todo5[a_:b_]
        # come DeepLoc: amminoacidi rari (U, Z, O, B) → X, residui separati da spazi; il token finale </s> si toglie
        enc_ = tok_r5([' '.join(re.sub('[UZOB]', 'X', s_)) for s_ in seq4k.to_numpy()[idx_]], return_tensors='pt', padding=True).to(DEVICE)
        with torch.no_grad():
            h_ = t5_r(input_ids=enc_['input_ids'], attention_mask=enc_['attention_mask']).last_hidden_state.float().cpu().numpy()
        for j_, i_ in enumerate(idx_):
            R5[off4[i_]:off4[i_ + 1]] = h_[j_, :L4[i_]].astype(np.float16)
        DONE_R5[idx_] = True
        for _ in range(int(n_l % 200 == 199)):
            R5.flush()
            DONE_R5.flush()
    R5.flush()
    DONE_R5.flush()
    del t5_r
    torch.cuda.empty_cache()
assert bool(np.all(DONE_R5[:])), 'embedding per residuo di ProtT5 incompleti: rilancia questa cella'
R5 = np.load(path_r5, mmap_mode='r')


In [ ]:
# 8d-2 · DeepLoc 2.0 "Accurate" rifatto dal suo codice (src/model.py, src/attr_prior.py, src/data.py, train_sl.py del
# repository TviNet/DeepLoc-2.0).
# PROTOCOLLO COMUNE, definito qui e usato da TUTTI i metodi della sezione 8d (DeepLoc 2.0, Wefers, NeuroLoc-Δ): sono le
# scelte di DeepLoc 2.0, prese dal suo codice e non dai nostri dati.
#   - etichette: le 9 classi curate di DeepLoc (proteine umane) più "proteina di membrana", stessi pesi dei positivi;
#   - per il fold k: training sulle 4 partizioni diverse da k; validazione = il 9% scelto a caso (come le loro 2.048
#     proteine su circa 22.600), LO STESSO per tutti i metodi, seme per seme;
#   - perdita: focal (γ = 1) con i pesi dei positivi di DeepLoc; AdamW, lr 1e-3; ReduceLROnPlateau (×0,1, pazienza 1,
#     minimo 1e-5) sulla perdita di validazione; arresto dopo 5 epoche senza miglioramento; massimo 14 epoche; si
#     tengono i pesi dell'epoca migliore;
#   - nessun iperparametro si sceglie guardando le nostre proteine: nessun metodo è avvantaggiato dalla scelta.
VAL_FRAC_DL, MAX_EP_DL, PAZ_DL, LOTTO_RES_DL, LOTTO_MAX_DL = 2048 / 22642, 14, 5, 16384, 128
# i loro pesi dei positivi (pos_weights_bce), nell'ordine delle nostre 9 classi, più "membrana" (prima colonna del loro modello)
POS_W_DL = torch.tensor([1, 1, 3, 2.3, 4, 4.5, 6.6, 7.7, 32, 1], dtype=torch.float32, device=DEVICE)
# i loro pesi dei 9 tipi di segnale (pos_weights_annot): SP, TM, MT, CH, TH, NLS, NES, PTS, GPI
# la divisione training/validazione di ogni (seme, fold), comune a tutti i metodi
SPLIT_DL = {(s_, k_): np.random.default_rng(100 * s_ + k_).permutation(int((part[gold_rows] != k_).sum()))
            < round(int((part[gold_rows] != k_).sum()) * VAL_FRAC_DL) for s_ in SEMI3 for k_ in FOLDS}
POS_W_SEG = torch.tensor([0.23, 0.92, 0.98, 2.63, 5.64, 1.60, 2.37, 1.87, 2.03], dtype=torch.float32, device=DEVICE)
# il loro filtro gaussiano (sigma 2, tagliato a 1 sigma: 5 residui)
KER_DL = torch.tensor(np.exp(-np.arange(-2, 3) ** 2 / 8) / np.exp(-np.arange(-2, 3) ** 2 / 8).sum(), dtype=torch.float32, device=DEVICE).view(1, 1, 5)
memb = proteins.reindex(columns=['membrana_deeploc'])['membrana_deeploc'].to_numpy(float)
Y_DL = np.c_[np.nan_to_num(GOLD), np.nan_to_num(memb)].astype(np.float32)
M_DL = np.c_[GOLD_M, np.isfinite(memb)].astype(np.float32)
# segnali annotati per residuo, accorciati come la sequenza (0 = nessun segnale), nella stessa disposizione di R5
tag_s = proteins.reindex(columns=['segnali_deeploc'])['segnali_deeploc'].fillna('').astype(str)
tag_s = tag_s.where(tag_s.str.len() == proteins['sequence'].str.len(), '')
tag_s = tag_s.where(tag_s.str.len() <= CLIP_DL, tag_s.str[:CLIP_DL // 2] + tag_s.str[-(CLIP_DL // 2):])
con_tag = np.flatnonzero(tag_s.str.len().to_numpy() > 0)
TAGS = np.zeros(int(off4[-1]), np.int64)
TAGS[np.concatenate([np.arange(off4[i], off4[i + 1]) for i in con_tag] + [np.zeros(0, int)])] = np.concatenate(
    [np.frombuffer(tag_s.iloc[i].encode(), np.uint8).astype(np.int64) - 48 for i in con_tag] + [np.zeros(0, np.int64)])
print(f"DeepLoc 2.0 rifatto: proteine curate {len(gold_rows):,}, con i segnali annotati {int(np.isin(gold_rows, con_tag).sum()):,}, "
      f"con l'etichetta di membrana {int(np.isfinite(memb[gold_rows]).sum()):,}")

OOF_DL = np.full((len(SEMI3), N_PROT, nC + 1), np.nan, np.float32)
HPA_DL = []
for seme, k in tqdm([(s_, k_) for s_ in SEMI3 for k_ in FOLDS], desc='DeepLoc 2.0 rifatto', unit='modello'):
    g_tr = gold_rows[part[gold_rows] != k]
    in_val = SPLIT_DL[(seme, k)]
    # lotti come i loro: proteine in ordine di lunghezza decrescente, al massimo 16.384 residui e 128 proteine per lotto
    LOTTI = {}
    for nome_l, righe in [('tr', g_tr[~in_val]), ('va', g_tr[in_val]), ('pr', np.r_[np.flatnonzero(part == k), test_rows])]:
        o_ = righe[np.argsort(-L4[righe], kind='stable')]
        inizi = [0]
        while inizi[-1] < len(o_):
            inizi.append(inizi[-1] + int(min(LOTTO_MAX_DL, max(1, LOTTO_RES_DL // L4[o_[inizi[-1]]]))))
        LOTTI[nome_l] = [o_[a_:b_] for a_, b_ in zip(inizi[:-1], inizi[1:])]
    torch.manual_seed(100 * seme + k)
    net = nn.ModuleDict({'ln0': nn.LayerNorm(1024), 'lin': nn.Linear(1024, 256), 'ln1': nn.LayerNorm(256),
                         'q': nn.Linear(256, 1, bias=False), 'clf': nn.Linear(256, nC + 1)}).to(DEVICE)
    nn.init.normal_(net['q'].weight, mean=0.0, std=1 / 256)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.1, patience=1, min_lr=1e-5)
    best_val, best_state, bad = np.inf, {n_: v_.detach().clone() for n_, v_ in net.state_dict().items()}, 0
    Z_K = np.full((N_PROT, nC + 1), np.nan, np.float32)
    for ep in range(MAX_EP_DL + 1):
        ultimo = int(ep == MAX_EP_DL)
        for _ in range(ultimo):
            net.load_state_dict(best_state)                       # all'ultimo giro: i pesi dell'epoca migliore, e si predice
        fasi = [[('tr', LOTTI['tr']), ('va', LOTTI['va'])], [('pr', LOTTI['pr'])]][ultimo]
        fasi = [(f_, l_[:len(l_) * int(bad < PAZ_DL or ultimo == 1)]) for f_, l_ in fasi]      # arresto anticipato
        perd, n_p = 0.0, 0
        for fase, lista in fasi:
            net.train(fase == 'tr')
            for idx in lista:
                Lb = torch.tensor(L4[idx], device=DEVICE)
                Lmax = int(L4[idx].max())
                msk = torch.arange(Lmax, device=DEVICE)[None] < Lb[:, None]
                righe_r = np.concatenate([np.arange(off4[i], off4[i + 1]) for i in idx])
                X = torch.zeros((len(idx), Lmax, 1024), device=DEVICE)
                X[msk] = torch.from_numpy(np.asarray(R5[righe_r], np.float32)).to(DEVICE)
                with torch.set_grad_enabled(fase == 'tr'), torch.autocast(DEV_TYPE, dtype=torch.bfloat16, enabled=AMP):
                    h = net['ln1'](net['lin'](net['ln0'](X))).float()
                    sc = (h * net['q'].weight.float().view(1, 1, -1)).sum(-1)
                    # filtro gaussiano sequenza per sequenza: oltre la lunghezza si ripete l'ultimo valore (il loro "replicate")
                    sc = torch.where(msk, sc, sc.gather(1, (Lb - 1)[:, None]))
                    sc = F.conv1d(F.pad(sc[:, None], (2, 2), mode='replicate'), KER_DL).squeeze(1).masked_fill(~msk, float('-inf'))
                    att = torch.softmax(sc, 1)
                    z_dl = net['clf']((h * att[..., None]).sum(1)).float()
                    # focal loss (γ = 1) con i pesi dei positivi, come la loro focal_loss
                    y_ = torch.from_numpy(Y_DL[idx]).to(DEVICE)
                    m_ = torch.from_numpy(M_DL[idx]).to(DEVICE)
                    bce_w = F.binary_cross_entropy_with_logits(z_dl, y_, pos_weight=POS_W_DL, reduction='none')
                    pt_ = torch.exp(-F.binary_cross_entropy_with_logits(z_dl, y_, reduction='none'))
                    focal = ((1 - pt_) * bce_w * m_).sum() / m_.sum().clamp(min=1)
                    # KL tra la distribuzione uniforme sui residui di ogni segnale annotato e l'attenzione (la loro kld_loss)
                    tg = torch.zeros((len(idx), Lmax), dtype=torch.int64, device=DEVICE)
                    tg[msk] = torch.from_numpy(TAGS[righe_r]).to(DEVICE)
                    oh = F.one_hot(tg, 10)[..., 1:].float() * msk[..., None]
                    cnt = oh.sum(1)
                    s_log = torch.einsum('blt,bl->bt', oh, torch.log(att.clamp_min(1e-30)) * msk)
                    kl = (torch.where(cnt > 0, -torch.log(cnt.clamp_min(1)) - s_log / cnt.clamp_min(1), torch.zeros_like(cnt)) * POS_W_SEG).sum() \
                         / ((cnt > 0).sum() + 1e-5)
                    # prior di Fourier sull'attenzione (fourier_att_prior_loss_dct: DCT ortonormale, limite L // 6, morbidezza 0,2),
                    # proteina per proteina; serve solo in training (la validazione guarda la sola focal loss, come la loro)
                    reg = torch.zeros((), device=DEVICE)
                    for j_ in range(len(idx) * int(fase == 'tr')):
                        x_ = F.pad(att[j_, :int(L4[idx[j_]])][None, None], (8, 8), mode='replicate')[0, 0]
                        n_ = x_.numel()
                        vc_ = torch.fft.fft(torch.cat([x_[::2], x_[1::2].flip(0)]))
                        k_ = -torch.arange(n_, device=DEVICE) * math.pi / (2 * n_)
                        v_ = vc_.real * torch.cos(k_) - vc_.imag * torch.sin(k_)
                        v_ = 2 * torch.cat([v_[:1] / (math.sqrt(n_) * 2), v_[1:] / (math.sqrt(n_ / 2) * 2)])
                        mg_ = v_.abs()
                        mg_ = (mg_ / torch.where(mg_.sum() == 0, torch.ones_like(mg_.sum()), mg_.sum()))[1:]
                        fl_ = int(L4[idx[j_]]) // 6
                        w_ = torch.cat([torch.ones(fl_, device=DEVICE), 1 / (1 + torch.arange(1, mg_.numel() - fl_ + 1, device=DEVICE) ** 0.2)])
                        reg = reg + 1 - (mg_ * w_).sum()
                    loss = focal + 0.1 * kl + 0.1 * reg / len(idx)
                for _ in range(int(fase == 'tr')):
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    opt.step()
                perd, n_p = perd + float(focal.detach()) * len(idx) * int(fase == 'va'), n_p + len(idx) * int(fase == 'va')
                Z_K[idx] = [Z_K[idx], z_dl.detach().cpu().numpy()][int(fase == 'pr')]
        # fine epoca: perdita di validazione (focal), scheduler, miglior modello, pazienza
        for _ in range(int(len(fasi[-1][1]) > 0 and ultimo == 0)):
            v_ep = perd / max(n_p, 1)
            sched.step(v_ep)
            best_state = [best_state, {n_: v_.detach().clone() for n_, v_ in net.state_dict().items()}][int(v_ep < best_val)]
            bad = [bad + 1, 0][int(v_ep < best_val)]
            best_val = min(best_val, v_ep)
    OOF_DL[SEMI3.index(seme), part == k] = Z_K[part == k]
    HPA_DL.append(Z_K[test_rows])
    del net, opt
torch.cuda.empty_cache()
P_oof_dl2r = (1 / (1 + np.exp(-np.mean(OOF_DL, 0))))[:, :nC]
P_hpa_dl2r = (1 / (1 + np.exp(-np.mean(HPA_DL, 0))))[:, :nC]
print(f"DeepLoc 2.0 rifatto: AUPRC macro fuori fold (proteine curate) "
      f"{np.mean([average_precision_score(GOLD[gold_rows, c], P_oof_dl2r[gold_rows, c]) for c in range(nC)]):.4f}")


In [ ]:
# 8d-3 · ProtT5-MHA di Wefers et al. 2026: l'architettura dal loro codice (models/aggregation.py, MultiHeadAttentionPool:
# auto-attenzione a 8 teste con dropout 0,2 e media sui residui; models/mlp.py, MLPClassifier: 1.024 → 512 → 512 → 512
# con ReLU e BatchNorm), addestrata con il PROTOCOLLO COMUNE di DeepLoc 2.0 (cella sopra). Nessuna griglia sui nostri
# dati: le scelte che nel loro lavoro venivano dalla griglia prendono il valore di DeepLoc 2.0:
#   - lunghezza: 4.000 residui, cioè l'ingresso di DeepLoc (gli stessi embedding per residuo), senza ritagli casuali;
#   - dropout del classificatore: nessuno (DeepLoc non ne usa);
#   - perdita: la focal di DeepLoc, sulle stesse 10 uscite (9 classi + membrana).
# Lotti: quelli di DeepLoc 2.0 (proteine in ordine di lunghezza, al massimo 16.384 residui e 128 proteine per lotto), che
# tengono anche la memoria sotto controllo con l'attenzione su 4.000 residui. La BatchNorm non si addestra su un lotto di
# una proteina sola: in training un lotto da 1 (al più uno, l'ultimo, con le proteine più corte) si salta
OOF_WF = np.full((len(SEMI3), N_PROT, nC + 1), np.nan, np.float32)
HPA_WF = []
for seme, k in tqdm([(s_, k_) for s_ in SEMI3 for k_ in FOLDS], desc='Wefers ProtT5-MHA (protocollo DeepLoc 2.0)', unit='modello'):
    g_tr = gold_rows[part[gold_rows] != k]
    tr_i, va_i = g_tr[~SPLIT_DL[(seme, k)]], g_tr[SPLIT_DL[(seme, k)]]
    LOTTI = {}
    for nome_l, righe in [('tr', tr_i), ('va', va_i), ('pr', np.r_[np.flatnonzero(part == k), test_rows])]:
        o_ = righe[np.argsort(-L4[righe], kind='stable')]
        inizi = [0]
        while inizi[-1] < len(o_):
            inizi.append(inizi[-1] + int(min(LOTTO_MAX_DL, max(1, LOTTO_RES_DL // L4[o_[inizi[-1]]]))))
        LOTTI[nome_l] = [o_[a_:b_] for a_, b_ in zip(inizi[:-1], inizi[1:])]
    LOTTI['tr'] = list(compress(LOTTI['tr'], [len(b_) > 1 for b_ in LOTTI['tr']]))
    torch.manual_seed(2000 + 100 * seme + k)
    net = nn.ModuleDict({'mha': nn.MultiheadAttention(1024, 8, dropout=0.2, batch_first=True),
                         'mlp': nn.Sequential(nn.Linear(1024, 512), nn.ReLU(), nn.BatchNorm1d(512),
                                              *[m_ for _ in range(2) for m_ in (nn.Linear(512, 512), nn.ReLU(), nn.BatchNorm1d(512))],
                                              nn.Linear(512, nC + 1))}).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.1, patience=1, min_lr=1e-5)
    best_val, best_state, bad = np.inf, {n_: v_.detach().clone() for n_, v_ in net.state_dict().items()}, 0
    Z_K = np.full((N_PROT, nC + 1), np.nan, np.float32)
    for ep in range(MAX_EP_DL + 1):
        ultimo = int(ep == MAX_EP_DL)
        attivo = int(bad < PAZ_DL) * (1 - ultimo)                           # arresto anticipato
        for _ in range(ultimo):
            net.load_state_dict(best_state)                                 # all'ultimo giro: i pesi migliori, e si predice
        fasi = [('tr', LOTTI['tr'] * attivo), ('va', LOTTI['va'] * attivo), ('pr', LOTTI['pr'] * ultimo)]
        perd, n_p = 0.0, 0
        for fase, lista in fasi:
            net.train(fase == 'tr')
            for idx in lista:
                Lb = torch.tensor(L4[idx], device=DEVICE)
                msk = torch.arange(int(L4[idx].max()), device=DEVICE)[None] < Lb[:, None]
                X = torch.zeros((len(idx), int(L4[idx].max()), 1024), device=DEVICE)
                X[msk] = torch.from_numpy(np.asarray(R5[np.concatenate([np.arange(off4[i], off4[i + 1]) for i in idx])], np.float32)).to(DEVICE)
                with torch.set_grad_enabled(fase == 'tr'), torch.autocast(DEV_TYPE, dtype=torch.bfloat16, enabled=AMP):
                    a_out = net['mha'](X, X, X, key_padding_mask=~msk, need_weights=False)[0].float()
                    z_wf = net['mlp']((a_out * msk[..., None]).sum(1) / msk.sum(1, keepdim=True)).float()
                    # la focal loss di DeepLoc 2.0 (γ = 1, pesi dei positivi), come nella cella sopra
                    y_ = torch.from_numpy(Y_DL[idx]).to(DEVICE)
                    m_ = torch.from_numpy(M_DL[idx]).to(DEVICE)
                    bce_w = F.binary_cross_entropy_with_logits(z_wf, y_, pos_weight=POS_W_DL, reduction='none')
                    pt_ = torch.exp(-F.binary_cross_entropy_with_logits(z_wf, y_, reduction='none'))
                    focal = ((1 - pt_) * bce_w * m_).sum() / m_.sum().clamp(min=1)
                for _ in range(int(fase == 'tr')):
                    opt.zero_grad(set_to_none=True)
                    focal.backward()
                    opt.step()
                perd, n_p = perd + float(focal.detach()) * len(idx) * int(fase == 'va'), n_p + len(idx) * int(fase == 'va')
                Z_K[idx] = [Z_K[idx], z_wf.detach().cpu().numpy()][int(fase == 'pr')]
        # fine epoca: perdita di validazione, scheduler, miglior modello, pazienza (come DeepLoc 2.0)
        for _ in range(attivo):
            v_ep = perd / max(n_p, 1)
            sched.step(v_ep)
            best_state = [best_state, {n_: v_.detach().clone() for n_, v_ in net.state_dict().items()}][int(v_ep < best_val)]
            bad = [bad + 1, 0][int(v_ep < best_val)]
            best_val = min(best_val, v_ep)
    OOF_WF[SEMI3.index(seme), part == k] = Z_K[part == k]
    HPA_WF.append(Z_K[test_rows])
    del net, opt
torch.cuda.empty_cache()
P_oof_wefr = (1 / (1 + np.exp(-np.mean(OOF_WF, 0))))[:, :nC]
P_hpa_wefr = (1 / (1 + np.exp(-np.mean(HPA_WF, 0))))[:, :nC]
print(f"Wefers ProtT5-MHA (protocollo DeepLoc 2.0): AUPRC macro fuori fold (proteine curate) "
      f"{np.mean([average_precision_score(GOLD[gold_rows, c], P_oof_wefr[gold_rows, c]) for c in range(nC)]):.4f}")


In [ ]:
# 8d-4 · NeuroLoc-Δ con il PROTOCOLLO COMUNE di DeepLoc 2.0 (cella 8d-2): stesse etichette, stessa divisione, stesse
# scelte di addestramento, nessun iperparametro scelto sui nostri dati. Due ingressi:
#   - "sola ProtT5": 7 riassunti (media, massimo, primi 20 e 60 residui, terzo centrale, ultimi 15 e 50) calcolati dagli
#     STESSI embedding per residuo di DeepLoc 2.0 e Wefers (ProtT5, sequenza accorciata a 4.000). Gli ingressi sono
#     identici: cambia solo come si riassumono i residui (riassunti fissi che tengono le due estremità, dove stanno i
#     segnali di smistamento, contro un'attenzione da imparare) e il classificatore;
#   - "rappresentazione completa": i blocchi di NeuroLoc-Δ fissati a priori come la combinazione più ampia della griglia
#     (ESM-2 e ProtT5 con tutti i riassunti, descrittori di smistamento, motivi terminali annidati per fold, struttura
#     AlphaFold), senza guardare la CV.
# La nostra architettura con le misure di DeepLoc 2.0: ogni blocco → lineare a 256 + LayerNorm (256 è la larghezza di
# DeepLoc), poi uno strato nascosto da 256 e l'uscita a 9 classi + membrana. Nessun dropout (DeepLoc non ne usa), nessuna
# media mobile dei pesi, nessuna perdita ausiliaria (frazioni, meccanismi): solo le etichette di DeepLoc. Lotti casuali
# da 128 proteine (il massimo per lotto di DeepLoc): gli ingressi sono vettori, non sequenze da impacchettare per residui
RIASSUNTI_R5 = ['mean', 'max', 'n20', 'n60', 'centro', 'c15', 'c50']
T5R = np.zeros((N_PROT, len(RIASSUNTI_R5) * 1024), np.float16)
for i in tqdm(range(N_PROT), desc='riassunti dei residui ProtT5', unit='proteina'):
    r_ = np.asarray(R5[off4[i]:off4[i + 1]], np.float32)
    L_ = len(r_)
    T5R[i] = np.concatenate([r_.mean(0), r_.max(0), r_[:20].mean(0), r_[:60].mean(0), r_[L_ // 3:2 * L_ // 3].mean(0),
                             r_[-15:].mean(0), r_[-50:].mean(0)]).astype(np.float16)
XR = torch.tensor(T5R, device=DEVICE)
BLOCCHI_C = ESM_B + T5_B + ['esm_max'] + ['t5_max'] * USE_T5 + HM
INGRESSI_NL = ['sola ProtT5', 'rappresentazione completa']
OOF_NL = {n_: np.full((len(SEMI3), N_PROT, nC + 1), np.nan, np.float32) for n_ in INGRESSI_NL}
HPA_NL = {n_: [] for n_ in INGRESSI_NL}
for nome_i, seme, k in tqdm([(n_, s_, k_) for n_ in INGRESSI_NL for s_ in SEMI3 for k_ in FOLDS], desc='NeuroLoc-Δ (protocollo DeepLoc 2.0)', unit='modello'):
    completa = int(nome_i == 'rappresentazione completa')
    X_all = [XR, XG][completa]
    blocchi_k = [{'motivi': f'motivi@{k}'}.get(b_, b_) for b_ in BLOCCHI_C]
    cols = [torch.arange(XR.shape[1], device=DEVICE),
            torch.tensor(np.concatenate([np.arange(start_of[b], start_of[b] + BLOCKS[b].shape[1]) for b in blocchi_k]), device=DEVICE)][completa]
    widths = [[1024] * len(RIASSUNTI_R5), [BLOCKS[b].shape[1] for b in blocchi_k]][completa]
    offs = np.cumsum([0] + widths)
    g_tr = gold_rows[part[gold_rows] != k]
    tr_i, va_i = g_tr[~SPLIT_DL[(seme, k)]], g_tr[SPLIT_DL[(seme, k)]]
    pr_i = np.r_[np.flatnonzero(part == k), test_rows]
    # standardizzazione stimata sul solo training del fold
    tr_t = torch.tensor(tr_i, device=DEVICE)
    s1 = sum(X_all[c_[:, None], cols].double().sum(0) for c_ in tr_t.split(CHUNK))
    s2 = sum(X_all[c_[:, None], cols].double().pow(2).sum(0) for c_ in tr_t.split(CHUNK))
    mu = (s1 / len(tr_i)).float()
    sd = ((s2 / len(tr_i) - (s1 / len(tr_i)) ** 2).clamp(min=0).sqrt() + 1e-3).float()
    rs_ = np.random.default_rng(3000 + 100 * seme + k)
    torch.manual_seed(3000 + 100 * seme + k)
    net = nn.ModuleDict({'proj': nn.ModuleList([nn.Sequential(nn.Linear(int(w_), 256), nn.LayerNorm(256)) for w_ in widths]),
                         'body': nn.Sequential(nn.GELU(), nn.Flatten(1), nn.Linear(len(widths) * 256, 256), nn.LayerNorm(256), nn.GELU(),
                                               nn.Linear(256, nC + 1))}).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.1, patience=1, min_lr=1e-5)
    best_val, best_state, bad = np.inf, {n_: v_.detach().clone() for n_, v_ in net.state_dict().items()}, 0
    Z_K = np.full((N_PROT, nC + 1), np.nan, np.float32)
    for ep in range(MAX_EP_DL + 1):
        ultimo = int(ep == MAX_EP_DL)
        attivo = int(bad < PAZ_DL) * (1 - ultimo)
        for _ in range(ultimo):
            net.load_state_dict(best_state)
        ord_tr = rs_.permutation(tr_i)
        fasi = [('tr', [ord_tr[a_:a_ + LOTTO_MAX_DL] for a_ in range(0, len(ord_tr), LOTTO_MAX_DL)] * attivo),
                ('va', [va_i[a_:a_ + 1024] for a_ in range(0, len(va_i), 1024)] * attivo),
                ('pr', [pr_i[a_:a_ + 1024] for a_ in range(0, len(pr_i), 1024)] * ultimo)]
        perd, n_p = 0.0, 0
        for fase, lista in fasi:
            net.train(fase == 'tr')
            for idx in lista:
                xb = ((X_all[torch.tensor(idx, device=DEVICE)[:, None], cols].float() - mu) / sd).clamp(-20, 20)
                with torch.set_grad_enabled(fase == 'tr'), torch.autocast(DEV_TYPE, dtype=torch.bfloat16, enabled=AMP):
                    z_nl = net['body'](torch.stack([pj(xb[:, b0:b1]) for pj, (b0, b1) in zip(net['proj'], zip(offs[:-1], offs[1:]))], 1)).float()
                    y_ = torch.from_numpy(Y_DL[idx]).to(DEVICE)
                    m_ = torch.from_numpy(M_DL[idx]).to(DEVICE)
                    bce_w = F.binary_cross_entropy_with_logits(z_nl, y_, pos_weight=POS_W_DL, reduction='none')
                    pt_ = torch.exp(-F.binary_cross_entropy_with_logits(z_nl, y_, reduction='none'))
                    focal = ((1 - pt_) * bce_w * m_).sum() / m_.sum().clamp(min=1)
                for _ in range(int(fase == 'tr')):
                    opt.zero_grad(set_to_none=True)
                    focal.backward()
                    opt.step()
                perd, n_p = perd + float(focal.detach()) * len(idx) * int(fase == 'va'), n_p + len(idx) * int(fase == 'va')
                Z_K[idx] = [Z_K[idx], z_nl.detach().cpu().numpy()][int(fase == 'pr')]
        for _ in range(attivo):
            v_ep = perd / max(n_p, 1)
            sched.step(v_ep)
            best_state = [best_state, {n_: v_.detach().clone() for n_, v_ in net.state_dict().items()}][int(v_ep < best_val)]
            bad = [bad + 1, 0][int(v_ep < best_val)]
            best_val = min(best_val, v_ep)
    OOF_NL[nome_i][SEMI3.index(seme), part == k] = Z_K[part == k]
    HPA_NL[nome_i].append(Z_K[test_rows])
    del net, opt
del XR
torch.cuda.empty_cache()
P_oof_nlpt, P_hpa_nlpt, P_oof_nlrc, P_hpa_nlrc = [(1 / (1 + np.exp(-np.mean(A_, 0))))[:, :nC]
                                                  for n_ in INGRESSI_NL for A_ in [OOF_NL[n_], HPA_NL[n_]]]
print('NeuroLoc-Δ con il protocollo di DeepLoc 2.0, AUPRC macro fuori fold (proteine curate):',
      {n_: round(float(np.mean([average_precision_score(GOLD[gold_rows, c], P_[gold_rows, c]) for c in range(nC)])), 4)
       for n_, P_ in [('sola ProtT5', P_oof_nlpt), ('rappresentazione completa', P_oof_nlrc)]})


## 9 · Confronto dei modelli sul test HPA indipendente (con DeepLoc 2.0 e Wefers et al. rifatti)
Per ogni modello, con lo stesso metodo: **soglie MCC** per classe e **soglie Jaccard** ; soglie **ad alta sensibilità** (sotto di esse cade il 10% dei positivi curati,
fuori fold: sul test HPA il richiamo misurato è più basso del 90%, perché le etichette hanno un'altra fonte, quindi non
è un "insieme conformale al 90%")

In [ ]:
STAGE_NAMES = {'stile_deeploc2': 'Riferimento semplice (ProtT5 medio + MLP sulle etichette DeepLoc; non è DeepLoc 2.0)',
               'stile_deeploc2_stessi_dati': 'Riferimento semplice sugli stessi dati (tutte le fonti, pool esteso)',
               'con_scheda': 'Rete + scheda testuale (aumentata dalla conoscenza dei curatori: non solo sequenza)',
               'abl_solo_medie': 'Variante: solo embedding medi, descrittori e struttura',
               'abl_senza_struct': 'Variante: senza struttura 3D AlphaFold',
               'abl_solo_esm2': 'Variante: solo ESM-2 (senza ProtT5)',
               'abl_senza_sorting': 'Variante: senza descrittori, motivi e struttura',
               'abl_con_attenzione': 'Variante: con attenzione tra token (Transformer)',
               'abl_solo_deeploc': 'Dati: etichette DeepLoc senza le altre fonti (restano frazioni e ausiliari)', 'abl_deeploc_fonti': 'Dati: DeepLoc + fonti (stesse proteine)',
               'abl_senza_hpa': 'Dati: senza le fonti HPA (le etichette del test sono HPA)'}
MODELS = {'Baseline (prevalenza)': (P_oof_dummy, P_hpa_dummy), 'Regressione logistica': (P_oof_lr, P_hpa_lr),
          'Rete neurale': (P_oof_nn, P_hpa_nn),
          # con α = 0 il k-NN non cambia nulla e la riga sarebbe un doppione della rete: entra solo se α > 0
          **dict([('Rete neurale + k-NN', (P_oof_knn, P_hpa_knn))][:int(ALPHA > 0)]),
          'Gradient boosting (alberi)': (P_oof_hgb, P_hpa_hgb), ENS_NAME: (P_oof_ens, P_hpa_ens),
          'Rete avanzata NeuroLoc-LA (attenzione sui residui + fusione)': (P_oof_adv, P_hpa_adv),
          'Ensemble + partner molecolari (non solo sequenza)': (P_oof_ctx, P_hpa_ctx),
          # lo stato dell'arte rifatto sulla stessa macchina, sugli stessi dati e con le stesse partizioni (sezione 8d)
          'DeepLoc 2.0 rifatto (stesse etichette, protocollo DeepLoc 2.0)': (P_oof_dl2r, P_hpa_dl2r), 'Wefers ProtT5-MHA (stesse etichette, protocollo DeepLoc 2.0)': (P_oof_wefr, P_hpa_wefr),
          'NeuroLoc-Δ, sola ProtT5 (stesse etichette e stessi ingressi, protocollo DeepLoc 2.0)': (P_oof_nlpt, P_hpa_nlpt), 'NeuroLoc-Δ, rappresentazione completa (stesse etichette, protocollo DeepLoc 2.0)': (P_oof_nlrc, P_hpa_nlrc)}
# ogni stadio è la media dei suoi 3 semi, come "Rete neurale" (lo stadio `cervello` riaddestra la rete vincente per lo
# 03 e sarebbe un doppione: non entra in tabella)
for stg, nome in STAGE_NAMES.items():
    MODELS[nome] = ((1 / (1 + np.exp(-np.stack([RUNS[r]['oof'][:, :nC] for r in RANK[stg]['run']])))).mean(0),
                    (1 / (1 + np.exp(-np.stack([RUNS[r]['hpa'][:, :nC] for r in RANK[stg]['run']])))).mean(0))

# l'intestazione del FASTA è l'ID del test (ENSP...), che non cambia tra un'esecuzione e l'altra: con il numero di riga
# (pid) i CSV scaricati dal server smettevano di corrispondere appena lo 01 veniva rilanciato
test_fa = proteins.loc[test_rows, ['id_test', 'sequence']]
for i0 in range(0, len(test_fa), 500):
    (DATA / f'hpa_test_per_deeploc2_{i0 // 500 + 1}.fasta').write_text(
        ''.join(f'>{p}\n{q}\n' for p, q in test_fa.iloc[i0:i0 + 500].itertuples(index=False)))
dl2 = pd.concat([pd.read_csv(f) for f in sorted((DATA / 'esterni').glob('deeploc2_*.csv'))] +
                [pd.DataFrame(columns=['Protein_ID', 'Localizations'] + CLASSES)], ignore_index=True)
dl2 = dl2.assign(pid=dl2['Protein_ID'].astype(str).str.split().str[0]).drop_duplicates('pid').set_index('pid')
dl2 = dl2.reindex(test_fa['id_test'].astype(str))
P_hpa_dl2 = dl2.reindex(columns=CLASSES).astype(float).to_numpy()
HAS_DL2 = int(np.isfinite(P_hpa_dl2).all())
OFFICIAL = [('DeepLoc 2.0 ufficiale', P_hpa_dl2)][:HAS_DL2]
print(f'Predizioni ufficiali di DeepLoc 2.0: coprono {np.isfinite(P_hpa_dl2).all(1).mean():.0%} delle {len(test_fa):,} proteine del test → '
      f'{["NON usate (servono tutte)", "usate"][HAS_DL2]}')
print(['>>> Senza le predizioni ufficiali il confronto con lo stato dell\'arte NON c\'è: il "riferimento semplice" non è DeepLoc 2.0. '
       'Carica i file data/hpa_test_per_deeploc2_*.fasta sul server (vedi il testo sopra) e rilancia da questa cella.', ''][HAS_DL2])

EPS, SCALES, grid_t = 0.10, np.linspace(0.8, 3.2, 49), np.linspace(0.02, 0.95, 94)
ev = [CLASSES.index(c) for c in HPA_EVAL]
y_true = Y_test[:, ev].astype(int)
Yg = GOLD[gold_rows].astype(bool)
npos, nneg = Yg.sum(0)[:, None], (~Yg).sum(0)[:, None]
THR, rows_tab = {}, []
for name, (Po, Ph) in MODELS.items():
    Pg = Po[gold_rows]
    pr = Pg[:, :, None] >= grid_t[None, None]                                    # [proteina, classe, soglia]
    tp, fp = (pr & Yg[:, :, None]).sum(0).astype(float), (pr & ~Yg[:, :, None]).sum(0).astype(float)
    fn, tn = npos - tp, nneg - fp
    with np.errstate(all='ignore'):
        mcc = np.nan_to_num((tp * tn - fp * fn) / np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
    thr_mcc = grid_t[mcc.argmax(1)]
    top = Pg == Pg.max(1, keepdims=True)
    pred_s = (Pg[None] >= (thr_mcc[None] * SCALES[:, None]).clip(0, .99)[:, None]) | top[None]      # [scala, proteina, classe]
    jac = ((pred_s & Yg[None]).sum(2) / (pred_s | Yg[None]).sum(2)).mean(1)
    thr_jac = (thr_mcc * SCALES[int(np.argmax(jac))]).clip(0, .99)
    Ph_u, Y_u = Po[human_gold], GOLD[human_gold].astype(int)
    # soglia ad alta sensibilità: sotto di essa cade il 10% dei positivi curati (fuori fold). Sul test HPA il richiamo
    # misurato è più basso del 90% (etichette di un'altra fonte): per questo non si chiama "insieme conformale al 90%"
    thr_conf = np.array([np.quantile(np.r_[Ph_u[Y_u[:, c] == 1, c], 0.0], EPS) for c in range(nC)])
    THR[name] = dict(mcc=thr_mcc, jac=thr_jac, conf=thr_conf)
    cover = ((Ph >= thr_conf) & (Y_test == 1))[:, ev].sum() / max((Y_test[:, ev] == 1).sum(), 1)
    auc_cv = np.mean([roc_auc_score(Yg[:, c], Pg[:, c]) for c in range(nC)])
    ap_cv = np.mean([average_precision_score(Yg[:, c], Pg[:, c]) for c in range(nC)])
    auc_te = np.mean([roc_auc_score(y_true[:, j], Ph[:, c]) for j, c in enumerate(ev)])
    ap_te = np.mean([average_precision_score(y_true[:, j], Ph[:, c]) for j, c in enumerate(ev)])
    # "microscopia": la classe forzata è la più probabile tra quelle che il test può osservare (domanda 11)
    for tname, t, force in [('MCC', thr_mcc, range(nC)), ('Jaccard', thr_jac, range(nC)), ('Jaccard · microscopia', thr_jac, ev)]:
        Pf = np.where(np.isin(np.arange(nC), list(force)), Ph, -1.0)
        y_pred = np.maximum(Ph >= t, Pf == Pf.max(1, keepdims=True)).astype(int)[:, ev]
        rows_tab.append({'modello': name, 'soglie': tname, 'AUC ROC (CV)': auc_cv, 'AUPRC (CV)': ap_cv,
                         'AUC ROC (test)': auc_te, 'AUPRC (test)': ap_te,
                         'Labels predette': y_pred.sum(1).mean(), 'Accuracy': (y_true == y_pred).all(1).mean(),
                         'Jaccard': jaccard_score(y_true, y_pred, average='samples', zero_division=0),
                         'Micro F1': f1_score(y_true, y_pred, average='micro', zero_division=0),
                         'Macro F1': f1_score(y_true, y_pred, average='macro', zero_division=0),
                         **{f'MCC {c}': matthews_corrcoef(y_true[:, j], y_pred[:, j]) for j, c in enumerate(HPA_EVAL)},
                         'Richiamo sul test alla soglia del 90% (CV)': cover})
for name, Ph in OFFICIAL:
    y_pred = np.stack([dl2['Localizations'].astype(str).str.contains(c, regex=False) for c in HPA_EVAL], 1).astype(int)
    rows_tab.append({'modello': name, 'soglie': 'sue (server)',
                     'AUC ROC (test)': np.mean([roc_auc_score(y_true[:, j], Ph[:, c]) for j, c in enumerate(ev)]),
                     'AUPRC (test)': np.mean([average_precision_score(y_true[:, j], Ph[:, c]) for j, c in enumerate(ev)]),
                     'Labels predette': y_pred.sum(1).mean(), 'Accuracy': (y_true == y_pred).all(1).mean(),
                     'Jaccard': jaccard_score(y_true, y_pred, average='samples', zero_division=0),
                     'Micro F1': f1_score(y_true, y_pred, average='micro', zero_division=0),
                     'Macro F1': f1_score(y_true, y_pred, average='macro', zero_division=0),
                     **{f'MCC {c}': matthews_corrcoef(y_true[:, j], y_pred[:, j]) for j, c in enumerate(HPA_EVAL)}})
confronto = pd.DataFrame(rows_tab).set_index(['modello', 'soglie'])
# i valori pubblicati di DeepLoc 2.0 (test intero di 1.717 proteine: ProtT5 Jaccard 0,53, Accuracy 0,39, Micro F1 0,60) NON
# sono in tabella: altro insieme, più isoforme per gene, altre soglie. Accanto ai nostri sembrerebbero un confronto
print(f'Test HPA: {len(test_rows):,} proteine del cervello, etichette reali per proteina {y_true.sum(1).mean():.2f}')
print(confronto.round(3).T.to_string())
confronto.round(3).to_excel(DATA / 'confronto_modelli.xlsx')

## 10 · Quale metodologia scegliere? Bootstrap appaiato sul test e scelta sulla CV

**Confronto con lo stato dell'arte (il risultato principale).** È quello «a parità di condizioni» della sezione 8d:
DeepLoc 2.0 rifatto, Wefers ProtT5-MHA e NeuroLoc-Δ (sola ProtT5, con gli stessi ingressi delle repliche, e
rappresentazione completa), tutti con le stesse etichette, la stessa divisione e il protocollo di addestramento di
DeepLoc 2.0, senza iperparametri scelti sui nostri dati. Si legge sul **test HPA**; le differenze hanno il p corretto
con Holm. Il sistema completo (dati in più, configurazione scelta in CV) è contesto: la sua AUPRC in CV è ottimistica.
Si ricampionano le proteine del test 2.000 volte e ogni volta si calcola l'AUC macro di **tutti** i modelli sulle stesse
proteine: da qui l'intervallo al 95% di ogni modello e della sua differenza dal riferimento semplice. 

La **scelta**: tra i modelli di sola sequenza quelli entro 2 deviazioni standard dalla migliore AUPRC in CV sono
equivalenti, e si prende il più semplice (l'ordine di semplicità è scritto nel codice prima dei risultati). Le
varianti hanno 3 semi come la rete vincente: tutti i modelli a confronto sono la media di 15 reti, e la tabella "una
rete per seme" dice se una differenza supera il rumore. La CV misura 9 classi con etichette curate, il test 6 classi
con etichette HPA: per questo c'è anche la colonna dell'AUPRC in CV sulle sole 6 classi del test.

In [ ]:
B = 2000
n_t = len(test_rows)
W = np.bincount((rng.integers(0, n_t, (B, n_t)) + n_t * np.arange(B)[:, None]).ravel(), minlength=B * n_t).reshape(B, n_t).astype(float)
ALL_TEST = [(n, m[1]) for n, m in MODELS.items()] + OFFICIAL
SOTA = 'Riferimento semplice (ProtT5 medio + MLP sulle etichette DeepLoc; non è DeepLoc 2.0)'
SAME = 'Riferimento semplice sugli stessi dati (tutte le fonti, pool esteso)'
CTX_MODEL = 'Ensemble + partner molecolari (non solo sequenza)'
ADVN = 'Rete avanzata NeuroLoc-LA (attenzione sui residui + fusione)'
SCHEDA = STAGE_NAMES['con_scheda']
# bootstrap appaiato di AUC e di AUPRC macro: lo stesso ricampionamento (pesi W) per tutti i modelli
boot, boot_ap = np.zeros((B, len(ALL_TEST))), np.zeros((B, len(ALL_TEST)))
for m_i, (_, Ph) in enumerate(ALL_TEST):
    auc_c, ap_c = [], []
    for j, c in enumerate(ev):
        order = np.argsort(Ph[:, c], kind='mergesort')
        starts = np.r_[0, np.flatnonzero(np.diff(Ph[order, c])) + 1]           # gruppi di punteggi uguali
        yo = y_true[order, j].astype(bool)
        gpos = np.add.reduceat(W[:, order] * yo, starts, axis=1)
        gneg = np.add.reduceat(W[:, order] * ~yo, starts, axis=1)
        # AUPRC pesata: proteine in ordine di punteggio decrescente; per ogni positivo la precisione (pesata) fin lì
        desc = order[::-1]
        wp_ = W[:, desc] * y_true[desc, j]
        with np.errstate(all='ignore'):
            auc_c.append((gpos * (np.cumsum(gneg, 1) - gneg + 0.5 * gneg)).sum(1) / (gpos.sum(1) * gneg.sum(1)))
            ap_c.append((wp_ * np.nan_to_num(np.cumsum(wp_, 1) / np.cumsum(W[:, desc], 1))).sum(1) / wp_.sum(1))
    boot[:, m_i], boot_ap[:, m_i] = np.nanmean(auc_c, 0), np.nanmean(ap_c, 0)
names = [n for n, _ in ALL_TEST]
d_sota, d_same = boot - boot[:, [names.index(SOTA)]], boot - boot[:, [names.index(SAME)]]
d_same_ap = boot_ap - boot_ap[:, [names.index(SAME)]]
ev_cv = [CLASSES.index(c) for c in HPA_EVAL]
scelta = pd.DataFrame({'AUPRC macro (CV, scelta)': [np.mean([average_precision_score(Yg[:, c], MODELS[n][0][gold_rows, c]) for c in range(nC)])
                                                    for n in MODELS] + [np.nan] * HAS_DL2,
                       # la CV misura 9 classi, il test 6: la stessa AUPRC in CV sulle sole 6 classi del test
                       'AUPRC macro (CV, 6 classi del test)': [np.mean([average_precision_score(Yg[:, c], MODELS[n][0][gold_rows, c]) for c in ev_cv])
                                                               for n in MODELS] + [np.nan] * HAS_DL2,
                       'AUC macro (test)': [np.mean([roc_auc_score(y_true[:, j], Ph[:, c]) for j, c in enumerate(ev)]) for _, Ph in ALL_TEST],
                       'IC 95% basso': np.nanquantile(boot, 0.025, 0), 'IC 95% alto': np.nanquantile(boot, 0.975, 0),
                       'Δ vs riferimento semplice': np.nanmean(d_sota, 0), 'Δ IC basso': np.nanquantile(d_sota, 0.025, 0),
                       'Δ IC alto': np.nanquantile(d_sota, 0.975, 0),
                       'Δ vs riferimento stessi dati': np.nanmean(d_same, 0), 'Δ stessi dati IC basso': np.nanquantile(d_same, 0.025, 0),
                       'Δ stessi dati IC alto': np.nanquantile(d_same, 0.975, 0),
                       'AUPRC macro (test)': [np.mean([average_precision_score(y_true[:, j], Ph[:, c]) for j, c in enumerate(ev)]) for _, Ph in ALL_TEST],
                       'AUPRC IC basso': np.nanquantile(boot_ap, 0.025, 0), 'AUPRC IC alto': np.nanquantile(boot_ap, 0.975, 0),
                       'Δ AUPRC vs riferimento stessi dati': np.nanmean(d_same_ap, 0), 'Δ AUPRC IC basso': np.nanquantile(d_same_ap, 0.025, 0),
                       'Δ AUPRC IC alto': np.nanquantile(d_same_ap, 0.975, 0)}, index=names)
scelta = scelta.sort_values('AUPRC macro (CV, scelta)', ascending=False)

# regola di scelta: tra i modelli di sola sequenza, quelli entro 2 deviazioni standard (tra semi) dal migliore in CV sono
# equivalenti; tra gli equivalenti si prende il PIÙ SEMPLICE. Restano fuori i modelli che usano altro oltre la sequenza
# (scheda, partner) e DeepLoc 2.0. L'ordine di semplicità è dichiarato qui, prima di guardare i risultati
SEMPLICITA = ['Baseline (prevalenza)', 'Regressione logistica', SOTA, SAME, 'Gradient boosting (alberi)',
              STAGE_NAMES['abl_solo_medie'], STAGE_NAMES['abl_senza_sorting'], STAGE_NAMES['abl_solo_esm2'], STAGE_NAMES['abl_senza_struct'],
              'Rete neurale',
              STAGE_NAMES['abl_con_attenzione'], 'Rete neurale + k-NN', ADVN, ENS_NAME]
# le prove sui dati ("Dati: …") sono esperimenti con meno dati apposta, e i metodi rifatti sono i concorrenti: non sono candidati
sola_seq = (~scelta.index.str.contains('non solo sequenza|DeepLoc 2.0 ufficiale|rifatto|protocollo DeepLoc') & ~scelta.index.str.startswith('Dati:'))
MIGLIORE_CV = scelta.index[sola_seq][0]
ds_semi = float(np.std(np.r_[RANK['cv']['score'].max(), RANK['semi']['score'].to_numpy()], ddof=1))
pari = scelta.index[(scelta['AUPRC macro (CV, scelta)'] >= scelta.loc[MIGLIORE_CV, 'AUPRC macro (CV, scelta)'] - 2 * ds_semi) & sola_seq]
BEST = pd.Series(np.arange(len(SEMPLICITA)), index=SEMPLICITA).reindex(pari).fillna(len(SEMPLICITA)).idxmin()
print(scelta.round(4).to_string())

# le varianti seme per seme: media e deviazione standard dell'AUPRC in CV di una rete sola (non della media dei semi)
semi_tab = pd.DataFrame({nome: [RANK[stg]['score'].mean(), RANK[stg]['score'].std(ddof=1), len(RANK[stg])] for stg, nome in STAGE_NAMES.items()},
                        index=['AUPRC CV di una rete (media dei semi)', 'DS tra semi', 'semi']).T
semi_tab.loc['Rete vincente (stessi 3 semi)'] = [np.r_[RANK['cv']['score'].max(), RANK['semi']['score'].to_numpy()].mean(), ds_semi, 1 + len(RANK['semi'])]
print('\nVarianti e riferimenti, una rete per seme (qui si legge se una differenza supera il rumore):')
print(semi_tab.sort_values('AUPRC CV di una rete (media dei semi)', ascending=False).round(4).to_string())

print(f'\nMigliore in CV tra i modelli di sola sequenza: {MIGLIORE_CV}')
print(f'Rumore tra semi della CV: DS {ds_semi:.4f}. Equivalenti (entro 2 DS): {list(pari)}')
print(f'Metodologia scelta = il più semplice tra gli equivalenti: {BEST}')
print("Attenzione: l'AUPRC in CV del modello scelto è ottimistica (configurazione, α, ensemble e modello scelti sulle stesse predizioni "
      "fuori fold): il numero da citare è quello sul test.")
print(f"Sul test: AUC {scelta.loc[BEST, 'AUC macro (test)']:.3f} (IC 95% {scelta.loc[BEST, 'IC 95% basso']:.3f}–{scelta.loc[BEST, 'IC 95% alto']:.3f}), "
      f"AUPRC {scelta.loc[BEST, 'AUPRC macro (test)']:.3f} (IC 95% {scelta.loc[BEST, 'AUPRC IC basso']:.3f}–{scelta.loc[BEST, 'AUPRC IC alto']:.3f})")
# il confronto con lo stato dell'arte esiste solo con le predizioni ufficiali di DeepLoc 2.0 sulle stesse proteine
for nome_u, _ in OFFICIAL:
    d_u, d_u_ap = boot[:, names.index(BEST)] - boot[:, names.index(nome_u)], boot_ap[:, names.index(BEST)] - boot_ap[:, names.index(nome_u)]
    print(f"Contro {nome_u} (stesse proteine, appaiato): AUC {np.nanmean(d_u):+.3f} (IC 95% {np.nanquantile(d_u, 0.025):+.3f} … {np.nanquantile(d_u, 0.975):+.3f}), "
          f"AUPRC {np.nanmean(d_u_ap):+.3f} (IC 95% {np.nanquantile(d_u_ap, 0.025):+.3f} … {np.nanquantile(d_u_ap, 0.975):+.3f})")
print(['Confronto con DeepLoc 2.0: NON disponibile (mancano le predizioni ufficiali, sezione 9).', ''][HAS_DL2])
# ogni contributo, misurato come differenza appaiata sul test (AUC e AUPRC) tra due modelli che differiscono per una cosa
for etichetta, m_a, m_b in [('Modello scelto contro il riferimento semplice sugli stessi dati (contributo del modello)', BEST, SAME),
                            ('Rete contro il riferimento semplice sugli stessi dati', 'Rete neurale', SAME),
                            ('Dati in più, sul riferimento semplice', SAME, SOTA), ('Dati in più, sulla rete', 'Rete neurale', STAGE_NAMES['abl_solo_deeploc']),
                            ('Fonti HPA nel training (stessa fonte delle etichette del test)', 'Rete neurale', STAGE_NAMES['abl_senza_hpa']),
                            ('Attenzione sui residui (NeuroLoc-LA contro la rete)', ADVN, 'Rete neurale'),
                            ('Scheda testuale (conoscenza dei curatori, non solo sequenza)', SCHEDA, 'Rete neurale')]:
    d_ab, d_ab_ap = boot[:, names.index(m_a)] - boot[:, names.index(m_b)], boot_ap[:, names.index(m_a)] - boot_ap[:, names.index(m_b)]
    print(f"{etichetta}: AUC {np.nanmean(d_ab):+.3f} (IC 95% {np.nanquantile(d_ab, 0.025):+.3f} … {np.nanquantile(d_ab, 0.975):+.3f}) | "
          f"AUPRC {np.nanmean(d_ab_ap):+.3f} (IC 95% {np.nanquantile(d_ab_ap, 0.025):+.3f} … {np.nanquantile(d_ab_ap, 0.975):+.3f})")
# STATO DELL'ARTE A PARITÀ DI CONDIZIONI (sezione 8d): stesse etichette (le curate di DeepLoc, proteine umane), stesse
# partizioni, stessa divisione per la validazione, stesso protocollo di addestramento (quello di DeepLoc 2.0), nessun
# iperparametro scelto sui nostri dati, 3 semi × 5 fold per tutti. Per queste quattro righe anche la CV sulle curate è
# pulita (nessuna scelta fatta guardandola). RISULTATO PRINCIPALE: il test HPA (etichette di un'altra fonte, mai viste)
DL2R, WEFR = 'DeepLoc 2.0 rifatto (stesse etichette, protocollo DeepLoc 2.0)', 'Wefers ProtT5-MHA (stesse etichette, protocollo DeepLoc 2.0)'
NLPT, NLRC = 'NeuroLoc-Δ, sola ProtT5 (stesse etichette e stessi ingressi, protocollo DeepLoc 2.0)', 'NeuroLoc-Δ, rappresentazione completa (stesse etichette, protocollo DeepLoc 2.0)'
NOSTRO_ORO = STAGE_NAMES['abl_solo_deeploc']
STATO_ARTE = [DL2R, WEFR, NLPT, NLRC, BEST]
sa_tab = scelta.loc[STATO_ARTE, ['AUC macro (test)', 'IC 95% basso', 'IC 95% alto', 'AUPRC macro (test)', 'AUPRC IC basso', 'AUPRC IC alto',
                                 'AUPRC macro (CV, scelta)', 'AUPRC macro (CV, 6 classi del test)']].copy()
sa_tab['ingressi'] = ['ProtT5 per residuo (4.000)', 'ProtT5 per residuo (4.000)', 'ProtT5 per residuo (4.000): 7 riassunti',
                      'ESM-2 + ProtT5 (riassunti), descrittori, motivi, struttura', 'come sopra, più le fonti di etichette in più']
sa_tab['iperparametri scelti sui nostri dati'] = ['no', 'no', 'no', 'no', 'sì: griglia e CV sulle curate (CV ottimistica)']
print("\nSTATO DELL'ARTE A PARITÀ DI CONDIZIONI — risultato principale: le colonne del test HPA")
print(sa_tab.round(3).to_string())
# differenze appaiate (stessi ricampionamenti del test), p a due code dal bootstrap e correzione di Holm dentro ogni
# famiglia: i 6 confronti a parità di condizioni × AUC e AUPRC; i 2 confronti con il sistema completo (contesto) a parte
CONFR_SA = [('a parità', 'NeuroLoc-Δ sola ProtT5 − DeepLoc 2.0 (stessi ingressi)', NLPT, DL2R),
            ('a parità', 'NeuroLoc-Δ sola ProtT5 − Wefers (stessi ingressi)', NLPT, WEFR),
            ('a parità', 'NeuroLoc-Δ rappresentazione completa − DeepLoc 2.0', NLRC, DL2R),
            ('a parità', 'NeuroLoc-Δ rappresentazione completa − Wefers', NLRC, WEFR),
            ('a parità', 'Wefers − DeepLoc 2.0', WEFR, DL2R),
            ('a parità', 'NeuroLoc-Δ: rappresentazione completa − sola ProtT5 (valore degli ingressi in più)', NLRC, NLPT),
            ('contesto', 'NeuroLoc-Δ completo (dati in più, scelto in CV) − DeepLoc 2.0', BEST, DL2R),
            ('contesto', 'NeuroLoc-Δ completo (dati in più, scelto in CV) − Wefers', BEST, WEFR)]
sa_diff = pd.DataFrame([{'famiglia': fam_, 'confronto': et_, 'metrica': nome_b, 'Δ': np.mean(d_ok),
                         'IC 2,5%': np.quantile(d_ok, 0.025), 'IC 97,5%': np.quantile(d_ok, 0.975),
                         'p': float(np.clip(2 * min(np.mean(d_ok <= 0), np.mean(d_ok >= 0)), 1 / B, 1))}
                        for fam_, et_, a_, b_ in CONFR_SA for nome_b, B_ in [('AUC', boot), ('AUPRC', boot_ap)]
                        for d_ in [B_[:, names.index(a_)] - B_[:, names.index(b_)]] for d_ok in [d_[np.isfinite(d_)]]])
# Holm: p ordinati, p_(i) × (m − i + 1), poi il massimo progressivo (la correzione non può far scendere un p successivo)
sa_diff['p Holm'] = np.nan
for fam_ in sa_diff['famiglia'].unique():
    p_f = sa_diff.loc[sa_diff['famiglia'] == fam_, 'p'].to_numpy()
    o_f = np.argsort(p_f)
    adj_f = np.empty(len(p_f))
    adj_f[o_f] = np.minimum(np.maximum.accumulate((len(p_f) - np.arange(len(p_f))) * p_f[o_f]), 1)
    sa_diff.loc[sa_diff['famiglia'] == fam_, 'p Holm'] = adj_f
sa_diff['dimostrata (Holm < 0,05)'] = np.where(sa_diff['p Holm'] < 0.05, 'sì', 'no')
print('\nDifferenze appaiate sul test HPA (p corretti con Holm dentro ogni famiglia):')
print(sa_diff.round(4).to_string(index=False))
print("Lettura: «a parità» = stesse etichette, divisione e protocollo, nessun iperparametro scelto sui nostri dati. Il primo "
      "confronto (sola ProtT5) ha anche gli stessi ingressi: misura solo il modo di aggregare i residui e il classificatore. "
      "L'ultimo «a parità» misura il valore degli ingressi in più di NeuroLoc-Δ. Il «contesto» usa dati in più ed è scelto in CV: "
      "non è un confronto alla pari e va scritto così. Con circa 6.400 proteine umane i modelli ad attenzione (pensati per "
      "20–28.000 proteine) possono essere svantaggiati: la conclusione vale a questa scala di dati.")
print("Un contributo il cui intervallo contiene lo zero NON è dimostrato. Il guadagno delle fonti HPA è in parte allineamento alla "
      "fonte del test: va dichiarato accanto al risultato.")


# metriche per strato: omologia con il training curato, e quanto della proteina canonica è nella sequenza del test (molte
# sequenze del test sono frammenti: l'etichetta HPA è del gene, e un frammento può non contenere il segnale di smistamento).
# AUC con la formula dei ranghi (NaN, senza errori, se in uno strato una classe non ha positivi o negativi)
t_ident = proteins.loc[test_rows, 'test_identica'].fillna(0).to_numpy() == 1
t_meta = proteins.loc[test_rows, 'test_copertura'].fillna(0).to_numpy() >= 0.5
strati = pd.DataFrame(index=list(dict.fromkeys([BEST, 'Rete neurale', DL2R, WEFR, NLPT, NLRC, SOTA, SAME, SCHEDA] + [n for n, _ in OFFICIAL])))
for nome_s, m_s in [('tutto il test', np.ones(len(test_rows), bool)), ('con omologa nel training', test_hom), ('senza omologa nel training', ~test_hom),
                    ('sequenza identica alla canonica', t_ident), ('almeno metà della canonica', t_meta), ('frammento (meno di metà)', ~t_meta)]:
    with np.errstate(all='ignore'):
        strati[f'AUC macro, {nome_s} (n={int(m_s.sum())})'] = [
            np.nanmean([(rankdata(P_TEST_[m_s, c])[y_true[m_s, j] == 1].sum() - (y_true[m_s, j] == 1).sum() * ((y_true[m_s, j] == 1).sum() + 1) / 2)
                        / ((y_true[m_s, j] == 1).sum() * (y_true[m_s, j] == 0).sum()) for j, c in enumerate(ev)])
            for P_TEST_ in [dict(ALL_TEST)[n_] for n_ in strati.index]]
print('\nTest per strato:')
print(strati.round(3).T.to_string())
with pd.ExcelWriter(DATA / 'scelta_metodologia.xlsx') as xw:
    scelta.round(4).to_excel(xw, sheet_name='modelli')
    strati.round(4).to_excel(xw, sheet_name='strati del test')
    semi_tab.round(4).to_excel(xw, sheet_name='varianti per seme')
    sa_tab.round(4).to_excel(xw, sheet_name="stato dell'arte")
    sa_diff.round(4).to_excel(xw, sheet_name="stato dell'arte, differenze", index=False)


## 11 · Curve ROC sul test HPA
**Figura 1 — modelli.** Un pannello per compartimento, la media macro (media dei TPR su una griglia comune di FPR) e la
precision-recall macro, più severa con le classi rare. Il riferimento semplice è la curva rossa; DeepLoc 2.0 ufficiale,
se ci sono le predizioni del server, la nera tratteggiata. La baseline è la diagonale.

**Figura 2 — metodologie.** A sinistra le ROC macro della rete vincente (un seme), delle sue varianti e delle prove sui
dati; a destra l'AUC macro di tutti i modelli con l'IC 95%; la linea verticale è il riferimento semplice, la stella la
scelta sulla CV.

In [ ]:
plt.style.use('default')                      # figure su sfondo bianco, pronte per il paper
MAIN = ['Baseline (prevalenza)', 'Regressione logistica', SOTA, 'Rete neurale', DL2R, WEFR, NLPT, NLRC, ENS_NAME, CTX_MODEL,
        SCHEDA] + [n for n, _ in OFFICIAL]
STYLE = {'Baseline (prevalenza)': ('#8a8a86', ':'), 'Regressione logistica': ('#2a78d6', '--'), SOTA: ('#d62a2a', '-'), SAME: ('#d62a2a', '--'),
         'Rete neurale': ('#eb6834', '-'), 'Gradient boosting (alberi)': ('#c79a1b', '-.'), ENS_NAME: ('#1baf7a', '-'),
         CTX_MODEL: ('#0e7c86', ':'), ADVN: ('#b8326b', '-'), SCHEDA: ('#7b4bd6', '-'),
         'DeepLoc 2.0 ufficiale': ('#111111', '--'), DL2R: ('#111111', '-'), WEFR: ('#555555', '-.'), NLPT: ('#eb6834', '--'), NLRC: ('#eb6834', ':')}
P_TEST = dict(ALL_TEST)
fpr_grid, rec_grid = np.linspace(0, 1, 201), np.linspace(0, 1, 201)
fig, axes = plt.subplots(2, 4, figsize=(19, 9.5), sharex=True, sharey=True)
for name in MAIN:
    Ph, (col, ls) = P_TEST[name], STYLE[name]
    tpr_m, prec_m = [], []
    for j, c in enumerate(ev):
        fpr, tpr, _ = roc_curve(y_true[:, j], Ph[:, c])
        tpr_m.append(np.interp(fpr_grid, fpr, tpr))
        axes.flat[j].plot(fpr, tpr, color=col, ls=ls, lw=1.8, label=f"{name.split(' (')[0]} ({roc_auc_score(y_true[:, j], Ph[:, c]):.3f})")
        prec, rec, _ = precision_recall_curve(y_true[:, j], Ph[:, c])
        prec_m.append(np.interp(rec_grid, rec[:-1][::-1], np.maximum.accumulate(prec[:-1])[::-1]))
    axes.flat[6].plot(fpr_grid, np.mean(tpr_m, 0), color=col, ls=ls, lw=2, label=f"{name.split(' (')[0]} ({scelta.loc[name, 'AUC macro (test)']:.3f})")
    ap_macro = np.mean([average_precision_score(y_true[:, j], Ph[:, c]) for j, c in enumerate(ev)])
    axes.flat[7].plot(rec_grid, np.mean(prec_m, 0), color=col, ls=ls, lw=2, label=f"{name.split(' (')[0]} (AUPRC {ap_macro:.3f})")
chance = [[0, 1]] * 7 + [[np.nan] * 2]
legend_loc = ['lower right'] * 7 + ['upper right']
for j, title in enumerate(HPA_EVAL + ['Media macro (ROC)', 'Media macro (precisione vs richiamo)']):
    ax = axes.flat[j]
    ax.plot([0, 1], chance[j], color='#c3c2b7', lw=1)
    ax.set_title(title, fontsize=12)
    ax.grid(color='#e6e5e0', lw=0.6)
    ax.legend(fontsize=7, loc=legend_loc[j], frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
for ax, lab in zip(axes[1], ['Falsi positivi (FPR)'] * 3 + ['Richiamo']):
    ax.set_xlabel(lab)
for ax in axes[:, 0]:
    ax.set_ylabel('Veri positivi (TPR)')
fig.suptitle(f"Test HPA indipendente ({len(test_rows):,} proteine umane del cervello): modelli, DeepLoc 2.0 e Wefers a parità di condizioni", fontsize=14)
fig.tight_layout()
fig.savefig(DATA / 'roc_confronto.png', dpi=150)
plt.show()

ABL = ['Rete neurale'] + list(STAGE_NAMES.values())[2:] + [SOTA]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 7), gridspec_kw={'width_ratios': [1, 1.25]})
for k_, name in enumerate(ABL):
    tpr_m = [np.interp(fpr_grid, *roc_curve(y_true[:, j], P_TEST[name][:, c])[:2]) for j, c in enumerate(ev)]
    ax1.plot(fpr_grid, np.mean(tpr_m, 0), lw=2, ls=['-', '--', '-.', ':', (0, (5, 1)), '-'][k_ % 6],
             color=['#eb6834', '#2a78d6', '#1baf7a', '#7b4bd6', '#c79a1b', '#d62a2a'][k_ % 6],
             label=f"{name.split(' (')[0]} ({scelta.loc[name, 'AUC macro (test)']:.3f})")
ax1.plot([0, 1], [0, 1], color='#c3c2b7', lw=1)
ax1.set(xlabel='Falsi positivi (FPR)', ylabel='Veri positivi (TPR)', title='ROC macro: rete vincente, varianti e prove sui dati (3 semi ciascuna)')
ax1.legend(fontsize=8, loc='lower right', frameon=False)
order = scelta.sort_values('AUC macro (test)').index
ax2.errorbar(scelta.loc[order, 'AUC macro (test)'], np.arange(len(order)),
             xerr=[scelta.loc[order, 'AUC macro (test)'] - scelta.loc[order, 'IC 95% basso'],
                   scelta.loc[order, 'IC 95% alto'] - scelta.loc[order, 'AUC macro (test)']],
             fmt='o', color='#2a2a2a', ecolor='#9a9a94', capsize=3)
ax2.axvline(scelta.loc[DL2R, 'AUC macro (test)'], color='#111111', ls='--', lw=1.2, label='DeepLoc 2.0 rifatto, a parità di condizioni')
ax2.plot(scelta.loc[BEST, 'AUC macro (test)'], list(order).index(BEST), marker='*', ms=18, color='#eb6834', ls='none', label='metodologia scelta')
ax2.set_yticks(np.arange(len(order)), [n.split(' (')[0] for n in order], fontsize=9)
ax2.set(xlabel='AUC macro sul test (IC 95%, bootstrap)', title='Tutti i modelli e le metodologie')
ax2.legend(fontsize=8, loc='lower right', frameon=False)
for ax in (ax1, ax2):
    ax.grid(color='#e6e5e0', lw=0.6)
    ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(DATA / 'roc_metodologie.png', dpi=150)
plt.show()

In [ ]:
r_b = int(RANK['cervello']['run'].iloc[0])
np.savez_compressed(DATA / 'esiti_02.npz', oof=RUNS[r_b]['oof'], hpa=RUNS[r_b]['hpa'], var=RUNS[r_b]['var'],
                    chan=RUNS[r_b]['chan'], folds=np.array(RUNS[r_b]['folds']), OBS=OBS.astype(np.float16), FRAC=FRAC, part=part,
                    gold_rows=gold_rows, human_gold=human_gold, test_rows=test_rows, var_rows=var_rows, train_ok=train_ok,
                    Y_test=Y_test, STRUCT=STRUCT_E, P_oof_best=MODELS[BEST][0], P_hpa_best=MODELS[BEST][1],
                    # il modello scelto è già di sola sequenza: P_*_seq resta per lo 03 (contesto di Wefers et al.)
                    P_oof_seq=MODELS[BEST][0], P_hpa_seq=MODELS[BEST][1],
                    P_oof_scheda=MODELS[STAGE_NAMES['con_scheda']][0], P_hpa_scheda=MODELS[STAGE_NAMES['con_scheda']][1],
                    valutabile=valutabile, test_hom=test_hom, is_neur=is_neur, is_oligo=is_oligo,
                    # stessi dati (etichette curate di DeepLoc): la nostra rete e i due metodi rifatti, per il confronto HOU dello 03
                    P_oof_oro=MODELS[NOSTRO_ORO][0], P_hpa_oro=MODELS[NOSTRO_ORO][1], P_oof_dl2r=P_oof_dl2r, P_hpa_dl2r=P_hpa_dl2r,
                    P_oof_wefr=P_oof_wefr, P_hpa_wefr=P_hpa_wefr,
                    # NeuroLoc-Δ con il protocollo di DeepLoc 2.0 (sezione 8d): sola ProtT5 e rappresentazione completa
                    P_oof_nlpt=P_oof_nlpt, P_hpa_nlpt=P_hpa_nlpt, P_oof_nlrc=P_oof_nlrc, P_hpa_nlrc=P_hpa_nlrc,
                    # il nostro modello con la migliore AUPRC in CV (sola sequenza) e le sue soglie MCC: confronto finale (03, sezione 13)
                    P_oof_top=MODELS[MIGLIORE_CV][0], P_hpa_top=MODELS[MIGLIORE_CV][1], thr_top=THR[MIGLIORE_CV]['mcc'])
(DATA / 'esiti_02.json').write_text(json.dumps({
    'CLASSES': CLASSES, 'NEURO': NEURO, 'ALL': ALL, 'SOURCES': SOURCES, 'AUX': AUX, 'SUPER': list(SUPER), 'HPA_EVAL': HPA_EVAL,
    'BEST': BEST, 'TOP': MIGLIORE_CV, 'ENS_NAME': ENS_NAME, 'ESM_TAG': tag, 'RES_DIR': str(RES), 'LOC_FILE': str(paths['LOC']), 'AF_DIR': str(AFDIR),
    'MAX_RES': MAX_RES, 'WIN': WIN, 'VUOTE': VUOTE, 'n_test': int(len(test_rows)), 'ESM': ESM, 'T5': T5, 'USE_T5': int(USE_T5),
    'TEXT_MODEL': TEXT_MODEL, 'SUMMARIES': SUMMARIES, 'D': int(D), 'D5': int(D5), 'HAND_COLS': list(hand.columns), 'STRUCT': STRUCT,
    'SCRUB2': SCRUB2, 'FOLDS': FOLDS}, ensure_ascii=False, indent=1))
np.savez_compressed(DATA / 'pwm_motivi.npz', **{n_: v_ for n_, v_ in PWM.items()})
np.save(DATA / 'unit.npy', unit.astype(np.float16))
cv.drop(columns=['mut_sequence', 'priority']).to_parquet(DATA / 'varianti_codificate.parquet', index=False)
hand.to_parquet(DATA / 'descrittori_sequenza.parquet', index=False)
print('Salvati per il notebook 03: esiti_02.npz/.json, varianti_codificate.parquet, descrittori_sequenza.parquet, '
      'modello_cervello/fold*.pt, pwm_motivi.npz, unit.npy')

## 11c · La curva finale: modello scelto contro baseline, logistica e riferimenti
La figura che riassume il notebook: stesse proteine del test. Baseline, logistica, riferimento semplice (e DeepLoc 2.0
ufficiale, se disponibile), rete a token, **rete avanzata NeuroLoc-LA** (attenzione sui residui), la metodologia scelta sulla CV (stella
nella figura 2) e la stessa con il contesto dei partner.

In [ ]:
# la rete finale (BEST) è in arancione; se la CV avesse scelto la logistica, la curva è una sola e prende quel colore
STYLE_F = {'Baseline (prevalenza)': ('#8a8a86', ':'), 'Regressione logistica': ('#2a78d6', '--'), SOTA: ('#d62a2a', '-'), SAME: ('#d62a2a', '--'),
           'DeepLoc 2.0 ufficiale': ('#111111', '--'), CTX_MODEL: ('#0e7c86', '-.'), 'Rete neurale': ('#c79a1b', '-.'), ADVN: ('#b8326b', '-'),
           ENS_NAME: ('#1baf7a', '-'), DL2R: ('#111111', '-'), WEFR: ('#555555', '-.'), NLPT: ('#c79a1b', '--'), NLRC: ('#c79a1b', ':')}
STYLE_F[BEST] = ('#eb6834', '-')
FINAL = list(dict.fromkeys(['Baseline (prevalenza)', 'Regressione logistica', SOTA] + [n for n, _ in OFFICIAL] + [DL2R, WEFR, NLPT, NLRC, BEST, CTX_MODEL]))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6.2))
for name in FINAL:
    Ph, (col, ls) = P_TEST[name], STYLE_F[name]
    tpr_m = np.mean([np.interp(fpr_grid, *roc_curve(y_true[:, j], Ph[:, c])[:2]) for j, c in enumerate(ev)], 0)
    prec_m = np.mean([np.interp(rec_grid, pr_[1][:-1][::-1], np.maximum.accumulate(pr_[0][:-1])[::-1])
                      for j, c in enumerate(ev) for pr_ in [precision_recall_curve(y_true[:, j], Ph[:, c])]], 0)
    lab = f"{name.split(' (')[0]}: AUC {scelta.loc[name, 'AUC macro (test)']:.3f} [{scelta.loc[name, 'IC 95% basso']:.3f}–{scelta.loc[name, 'IC 95% alto']:.3f}]"
    ax1.plot(fpr_grid, tpr_m, color=col, ls=ls, lw=2.4, label=lab)
    ax2.plot(rec_grid, prec_m, color=col, ls=ls, lw=2.4,
             label=f"{name.split(' (')[0]}: AUPRC {np.mean([average_precision_score(y_true[:, j], Ph[:, c]) for j, c in enumerate(ev)]):.3f}")
ax1.plot([0, 1], [0, 1], color='#c3c2b7', lw=1)
ax1.set(xlabel='Falsi positivi (FPR)', ylabel='Veri positivi (TPR)', title='ROC macro sul test HPA')
ax2.set(xlabel='Richiamo', ylabel='Precisione', title='Precisione vs richiamo, macro')
for ax in (ax1, ax2):
    ax.legend(fontsize=8.5, loc=['lower right', 'lower left'][int(ax is ax2)], frameon=False)
    ax.grid(color='#e6e5e0', lw=0.6)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle(f'NeuroLoc-Δ contro DeepLoc 2.0 e Wefers et al. a parità di condizioni ({len(test_rows):,} proteine del test)', fontsize=13)
fig.tight_layout()
fig.savefig(DATA / 'roc_finale.png', dpi=150)
plt.show()

## 11d · Confronto finale con i valori pubblicati: DeepLoc 2.0 e gli altri metodi sul test HPA
I numeri dei metodi pubblicati sono **quelli stampati nell'articolo di DeepLoc 2.0** (Thumuluri et al., *Nucleic Acids
Res* 2022;50:W228–W234, Tabella 2): test indipendente HPA di 1.717 voci, senza sequenze oltre il 30% di identità con il
training di DeepLoc. Ci sono YLoc+, DeepLoc 1.0, Fuel-mLoc, LAProtT5 e DeepLoc 2.0 (ESM1b e ProtT5). Le metriche sono
le loro: accuratezza (tutte le etichette giuste), Jaccard, F1 micro e macro, MCC per classe.

Il nostro modello è quello con la **migliore AUPRC in CV tra i modelli di sola sequenza**: lo stesso tipo di ingresso
dei metodi pubblicati, scelto in CV e non sul test. Valgono le stesse regole di DeepLoc 2.0:
- soglia per classe che massimizza l'MCC sul training;
- se nessuna classe supera la soglia, la classe più vicina alla soglia.

Il benchmark conta più isoforme dello stesso gene, con la stessa etichetta. Qui ogni gene **pesa quante voci aveva**,
così le metriche sono pesate come quelle pubblicate.

**Curve ROC.** L'articolo non stampa le AUC sul test HPA (sono nella Tabella S7 del supplemento). Per mettere i metodi
pubblicati nel piano ROC si usa il loro MCC per classe: con i positivi e i negativi del benchmark, un MCC corrisponde a
una **curva di punti di lavoro** (stesso MCC). Dove la nostra curva ROC passa sopra quella curva, esiste una soglia con
MCC più alto del loro. Se aggiungi le AUC della Tabella S7 in `data/esterni/deeploc2_auc_hpa.csv` (colonne `metodo`,
`classe`, `auc`), entrano nella tabella e nelle legende.

Differenze che restano, da dichiarare:
- le voci fuori dal perimetro cerebrale o non collegate a UniProt non ci sono (la copertura è stampata);
- i metodi pubblicati sono addestrati sui loro dati.


In [ ]:
# 11d · confronto finale con i valori pubblicati (DeepLoc 2.0, Tabella 2, doi:10.1093/nar/gkac278) sul test HPA
TOP = MIGLIORE_CV
P_top, thr_top = MODELS[TOP][1], THR[TOP]['mcc']
NOME_TOP = f"NeuroLoc-Δ ({TOP.split(' (')[0]})"
w_iso = proteins.loc[test_rows].reindex(columns=['n_isoforme_test'])['n_isoforme_test'].fillna(1).to_numpy(float)
METR_PUB = ['Etichette predette per proteina', 'Accuracy', 'Jaccard', 'Micro F1', 'Macro F1'] + [f'MCC {c}' for c in HPA_EVAL]
PUB_HPA = pd.DataFrame({'YLoc+ (Animal)': [1.44, 0.23, 0.41, 0.51, 0.34, 0.14, 0.20, 0.20, 0.37, 0.12, 0.08],
                        'DeepLoc 1.0': [0.89, 0.37, 0.42, 0.46, 0.35, 0.23, 0.28, 0.23, 0.39, 0.23, 0.10],
                        'Fuel-mLoc': [1.00, 0.38, 0.46, 0.52, 0.39, 0.23, 0.41, 0.32, 0.33, 0.14, 0.24],
                        'LAProtT5': [0.94, 0.45, 0.52, 0.56, 0.43, 0.33, 0.45, 0.30, 0.59, 0.22, 0.26],
                        'DeepLoc 2.0 (ESM1b)': [1.15, 0.34, 0.48, 0.57, 0.44, 0.29, 0.41, 0.34, 0.60, 0.20, 0.17],
                        'DeepLoc 2.0 (ProtT5)': [1.21, 0.39, 0.53, 0.60, 0.46, 0.36, 0.44, 0.36, 0.56, 0.17, 0.31]}, index=METR_PUB)
# positivi per classe nel benchmark pubblicato (colonna "Count" della Tabella 2; 1,22 etichette vere per voce)
CONTE_PUB, N_PUB = pd.Series([562, 893, 287, 196, 77, 86], index=HPA_EVAL), 1717

# le predizioni con le regole di DeepLoc 2.0: soglie MCC dal training; nessuna classe sopra soglia → la più vicina
pred_top = P_top >= thr_top
nessuna = ~pred_top.any(1)
pred_top[np.flatnonzero(nessuna), np.argmax(P_top - thr_top, 1)[nessuna]] = True
Yt = Y_test.astype(int)
con_pos = np.flatnonzero(Yt.sum(0) > 0)              # F1 macro sulle classi che il test contiene
nostro_pub = pd.Series([np.average(pred_top.sum(1), weights=w_iso), np.average((pred_top == Yt).all(1), weights=w_iso),
                        jaccard_score(Yt, pred_top, average='samples', sample_weight=w_iso, zero_division=0),
                        f1_score(Yt, pred_top, average='micro', sample_weight=w_iso, zero_division=0),
                        f1_score(Yt, pred_top, average='macro', labels=con_pos, sample_weight=w_iso, zero_division=0)]
                       + [matthews_corrcoef(Yt[:, CLASSES.index(c)], pred_top[:, CLASSES.index(c)], sample_weight=w_iso) for c in HPA_EVAL],
                       index=METR_PUB)
confronto_pub = PUB_HPA.assign(**{NOME_TOP: nostro_pub})
print(f"Copertura del benchmark pubblicato: {int(w_iso.sum()):,} voci su {N_PUB:,} ({w_iso.sum() / N_PUB:.0%}), {len(test_rows):,} geni | "
      f"etichette vere per voce: {np.average(Yt.sum(1), weights=w_iso):.2f} (pubblicato: 1,22)")
print(pd.DataFrame({'positivi pubblicati (1.717 voci)': CONTE_PUB,
                    'positivi qui (pesati per voce)': [float((Yt[:, CLASSES.index(c)] * w_iso).sum()) for c in HPA_EVAL]}).round(0).to_string())
print(f"\nTEST HPA: valori pubblicati (Tabella 2 di DeepLoc 2.0) e {NOME_TOP}, stesse metriche e stesse regole di soglia:")
print(confronto_pub.round(3).to_string())

# AUC per classe del nostro modello, pesate per voce, con IC bootstrap per gene (1.000 ricampionamenti); le AUC pubblicate
# della Tabella S7, se le aggiungi in data/esterni/deeploc2_auc_hpa.csv
auc_top = pd.Series([roc_auc_score(Yt[:, CLASSES.index(c)], P_top[:, CLASSES.index(c)], sample_weight=w_iso) for c in HPA_EVAL], index=HPA_EVAL)
B_P = 1000
W_p = rng.multinomial(len(test_rows), np.full(len(test_rows), 1 / len(test_rows)), size=B_P) * w_iso[None]
auc_bp = np.array([[roc_auc_score(Yt[W_p[b] > 0, CLASSES.index(c)], P_top[W_p[b] > 0, CLASSES.index(c)], sample_weight=W_p[b][W_p[b] > 0])
                    for c in HPA_EVAL] for b in range(B_P)])
auc_tab = pd.DataFrame({f'AUC {NOME_TOP}': auc_top, 'IC 2,5%': np.quantile(auc_bp, 0.025, 0), 'IC 97,5%': np.quantile(auc_bp, 0.975, 0)})
auc_pub = pd.concat([pd.read_csv(f, encoding='utf-8') for f in sorted((DATA / 'esterni').glob('deeploc2_auc_hpa*.csv'))] +
                    [pd.DataFrame(columns=['metodo', 'classe', 'auc'])], ignore_index=True)
auc_tab = auc_tab.join(auc_pub.pivot_table(index='classe', columns='metodo', values='auc').add_prefix('AUC pubblicata · '))
auc_tab.loc['macro'] = auc_tab.mean()
print(f"\nAUC per classe sul test HPA ({NOME_TOP}, pesate per voce):")
print(auc_tab.round(3).to_string())
print(['>>> AUC pubblicate assenti: l\'articolo le dà solo nella Tabella S7 del supplemento. Per averle qui mettile in '
       'data/esterni/deeploc2_auc_hpa.csv (colonne metodo, classe, auc) e rilancia questa cella.', ''][int(len(auc_pub) > 0)])

# la figura finale: per ogni classe la nostra curva ROC, il nostro punto di lavoro e, per i metodi pubblicati, la curva dei
# punti con il loro stesso MCC alle proporzioni del benchmark (sopra la curva = MCC più alto del loro)
t_g, f_g = np.linspace(0, 1, 2001), np.linspace(0, 1, 401)
ISO = [('DeepLoc 2.0 (ProtT5)', '#111111', '-'), ('DeepLoc 2.0 (ESM1b)', '#111111', '--'), ('LAProtT5', '#8a8a86', '-.')]
fig, axs = plt.subplots(2, 4, figsize=(20, 9.5), facecolor='white')
mcc_bench = {}
for j, (c, ax) in enumerate(zip(HPA_EVAL, axs.ravel())):
    ci, P_c = CLASSES.index(c), CONTE_PUB[c]
    N_c = N_PUB - P_c
    f_, t_, _ = roc_curve(Yt[:, ci], P_top[:, ci], sample_weight=w_iso)
    ax.plot(f_, t_, color='#eb6834', lw=2.4, label=f'{NOME_TOP}: AUC {auc_top[c]:.3f} [{auc_tab.loc[c, "IC 2,5%"]:.3f}–{auc_tab.loc[c, "IC 97,5%"]:.3f}]')
    tpr_o = (w_iso * pred_top[:, ci] * Yt[:, ci]).sum() / (w_iso * Yt[:, ci]).sum()
    fpr_o = (w_iso * pred_top[:, ci] * (1 - Yt[:, ci])).sum() / (w_iso * (1 - Yt[:, ci])).sum()
    TP_, FP_ = tpr_o * P_c, fpr_o * N_c
    mcc_bench[c] = (TP_ * (N_c - FP_) - FP_ * (P_c - TP_)) / np.sqrt((TP_ + FP_) * P_c * N_c * (N_c - FP_ + P_c - TP_))
    ax.plot(fpr_o, tpr_o, 'o', ms=9, color='#eb6834', mec='black',
            label=f'nostro punto di lavoro (soglia MCC): MCC {nostro_pub[f"MCC {c}"]:.2f} | {mcc_bench[c]:.2f} alle proporzioni del benchmark')
    TPg, FPg = t_g[None, :] * P_c, f_g[:, None] * N_c
    with np.errstate(all='ignore'):
        MCCg = (TPg * (N_c - FPg) - FPg * (P_c - TPg)) / np.sqrt((TPg + FPg) * P_c * N_c * (N_c - FPg + P_c - TPg))
    for nome_p, col_p, ls_p in ISO:
        ok_g = np.nan_to_num(MCCg, nan=-1) >= PUB_HPA.loc[f'MCC {c}', nome_p]
        ax.plot(f_g, np.where(ok_g.any(1), t_g[ok_g.argmax(1)], np.nan), color=col_p, ls=ls_p, lw=1.4,
                label=f'{nome_p}, pubblicato: MCC {PUB_HPA.loc[f"MCC {c}", nome_p]:.2f} (punti con lo stesso MCC)')
    ax.plot([0, 1], [0, 1], ':', color='#c3c2b7')
    ax.set(title=f'{c} ({P_c} positivi nel benchmark)', xlabel='Falsi positivi (FPR)', ylabel='Veri positivi (TPR)', xlim=(0, 1), ylim=(0, 1.01))
    ax.legend(fontsize=6.5, loc='lower right')
    ax.grid(color='#e6e5e0', lw=0.6)
# MCC per classe e metriche riassuntive: valori pubblicati e nostri
ax7, ax8 = axs.ravel()[6], axs.ravel()[7]
col_bar = ['#c3c2b7', '#a5a49c', '#8a8a86', '#5f5e59', '#3b3a36', '#111111', '#eb6834']
for b_i, nome_b in enumerate(confronto_pub.columns):
    ax7.bar(np.arange(len(HPA_EVAL)) + b_i * 0.12 - 0.36, confronto_pub.loc[[f'MCC {c}' for c in HPA_EVAL], nome_b], 0.12, color=col_bar[b_i], label=nome_b)
    ax8.bar(np.arange(4) + b_i * 0.12 - 0.36, confronto_pub.loc[['Accuracy', 'Jaccard', 'Micro F1', 'Macro F1'], nome_b], 0.12, color=col_bar[b_i])
ax7.set_xticks(np.arange(len(HPA_EVAL)), [c.replace('Endoplasmic reticulum', 'ER').replace('Golgi apparatus', 'Golgi') for c in HPA_EVAL], fontsize=8)
ax7.set(title='MCC per classe (pubblicati e nostro)', ylabel='MCC')
ax7.legend(fontsize=6.5, loc='upper left')
ax8.set_xticks(np.arange(4), ['Accuracy', 'Jaccard', 'Micro F1', 'Macro F1'])
ax8.set(title='Metriche multi-etichetta (pubblicati e nostro)')
for ax in (ax7, ax8):
    ax.grid(axis='y', color='#e6e5e0', lw=0.6)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle(f'Test HPA: {NOME_TOP} contro i valori pubblicati (DeepLoc 2.0, Tabella 2; {int(w_iso.sum()):,} voci su 1.717)', fontsize=14)
fig.tight_layout()
fig.savefig(DATA / 'roc_confronto_pubblicati.png', dpi=150, facecolor='white')
plt.show()
with pd.ExcelWriter(DATA / 'confronto_pubblicati.xlsx') as xw:
    confronto_pub.round(4).to_excel(xw, sheet_name='HPA, Tabella 2')
    auc_tab.round(4).to_excel(xw, sheet_name='HPA, AUC')
    pd.Series(mcc_bench).round(4).to_frame('MCC alle proporzioni del benchmark').to_excel(xw, sheet_name='HPA, MCC riproporzionato')
